# Structured Radiology Report Summarisation from Chest CT Using RadGenome-ChestCT
Clovers Akitoye, . MSc research project, project code DAIM2025A_067.
## Data pipeline and preprocessing


This notebook constructs the lung-region cohort from RadGenome-ChestCT, assembles the structured target reports the decoder learns to produce, preprocesses the chest CT volumes into a fixed volumetric representation, and caches that representation to disk so the encoder and training stages do not repeat the work.

The text layer of region-grounded report sentences comes from RadGenome-ChestCT, released under CC-BY-4.0. The image layer of CT volumes comes from CT-RATE, which is gated and requires accepting its terms on Hugging Face. The two are joined on the volume identifier.

## 1. Environment and data

This section installs the dependencies, authenticates with Hugging Face, and loads the grounded report metadata. The reports are combined into a single table with a split column so the train and validation partitions are fixed before any filtering. Fixing the partition early prevents reconstructions of the same scan from drifting across the train and test boundary later in the pipeline.

In [ ]:
# Storage redirect. The home quota is 50 GB, so model and dataset caches are placed on
# scratch. This must precede any Hugging Face or torch call.
import os

SCRATCH = "~/scratch"
HF_CACHE = os.path.join(SCRATCH, "hf_cache")
os.makedirs(HF_CACHE, exist_ok=True)

os.environ["HF_HOME"] = HF_CACHE                       # HuggingFace root
os.environ["HF_HUB_CACHE"] = os.path.join(HF_CACHE, "hub")  # downloaded datasets/models
os.environ["TRANSFORMERS_CACHE"] = os.path.join(HF_CACHE, "transformers")
os.environ["TORCH_HOME"] = os.path.join(SCRATCH, "torch_cache")  # Med3D / pretrained weights

print("Caches pointed at scratch:")
for k in ["HF_HOME", "HF_HUB_CACHE", "TRANSFORMERS_CACHE", "TORCH_HOME"]:
    print(f"  {k} = {os.environ[k]}")

In [ ]:
import sys, torch
print(sys.executable)
print(torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
import sys
!{sys.executable} -m pip install --quiet packaging

In [ ]:
import packaging; print(packaging.__version__)

In [ ]:
!{sys.executable} -m pip install --quiet ipywidgets

In [ ]:
# Imports.
import os, re, gc, glob, shutil, tarfile, tempfile
import numpy as np
import pandas as pd
import nibabel as nib
import torch
import matplotlib.pyplot as plt
from datasets import load_dataset
from huggingface_hub import login, hf_hub_download
from huggingface_hub.utils import HfHubHTTPError
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd, Spacingd,
    ScaleIntensityRanged, MaskIntensityd, Lambdad, Lambda, Resize,
)

In [ ]:
# Authenticate for the gated CT-RATE volumes. The token is entered interactively and is
# not written into the notebook.
login()

In [ ]:
# The full configuration is loaded rather than a single split, so that the available split
# names are read rather than assumed. Grounded reports form the text layer: scan name,
# anatomy path, and the sentence grounded to that region.
ds = load_dataset("RadGenome/RadGenome-ChestCT", "grounded reports")
print(ds)   # prints the split names and their row counts

In [ ]:
# All splits are combined into one table with a split column. Train rows form the training
# pool and the validation split serves as the held-out test set. Filtering once and slicing
# by split at the end fixes the partition here, so that no scan can cross the train and test
# boundary later.
frames = []
for name in ds.keys():
    part = ds[name].to_pandas()
    part["split"] = name
    frames.append(part)

df = pd.concat(frames, ignore_index=True)

print("total rows:", len(df))
print(df.groupby("split")["Volumename"].nunique().rename("unique_volumes"))
print()
print(df["split"].value_counts())
df.head(5)

## 2. Cohort construction

The cohort is restricted to the nine lung-region anatomy paths defined in Muralidharan (2025), which removes mediastinum, abdomen, trachea, and the generic non-lobar paths. A scan is kept only if a lung segmentation mask exists for it in the RadGenome region-mask archive, since the mask is required for region-grounded preprocessing. Where a study has more than one reconstruction, only the first is retained. Keeping multiple reconstructions of one study wastes compute and risks information leakage if the reconstructions split across the train and test partitions.

The two cells below first inspect how many scans and which anatomy paths are present, which confirms the filter targets the right strings before it is applied.

In [ ]:
# Row count and number of unique scans.
df['Volumename'].nunique()

In [ ]:
# Distribution of anatomy paths.
df['Anatomy'].value_counts()

In [ ]:
# The nine lung-region anatomy strings from Muralidharan (2025). Generic lung paths, null
# entries and all non-lung paths such as mediastinum, abdomen and trachea are excluded.
lung_regions = [
    "lung/lung/left lung/left lung upper lobe",     # Left Upper Lobe
    "lung/lung/left lung/left lung lower lobe",     # Left Lower Lobe
    "lung/lung/right lung/right lung upper lobe",   # Right Upper Lobe
    "lung/lung/right lung/right lung middle lobe",  # Right Middle Lobe
    "lung/lung/right lung/right lung lower lobe",   # Right Lower Lobe
    "lung/lung/lung upper lobe",                    # Upper Lobes (general)
    "lung/lung/lung lower lobe",                    # Lower Lobes (general)
    "lung/lung/right lung",                         # Right Lung (general)
    "lung/lung/left lung",                          # Left Lung (general)
]

df_lung = df[df["Anatomy"].isin(lung_regions)].copy()

# Reconstruction-agnostic study identifier: train_16242_a_1.nii.gz maps to train_16242_a.
df_lung["base_scan"] = df_lung["Volumename"].str.replace(r"_\d+\.nii\.gz$", "", regex=True)

print("rows after region filter:", len(df_lung))
print()
print("unique volumes per split:")
print(df_lung.groupby("split")["Volumename"].nunique())
print()
print("unique base scans per split:")
print(df_lung.groupby("split")["base_scan"].nunique())
print()
print(df_lung["Anatomy"].value_counts())

In [ ]:
def scans_with_masks(split):
    arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                          filename=f"dataset/{split}_region_mask.tar.gz")   # cached
    with tarfile.open(arc, "r:gz") as tar:
        # Archive layout is <wrapper>/seg_<stem>/lung.nii.gz, so the stem is the parent folder.
        return {n.split("/")[-2].replace("seg_", "") + ".nii.gz"
                for n in tar.getnames() if n.endswith("/lung.nii.gz")}

have_train = scans_with_masks("train")
have_valid = scans_with_masks("valid")
print("scans with a lung mask:", "train", len(have_train), " valid", len(have_valid))

have_by_split = {"train": have_train, "validation": have_valid}
df_cohort = df_lung[df_lung.apply(lambda r: r["Volumename"] in have_by_split[r["split"]], axis=1)].copy()

print()
print("cohort volumes per split (after mask intersection):")
print(df_cohort.groupby("split")["Volumename"].nunique())

In [ ]:
df_cohort["base_scan"] = df_cohort["Volumename"].str.replace(r"_\d+\.nii\.gz$", "", regex=True)

# One reconstruction per study. Reconstruction pairs such as _a_1 and _a_2 share a report,
# so retaining both duplicates compute and risks leakage if the pair straddles the split.
# The first volume name per study is retained, within each split.
chosen = (df_cohort[["split", "base_scan", "Volumename"]]
          .drop_duplicates()
          .sort_values("Volumename")
          .groupby(["split", "base_scan"], as_index=False)
          .first())

df_final = df_cohort[df_cohort["Volumename"].isin(set(chosen["Volumename"]))].copy()

print("studies per split (after reconstruction dedup):")
print(df_final.groupby("split")["Volumename"].nunique())
print("sentence rows kept:", len(df_final))

In [ ]:
# Verification that the identifier strip applied and that no reconstruction pairs remain.
print(df_cohort[["Volumename", "base_scan"]].drop_duplicates().head())
recon = (df_cohort.drop_duplicates("Volumename")
         .groupby(["split", "base_scan"])["Volumename"].nunique())
print("studies with >1 reconstruction:", int((recon > 1).sum()))
print("max reconstructions on any study:", int(recon.max()))

## 3. Target report assembly

Each report is assembled from the grounded sentences in a fixed section order, with the nine anatomy paths mapped to readable region headings. Repeated sentences within a region are removed. A fixed layout matters because the decoder learns the report template, so a consistent structure across every target reduces the variance the model has to absorb.

In [ ]:
# The nine anatomy paths mapped to the section names used in Muralidharan (2025), Table 1.
region_name = {
    "lung/lung/left lung/left lung upper lobe":    "LEFT UPPER LOBE",
    "lung/lung/left lung/left lung lower lobe":    "LEFT LOWER LOBE",
    "lung/lung/right lung/right lung upper lobe":  "RIGHT UPPER LOBE",
    "lung/lung/right lung/right lung middle lobe": "RIGHT MIDDLE LOBE",
    "lung/lung/right lung/right lung lower lobe":  "RIGHT LOWER LOBE",
    "lung/lung/lung upper lobe":                   "UPPER LOBES",
    "lung/lung/lung lower lobe":                   "LOWER LOBES",
    "lung/lung/right lung":                        "RIGHT LUNG",
    "lung/lung/left lung":                         "LEFT LUNG",
}
# A fixed section order gives every target report the same layout, so that the decoder
# learns a stable template rather than an arbitrary ordering.
region_order = ["LEFT UPPER LOBE", "LEFT LOWER LOBE", "RIGHT UPPER LOBE",
                "RIGHT MIDDLE LOBE", "RIGHT LOWER LOBE", "UPPER LOBES",
                "LOWER LOBES", "RIGHT LUNG", "LEFT LUNG"]

df_final["section"] = df_final["Anatomy"].map(region_name)

def build_report(g):
    parts = []
    for reg in region_order:
        seen, uniq = set(), []
        for s in g.loc[g["section"] == reg, "Sentence"]:
            s = str(s).strip()
            if s and s not in seen:          # dedup repeated sentences within a region
                seen.add(s); uniq.append(s)
        if uniq:
            parts.append(f"[{reg}] " + " ".join(uniq))
    return "\n".join(parts)

reports_y = (df_final.groupby(["split", "Volumename", "base_scan"])
             .apply(build_report)
             .reset_index(name="report_text"))

print(reports_y.groupby("split")["Volumename"].nunique())
print()
print(reports_y["report_text"].str.len().describe())
print()
print(reports_y.iloc[0]["report_text"][:600])   # eyeball one target

## 4. Volumetric preprocessing

Each volume passes through a single MONAI pipeline. The image is reoriented to a common axis convention, resampled onto a fixed voxel grid of 1.5 by 1.5 by 2.0 mm, windowed to the lung Hounsfield range of -1000 to 400 and normalised to the unit interval, and finally masked so that every voxel outside the lung is set to zero. Setting non-lung voxels to zero after windowing is equivalent to Muralidharan's approach of filling them with -1000 HU before windowing, because the window maps -1000 HU to zero.

RadGenome stores each region mask on the voxel grid of its source volume, but the stored affine can disagree with the volume affine. Resampling the mask through world space using that affine drops it outside the image and returns an empty mask. The pipeline therefore places the mask directly onto the image voxel grid and copies the image header, rather than trusting the affine. This is valid because the mask was segmented from the same volume and covers the same field of view.

In [ ]:
def align_mask_to_image(data):
    # The mask is resampled onto the image voxel grid and inherits the image header, so that
    # subsequent orientation and spacing operations treat both identically.
    img, msk = data["image"], data["mask"]
    if img.shape != msk.shape:
        msk = Resize(spatial_size=img.shape[1:], mode="nearest")(msk)   # nearest keeps it binary
    msk.affine = img.affine.clone()
    data["mask"] = msk
    return data

# Lung window of -1000 to 400 HU, a fixed voxel grid, and a lung-only image.
preprocess = Compose([
    LoadImaged(keys=["image", "mask"]),
    EnsureChannelFirstd(keys=["image", "mask"]),              # -> (1, H, W, D)
    Lambda(align_mask_to_image),                              # mask onto the image grid
    Orientationd(keys=["image", "mask"], axcodes="RAS"),      # same orientation for every scan
    Spacingd(keys=["image", "mask"], pixdim=(1.5, 1.5, 2.0),
             mode=("bilinear", "nearest")),                   # nearest keeps the mask binary
    Lambdad(keys=["mask"], func=lambda m: (m > 0).float()),   # force mask to 0/1
    ScaleIntensityRanged(keys=["image"], a_min=-1000, a_max=400,
                         b_min=0.0, b_max=1.0, clip=True),     # window and normalise to [0,1]
    MaskIntensityd(keys=["image"], mask_key="mask"),          # zero everything outside the lung
])
print("preprocess pipeline ready")

In [ ]:
def ctrate_path(volname):
    # CT-RATE layout: dataset/<split>_fixed/<split>_<patient>/<split>_<patient>_<scan>/<file>.
    # For example valid_53_a_1.nii.gz resolves to dataset/valid_fixed/valid_53/valid_53_a/valid_53_a_1.nii.gz.
    split, patient, scan, _ = volname.replace(".nii.gz", "").split("_")
    base = f"{split}_{patient}"
    return f"dataset/{split}_fixed/{base}/{base}_{scan}/{volname}"

### Verification

The following cells check the pipeline on a small sample and estimate the cached size before committing to the full run. They re-download a few volumes, so they are verification steps rather than part of the main run. A correct result shows lung coverage of a few per cent of the resampled volume, with the masked image grey inside the lungs and black elsewhere.

In [ ]:
N_SAMPLES = 3
# Samples are drawn from the validation cohort rather than train, because the validation
# mask archive is smaller and faster to read.
sample_vols = (reports_y[reports_y["split"] == "validation"]["Volumename"]
               .head(N_SAMPLES).tolist())
print("sample volumes:", sample_vols)

def get_volume(volname):
    # The volumes are gated. The CT-RATE terms must be accepted or this returns HTTP 401.
    try:
        return hf_hub_download("ibrahimhamamci/CT-RATE", repo_type="dataset",
                               filename=ctrate_path(volname))
    except HfHubHTTPError as e:
        raise SystemExit(
            f"could not fetch {volname}. if this is 401/403, accept the CT-RATE terms at "
            f"https://huggingface.co/datasets/ibrahimhamamci/CT-RATE first.\n{e}")

def get_masks(volnames):
    # Lung masks are read from the validation archive cached earlier.
    arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                          filename="dataset/valid_region_mask.tar.gz")   # cached, instant
    want = {v.replace(".nii.gz", ""): None for v in volnames}
    out_dir = tempfile.mkdtemp()
    with tarfile.open(arc, "r:gz") as tar:
        for m in tar:
            if not m.name.endswith("/lung.nii.gz"):
                continue
            stem = m.name.split("/")[-2].replace("seg_", "")     # seg_valid_53_a_1 -> valid_53_a_1
            if stem in want and want[stem] is None:
                dst = os.path.join(out_dir, f"{stem}_lung.nii.gz")
                with open(dst, "wb") as g:
                    g.write(tar.extractfile(m).read())
                want[stem] = dst
                if all(v is not None for v in want.values()):
                    break
    return want

masks = get_masks(sample_vols)

# The pipeline is applied to each sample and the output checked.
fig, axes = plt.subplots(N_SAMPLES, 2, figsize=(8, 4 * N_SAMPLES))
if N_SAMPLES == 1:
    axes = axes.reshape(1, -1)

for i, vol in enumerate(sample_vols):
    stem = vol.replace(".nii.gz", "")
    out = preprocess({"image": get_volume(vol), "mask": masks[stem]})
    img, msk = out["image"], out["mask"]

    # An empty mask zeroes the entire image, so a check for the absence of voxels outside the
    # lung passes trivially on a blank image and establishes nothing.
    # The mask is therefore checked for lung content: a genuine mask holds tens of thousands of
    # voxels, approximately 3 to 8 per cent of the resampled volume.
    lung  = int((msk > 0).sum())
    cover = 100 * lung / msk.numel()
    flag  = "   <-- EMPTY, alignment failed" if lung < 1000 else ""
    print(f"\n{vol}")
    print("  shape:", tuple(img.shape),
          "| windowed intensity:", round(float(img.min()), 3), "to", round(float(img.max()), 3))
    print(f"  lung voxels: {lung}  ({cover:.1f}% of volume){flag}")

    mid = img.shape[-1] // 2                                  # middle axial slice
    axes[i, 0].imshow(img[0, :, :, mid].cpu().numpy().T, cmap="gray", origin="lower")
    axes[i, 0].set_title(f"{stem}\nlung-masked, windowed"); axes[i, 0].axis("off")
    axes[i, 1].imshow(msk[0, :, :, mid].cpu().numpy().T, cmap="gray", origin="lower")
    axes[i, 1].set_title("lung mask"); axes[i, 1].axis("off")

plt.tight_layout(); plt.show()

In [ ]:
# Size probe. A small number of volumes are cached, the compressed size measured, and the
# total projected, so that the cache ceiling is set from measurement rather than estimate.

probe_vols = list(df_final[df_final["split"] == "validation"]["Volumename"].head(8))
masks_p = get_masks(probe_vols)                       # reuse cell 13's extractor
tmp, sizes = tempfile.mkdtemp(), []
for v in probe_vols:
    out = preprocess({"image": get_volume(v), "mask": masks_p[v.replace(".nii.gz", "")]})
    arr = out["image"][0].cpu().numpy().astype(np.float16)
    p = os.path.join(tmp, "probe.npz")
    np.savez_compressed(p, image=arr, affine=out["image"].affine.cpu().numpy())
    sizes.append(os.path.getsize(p) / 1e6)

mb = sum(sizes) / len(sizes)
print(f"avg compressed size: {mb:.1f} MB per volume")
print(f"validation (999):    ~{mb*999/1000:.1f} GB")
print(f"train (15127):       ~{mb*15127/1000:.0f} GB")
print(f"full cohort (16126): ~{mb*16126/1000:.0f} GB")
print("free disk right now: ", round(__import__('shutil').disk_usage('.').free/1e9, 1), "GB")

## 5. Caching the preprocessed cohort

Preprocessing every volume on each training run would be wasteful, so each preprocessed image is saved once. The loop streams one raw volume at a time, preprocesses it, writes the result as a compressed half-precision array, and deletes the raw volume immediately, which keeps only one large file on disk at any point. The run is resumable through a manifest, so an interruption does not lose completed work. Two limits, a cache ceiling and a free-disk floor, stop the run before it can exhaust the disk.

A volume is excluded if its lung mask contains fewer than 1000 voxels after alignment. This is a deliberate quality control: an empty or near-empty mask would produce a black image, and training on it would add noise rather than signal.

In [ ]:
# Parameters, set from the size probe above.
SPLIT          = "validation"                       # validation first: the smaller split, and the held-out test set
OUT_DIR        = os.path.expanduser("~/preproc_cache")
MANIFEST       = os.path.join(OUT_DIR, "manifest.csv")
MAX_CACHE_GB   = 5                                  # hard ceiling on this cache
SAFETY_FREE_GB = 20                                 # stop if free disk drops below this, no matter what

# Requires df_final and preprocess to be defined in the kernel.
assert "df_final" in globals(), "run cell 9 first (df_final missing)"
assert "preprocess" in globals(), "run cell 12 first (preprocess missing)"

ARCH_SPLIT = {"train": "train", "validation": "valid"}[SPLIT]
os.makedirs(OUT_DIR, exist_ok=True)
SCRATCH = tempfile.mkdtemp(prefix="radgenome_scratch_")   # one raw volume lives here, then dies

want = set(df_final[df_final["split"] == SPLIT]["Volumename"])
free_gb  = lambda: shutil.disk_usage(OUT_DIR).free / 1e9
cache_gb = lambda: sum(os.path.getsize(os.path.join(OUT_DIR, f))
                       for f in os.listdir(OUT_DIR)) / 1e9

rows = pd.read_csv(MANIFEST).to_dict("records") if os.path.exists(MANIFEST) else []
seen = {r["volumename"] for r in rows}             # resumable: don't redo what's logged

arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                      filename=f"dataset/{ARCH_SPLIT}_region_mask.tar.gz")   # cached, instant
done = 0
with tarfile.open(arc, "r:gz") as tar:             # single pass over the mask archive
    for m in tar:
        if not m.name.endswith("/lung.nii.gz"):
            continue
        vol  = m.name.split("/")[-2].replace("seg_", "") + ".nii.gz"
        stem = vol.replace(".nii.gz", "")
        if vol not in want or vol in seen:
            continue
        if cache_gb() >= MAX_CACHE_GB:
            print(f"\nhit MAX_CACHE_GB ({MAX_CACHE_GB}). stopping clean."); break
        if free_gb() <= SAFETY_FREE_GB:
            print(f"\nfree disk {round(free_gb(),1)} GB, below floor. stopping clean."); break

        msk_path = os.path.join(SCRATCH, f"{stem}_lung.nii.gz")
        with open(msk_path, "wb") as g:
            g.write(tar.extractfile(m).read())
        vol_path = ""
        try:
            vol_path = hf_hub_download("ibrahimhamamci/CT-RATE", repo_type="dataset",
                                       filename=ctrate_path(vol), local_dir=SCRATCH)
            out  = preprocess({"image": vol_path, "mask": msk_path})
            lung = int((out["mask"] > 0).sum())
            if lung < 1000:                        # alignment failed, log it, don't cache garbage
                rows.append(dict(volumename=vol, stem=stem, split=SPLIT,
                                 status="fail", error="empty_mask", lung_voxels=lung, size_mb=0))
            else:
                op = os.path.join(OUT_DIR, f"{stem}.npz")
                np.savez_compressed(op,
                    image=out["image"][0].cpu().numpy().astype(np.float16),   # float16 = half the bytes
                    affine=out["image"].affine.cpu().numpy().astype(np.float32))
                rows.append(dict(volumename=vol, stem=stem, split=SPLIT, status="ok",
                                 error="", lung_voxels=lung,
                                 size_mb=round(os.path.getsize(op)/1e6, 2)))
                done += 1
        except Exception as e:
            rows.append(dict(volumename=vol, stem=stem, split=SPLIT,
                             status="fail", error=str(e)[:200], lung_voxels=0, size_mb=0))
        finally:
            for f in (msk_path, vol_path):
                if f and os.path.exists(f): os.remove(f)   # reclaim scratch immediately
            gc.collect()
        if done % 25 == 0 and done:
            pd.DataFrame(rows).to_csv(MANIFEST, index=False)   # flush so a crash loses nothing
            print(f"  {done} cached | cache {round(cache_gb(),1)} GB | free {round(free_gb(),1)} GB")

pd.DataFrame(rows).to_csv(MANIFEST, index=False)
shutil.rmtree(SCRATCH, ignore_errors=True)
print(f"\ndone. {done} cached this run. cache {round(cache_gb(),1)} GB, free {round(free_gb(),1)} GB.")


# Loader for downstream stages. Half precision on disk, single precision for the model.
def load_study(stem):
    d = np.load(os.path.join(OUT_DIR, f"{stem}.npz"))
    return torch.from_numpy(d["image"]).float().unsqueeze(0)   # (1, H, W, D) encoder input

# The usable cohort, recorded to disk rather than inferred.
m = pd.read_csv(MANIFEST)
print("\nmanifest:")
print(m["status"].value_counts())
if (m["status"] == "fail").any():
    print(m.loc[m["status"] == "fail", "error"].value_counts())

### Disk maintenance

The reclaim cell removes the training mask archive and leftover raw volumes, which are not needed for the validation milestone. The reset cell clears the cache to allow a clean rebuild, and is guarded so that running the whole notebook does not erase a completed cache by accident.

In [ ]:
# One-time reclaim. Removes the training mask archive, approximately 17 GB and not required
# for the validation milestone, together with residual raw volumes. Files are located
# directly in the cache and hf_hub_download is never called, so the archive cannot be
# re-fetched. Both steps are idempotent.
hub = os.path.expanduser("~/.cache/huggingface/hub")

hits = glob.glob(os.path.join(hub, "datasets--RadGenome--RadGenome-ChestCT",
                               "**", "train_region_mask.tar.gz"), recursive=True)
if hits:
    for link in hits:
        blob = os.path.realpath(link)
        if os.path.exists(blob): os.remove(blob)
        if os.path.lexists(link): os.remove(link)
    print(f"dropped train mask archive ({len(hits)} ref)")
else:
    print("train mask archive not in cache, nothing to drop")

shutil.rmtree(os.path.join(hub, "datasets--ibrahimhamamci--CT-RATE"), ignore_errors=True)
print("cleared raw CT-RATE cache")
print("free now:", round(shutil.disk_usage(".").free / 1e9, 1), "GB")

In [ ]:
# Clears the cache for a rebuild. Guarded, so that a linear re-execution cannot erase a
# completed cache. DO_RESET is set to True only when the cache is being rebuilt.
DO_RESET = False

if DO_RESET:
    before = shutil.disk_usage(".").free / 1e9
    OUT_DIR = os.path.expanduser("~/preproc_cache")
    if os.path.exists(OUT_DIR):
        n = len([f for f in os.listdir(OUT_DIR) if f.endswith(".npz")])
        shutil.rmtree(OUT_DIR); print(f"wiped preproc_cache ({n} npz files)")
    os.makedirs(OUT_DIR, exist_ok=True)
    shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/datasets--ibrahimhamamci--CT-RATE"),
                  ignore_errors=True)
    for d in glob.glob(os.path.join(tempfile.gettempdir(), "radgenome_scratch_*")):
        shutil.rmtree(d, ignore_errors=True)
    after = shutil.disk_usage(".").free / 1e9
    print(f"reset done. free {before:.1f} -> {after:.1f} GB (kept the RadGenome valid mask archive)")
else:
    print("DO_RESET is False, skipping. set it to True to wipe and start the cache fresh.")

In [ ]:
!du -sh ~/.cache/huggingface/hub/* | sort -h
!du -sh preproc_cache

### Excluded volumes

The triage cell reads the raw masks for the excluded volumes directly from the archive, before any processing. If the raw masks are near-empty, the exclusion reflects a genuine fault in the source segmentation rather than a fault in the pipeline.

In [ ]:
# The raw mask for each excluded volume is read directly from the archive, before any
# processing. A near-zero raw voxel count establishes that the source mask is empty and that
m = pd.read_csv(os.path.expanduser("~/preproc_cache/manifest.csv"))
failed = set(m.loc[m["error"] == "empty_mask", "stem"])
print(f"{len(failed)} empty_mask exclusions to check\n")

arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                      filename="dataset/valid_region_mask.tar.gz")   # cached, instant

rows = []
with tarfile.open(arc, "r:gz") as tar:
    for member in tar:
        if not member.name.endswith("/lung.nii.gz"):
            continue
        stem = member.name.split("/")[-2].replace("seg_", "")
        if stem not in failed:
            continue
        tmp = os.path.join(tempfile.mkdtemp(), "m.nii.gz")
        with open(tmp, "wb") as g:
            g.write(tar.extractfile(member).read())
        arr = nib.load(tmp).get_fdata()
        rows.append({"stem": stem, "raw_shape": arr.shape,
                     "raw_lung_voxels": int(np.count_nonzero(arr))})

triage = pd.DataFrame(rows)
print(triage.to_string(index=False))

In [ ]:
import os
p = os.path.expanduser("~/preproc_cache")
print(len(os.listdir(p)), "files in cache")
print([f for f in os.listdir(p)][:3])

## 6. Three-dimensional encoder

The first of the three fixes replaces the two-dimensional encoder with a volumetric one. Med3D supplies a 3D ResNet-34 pretrained on eight segmentation datasets, which is the closest available match in depth and family to the 2D ResNet-34 used in the other arm, so that a later comparison between the two isolates the convolution dimensionality rather than the architecture.

The weights are distributed through the MedicalNet repository and a separate release archive. The cells below obtain both, then push a single cached volume through the backbone to establish three quantities that constrain everything downstream: whether the pretrained weights load rather than falling back to random initialisation, the channel width of the feature map, which fixes the input width of the bridge, and the downsampling factor between input and feature map.

In [ ]:
!git clone https://github.com/Tencent/MedicalNet

In [ ]:
print(os.listdir())

In [ ]:
!pip install gdown --quiet
!gdown 13tnSvXY7oDIEloNFiGTsjUIYfS3g3BfG -O medicalnet_files.zip

In [ ]:
import zipfile, glob, shutil, os

with zipfile.ZipFile("medicalnet_files.zip") as z:
    z.extractall("medicalnet_dl")           # unzip into a temp folder

hit = glob.glob("medicalnet_dl/**/resnet_34.pth", recursive=True)  # find the file wherever it landed
print("found:", hit)

os.makedirs("MedicalNet/pretrain", exist_ok=True)
shutil.copy(hit[0], "MedicalNet/pretrain/resnet_34.pth")
print("weights in place:", os.path.exists("MedicalNet/pretrain/resnet_34.pth"))

In [ ]:
# Med3D smoke test: confirm the pretrained weights load, and read off the feature width and downsample factor.
import os, sys, torch, numpy as np
sys.path.append("MedicalNet")
from models import resnet                       # MedicalNet's 3D resnet builder

WEIGHTS   = "MedicalNet/pretrain/resnet_34.pth"
DEVICE    = "cpu"
CACHE_NPZ = os.path.expanduser("~/preproc_cache/valid_1002_a_1.npz")

# The backbone is constructed. num_seg_classes is required by the constructor but the
# segmentation head is unused; the feature map is captured before it with a hook.
model = resnet.resnet34(sample_input_D=64, sample_input_H=224, sample_input_W=224,
                        num_seg_classes=2, shortcut_type="A",
                        no_cuda=(DEVICE != "cuda"))

# The released weights were saved under DataParallel, so keys carry a module prefix that
# does not match a bare model and is stripped. strict=False prevents the unused segmentation
# head keys from failing the load. The printed key report is the evidence that it loaded.
sd = torch.load(WEIGHTS, map_location="cpu")
sd = sd.get("state_dict", sd)
sd = {k.replace("module.", ""): v for k, v in sd.items()}
rep = model.load_state_dict(sd, strict=False)
print("missing  (should be ONLY conv_seg / seg head):", rep.missing_keys[:6])
print("unexpected (should be ~none):", rep.unexpected_keys[:6])

# Capture the layer4 output, the dense feature map, before the segmentation head.
feat = {}
model.layer4.register_forward_hook(lambda m, i, o: feat.__setitem__("map", o.detach()))
model.eval().to(DEVICE)

vol = torch.from_numpy(np.load(CACHE_NPZ)["image"]).float()[None, None].to(DEVICE)  # (1,1,H,W,D)
with torch.no_grad():
    model(vol)

fmap = feat["map"]
print("\ninput volume:", tuple(vol.shape))
print("feature map :", tuple(fmap.shape), " -> channels =", fmap.shape[1])  # 512 for resnet34
ds = [round(vol.shape[i + 2] / fmap.shape[i + 2], 2) for i in range(3)]
print("downsample  :", ds, "(input voxels per feature voxel, per axis)")
print("finite?     :", bool(torch.isfinite(fmap).all()),
      "| std:", round(float(fmap.std()), 4))   # a non-zero standard deviation indicates signal rather than a constant output

In [ ]:
import os, tarfile
from huggingface_hub import hf_hub_download

arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                      filename="dataset/valid_region_mask.tar.gz")   # already cached

with tarfile.open(arc, "r:gz") as tar:
    names = tar.getnames()

# Every mask file within a single scan folder, to establish which regions are distributed.
first  = next(n for n in names if n.endswith("/lung.nii.gz"))
folder = os.path.dirname(first)
print("folder:", folder)
for n in sorted(names):
    if n.startswith(folder) and n.endswith(".nii.gz"):
        print("  ", os.path.basename(n))

## 7. Stage A cohort and the two-dimensional encoder

Supervision confirmed a two-dimensional prototype before the volumetric arm, so the training pool is narrowed from 15,127 studies to a size a single GPU can cycle through repeatedly within the available time. The validation split is left at full size, since it serves as the held-out test set and its size determines the precision of every reported metric.

The draw is stratified by report length. Length is used as a proxy for finding richness because the CT-RATE abnormality label table was deliberately excluded from this pipeline, and an unstratified draw from a corpus this skewed collapses towards the shortest and most formulaic reports.

The encoder is a frozen ImageNet ResNet-34 truncated before the classifier, giving a 512-channel feature map per slice which is then averaged over the lung mask. Freezing follows the BLIP-2 recipe: only the bridge and the decoder adapters train, which keeps the trainable surface small enough for the cohort size.

In [ ]:
# Stage A cohort: narrow the training pool, stratified by report length, with a recorded seed.
import os
SEED, N_TRAIN = 42, 2500
CACHE_DIR = os.path.expanduser("~/preproc_cache")

train_y = reports_y[reports_y["split"] == "train"].copy()
val_y   = reports_y[reports_y["split"] == "validation"].copy()

train_y["len"]  = train_y["report_text"].str.len()
train_y["band"] = pd.qcut(train_y["len"], q=4, labels=False, duplicates="drop")

frac = N_TRAIN / len(train_y)
stage_a_train = (train_y.groupby("band", group_keys=False)
                        .sample(frac=frac, random_state=SEED))   # ~2,500, all bands kept

print("train pool:", len(train_y), "-> stage A train:", len(stage_a_train))
print("length spread kept (chars):")
print(stage_a_train["len"].describe().round(0))

# Manifests read by the encoder stage, so that the cohort is fixed on disk.
stage_a_train[["Volumename", "report_text"]].to_csv("stage_a_train.csv", index=False)
val_y[["Volumename", "report_text"]].to_csv("stage_a_val.csv", index=False)

# Which studies are already preprocessed and which remain to be cached.
def cached(v): return os.path.exists(os.path.join(CACHE_DIR, v.replace(".nii.gz", ".npz")))
print(f"\ncached already -> train {stage_a_train['Volumename'].map(cached).sum()}"
      f"/{len(stage_a_train)} | val {val_y['Volumename'].map(cached).sum()}/{len(val_y)}")
print("the gap is what the preprocessing loop still has to cache before features can run")

In [ ]:
import os
CACHE_DIR = os.path.expanduser("~/preproc_cache")
OUT_DIR   = CACHE_DIR

In [ ]:
%pip install torchvision

In [ ]:
# Stage A encoder: frozen 2D ResNet-34 with masked per-slice pooling.
import os, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
import torchvision

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ResNet-34 without the pooling and classifier layers, retaining the (512, h, w) map per slice.
backbone = torchvision.models.resnet34(weights="IMAGENET1K_V1")
encoder  = nn.Sequential(*list(backbone.children())[:-2]).eval().to(DEVICE)
for p in encoder.parameters():
    p.requires_grad = False                                   # frozen

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def encode_volume(npz_path, slice_batch=32):
    data = np.load(npz_path)
    img  = torch.from_numpy(data["image"]).float()            # (1, H, W, D)
    if img.ndim == 4:
        img = img[0]                                          # (H, W, D)
    if "mask" in data:
        m = torch.from_numpy(data["mask"]).float()
        lung = m[0] if m.ndim == 4 else m
    else:
        lung = (img > 0).float()                              # masked image -> lung is >0

    # Only slices containing lung are retained. Chest CT contains many empty slices at either
    # extreme, and encoding them adds cost and noise to the sequence.
    idx = torch.where(lung.sum(dim=(0, 1)) > 0)[0]
    img, lung = img[..., idx], lung[..., idx]                 # (H, W, d)

    sl = img.permute(2, 0, 1).unsqueeze(1).repeat(1, 3, 1, 1) # (d, 3, H, W)
    sl = (sl - MEAN) / STD                                    # imagenet normalisation
    lm = lung.permute(2, 0, 1).unsqueeze(1)                   # (d, 1, H, W)

    out = []
    for s in range(0, sl.shape[0], slice_batch):
        fb = encoder(sl[s:s + slice_batch].to(DEVICE))        # (b, 512, h, w)
        mb = F.interpolate(lm[s:s + slice_batch].to(DEVICE),
                           size=fb.shape[-2:], mode="nearest") # mask to feature size
        num = (fb * mb).sum(dim=(2, 3))
        den = mb.sum(dim=(2, 3)).clamp(min=1.0)
        out.append((num / den).cpu())                         # masked average -> (b, 512)
    return torch.cat(out, dim=0)                              # (lung slices, 512)

# Verified on one cached validation scan before the cohort is encoded.
seq = encode_volume(os.path.join(CACHE_DIR, "valid_1002_a_1.npz"))
print("device   :", DEVICE)
print("seq shape:", tuple(seq.shape), "-> (lung slices, 512)")
print("finite?  :", bool(torch.isfinite(seq).all()), "| std:", round(float(seq.std()), 4))

In [ ]:
# Cache the narrowed Stage A training cohort.
SPLIT          = "train"
OUT_DIR        = os.path.expanduser("~/preproc_cache")
MANIFEST       = os.path.join(OUT_DIR, "manifest.csv")
MAX_CACHE_GB   = 10                                  # was 5; train pushes total to ~5.6 GB
SAFETY_FREE_GB = 20

assert "preprocess" in globals(), "run the preprocess cell first"
ARCH_SPLIT = "train"
os.makedirs(OUT_DIR, exist_ok=True)
SCRATCH = tempfile.mkdtemp(prefix="radgenome_scratch_")

# The narrowed cohort fixed above, not the full training pool.
want = set(pd.read_csv("stage_a_train.csv")["Volumename"])
print("targeting", len(want), "narrowed train volumes")

free_gb  = lambda: shutil.disk_usage(OUT_DIR).free / 1e9
cache_gb = lambda: sum(os.path.getsize(os.path.join(OUT_DIR, f))
                       for f in os.listdir(OUT_DIR)) / 1e9

rows = pd.read_csv(MANIFEST).to_dict("records") if os.path.exists(MANIFEST) else []
seen = {r["volumename"] for r in rows}               # val rows already here; train resumes here too

arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                      filename=f"dataset/{ARCH_SPLIT}_region_mask.tar.gz")   # ~17 GB, one-time
done = 0
with tarfile.open(arc, "r:gz") as tar:
    for m in tar:
        if not m.name.endswith("/lung.nii.gz"):
            continue
        vol  = m.name.split("/")[-2].replace("seg_", "") + ".nii.gz"
        stem = vol.replace(".nii.gz", "")
        if vol not in want or vol in seen:
            continue
        if cache_gb() >= MAX_CACHE_GB:
            print(f"\nhit MAX_CACHE_GB ({MAX_CACHE_GB}). stopping clean."); break
        if free_gb() <= SAFETY_FREE_GB:
            print(f"\nfree disk {round(free_gb(),1)} GB, below floor. stopping clean."); break

        msk_path = os.path.join(SCRATCH, f"{stem}_lung.nii.gz")
        with open(msk_path, "wb") as g:
            g.write(tar.extractfile(m).read())
        vol_path = ""
        try:
            vol_path = hf_hub_download("ibrahimhamamci/CT-RATE", repo_type="dataset",
                                       filename=ctrate_path(vol), local_dir=SCRATCH)
            out  = preprocess({"image": vol_path, "mask": msk_path})
            lung = int((out["mask"] > 0).sum())
            if lung < 1000:
                rows.append(dict(volumename=vol, stem=stem, split=SPLIT,
                                 status="fail", error="empty_mask", lung_voxels=lung, size_mb=0))
            else:
                op = os.path.join(OUT_DIR, f"{stem}.npz")
                np.savez_compressed(op,
                    image=out["image"][0].cpu().numpy().astype(np.float16),
                    affine=out["image"].affine.cpu().numpy().astype(np.float32))
                rows.append(dict(volumename=vol, stem=stem, split=SPLIT, status="ok",
                                 error="", lung_voxels=lung,
                                 size_mb=round(os.path.getsize(op)/1e6, 2)))
                done += 1
        except Exception as e:
            rows.append(dict(volumename=vol, stem=stem, split=SPLIT,
                             status="fail", error=str(e)[:200], lung_voxels=0, size_mb=0))
        finally:
            for f in (msk_path, vol_path):
                if f and os.path.exists(f): os.remove(f)
            gc.collect()
        if done % 25 == 0 and done:
            pd.DataFrame(rows).to_csv(MANIFEST, index=False)
            print(f"  {done} cached | cache {round(cache_gb(),1)} GB | free {round(free_gb(),1)} GB")

pd.DataFrame(rows).to_csv(MANIFEST, index=False)
shutil.rmtree(SCRATCH, ignore_errors=True)
print(f"\ndone. {done} train cached this run. cache {round(cache_gb(),1)} GB, free {round(free_gb(),1)} GB.")

m = pd.read_csv(MANIFEST)
print("\nmanifest by split/status:")
print(m.groupby(["split", "status"]).size())

In [ ]:
# Self-contained feature extraction, valid after a kernel restart.
import os, numpy as np, pandas as pd, torch, torch.nn as nn
import torch.nn.functional as F, torchvision

CACHE_DIR = os.path.expanduser("~/preproc_cache")
OUT_DIR   = CACHE_DIR
MANIFEST  = os.path.join(OUT_DIR, "manifest.csv")
FEAT_DIR  = os.path.expanduser("~/feat_cache_2d")
os.makedirs(FEAT_DIR, exist_ok=True)

DEVICE   = "mps" if torch.backends.mps.is_available() else "cpu"
backbone = torchvision.models.resnet34(weights="IMAGENET1K_V1")
encoder  = nn.Sequential(*list(backbone.children())[:-2]).eval().to(DEVICE)
for p in encoder.parameters():
    p.requires_grad = False
MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

@torch.no_grad()
def encode_volume(npz_path, slice_batch=32):
    data = np.load(npz_path)
    img  = torch.from_numpy(data["image"]).float()
    if img.ndim == 4: img = img[0]
    lung = (img > 0).float()                                   # masked image -> lung is >0
    idx  = torch.where(lung.sum(dim=(0, 1)) > 0)[0]            # keep lung-bearing slices
    img, lung = img[..., idx], lung[..., idx]
    sl = img.permute(2, 0, 1).unsqueeze(1).repeat(1, 3, 1, 1)
    sl = (sl - MEAN) / STD
    lm = lung.permute(2, 0, 1).unsqueeze(1)
    out = []
    for s in range(0, sl.shape[0], slice_batch):
        fb  = encoder(sl[s:s + slice_batch].to(DEVICE))
        mb  = F.interpolate(lm[s:s + slice_batch].to(DEVICE), size=fb.shape[-2:], mode="nearest")
        num = (fb * mb).sum(dim=(2, 3)); den = mb.sum(dim=(2, 3)).clamp(min=1.0)
        out.append((num / den).cpu())
    return torch.cat(out, dim=0)                               # (lung slices, 512)

m  = pd.read_csv(MANIFEST)
ok = m[m["status"] == "ok"]
print("studies to feature-extract:", len(ok))

index, done = [], 0
for _, r in ok.iterrows():
    stem = r["stem"]; fp = os.path.join(FEAT_DIR, f"{stem}.npy")
    if os.path.exists(fp):
        index.append((stem, r["split"])); continue
    npz = os.path.join(OUT_DIR, f"{stem}.npz")
    if not os.path.exists(npz): continue
    np.save(fp, encode_volume(npz).numpy().astype(np.float16))
    index.append((stem, r["split"])); done += 1
    if done % 100 == 0: print(f"  {done} extracted")

idx = pd.DataFrame(index, columns=["stem", "split"]).drop_duplicates("stem")
idx.to_csv(os.path.join(FEAT_DIR, "feature_index.csv"), index=False)
print(f"\ndone. {done} new, {len(idx)} total.")
print(idx["split"].value_counts())

def load_feats(stem):
    return torch.from_numpy(np.load(os.path.join(FEAT_DIR, f"{stem}.npy"))).float()

## 8. Cross-attention bridge

The second fix replaces soft-prompt conditioning with a bridge modelled on the BLIP-2 Q-Former. A set of learned queries attends over itself, cross-attends to the encoder sequence, and projects into the decoder embedding width. Soft prompts in the baseline pipeline never read the image, which left the frozen decoder free to ignore them entirely.

The final projection width is read from the BioGPT-Large configuration rather than assumed, since a mismatch would silently produce a visual prefix the decoder cannot consume. The unit test that follows checks shape, finiteness, trainability and gradient flow on synthetic sequences of unequal length, because real studies contain variable numbers of lung slices and padding has to be handled before batching rather than after.

In [ ]:
# Bridge definition, with the decoder width read from the model configuration.
!pip install transformers --quiet

import torch, torch.nn as nn
from transformers import AutoConfig

# The decoder width is read from the configuration rather than assumed. The final projection
# must match it, or the visual prefix cannot be consumed by the decoder.
cfg = AutoConfig.from_pretrained("microsoft/BioGPT-Large")
HID = cfg.hidden_size
print("BioGPT-Large hidden size:", HID)

class QFormer(nn.Module):
    """
    BLIP-2-inspired bridge. A fixed set of LEARNED query tokens pulls information
    out of the per-slice image features by cross-attention, coordinates among
    themselves by self-attention, then projects to the decoder's hidden size so
    the queries act as a soft visual prefix the decoder reads from.

    This is the direct replacement for Sandeep's soft-prompt. His prompt vectors
    were static and never looked at the image, so the frozen decoder ignored them.
    These queries actively attend to the volume, which is why the conditioning
    survives instead of being washed out.
    """
    def __init__(self, img_dim=512, d_model=512, n_queries=32, n_heads=8,
                 n_layers=4, decoder_hidden=HID, ff_mult=4, dropout=0.1):
        super().__init__()
        self.queries  = nn.Parameter(torch.randn(1, n_queries, d_model) * 0.02)  # learned
        self.img_proj = nn.Linear(img_dim, d_model)            # slice features -> model width
        self.layers   = nn.ModuleList()
        for _ in range(n_layers):
            self.layers.append(nn.ModuleDict(dict(
                sa = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
                ca = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
                ff = nn.Sequential(nn.Linear(d_model, d_model*ff_mult), nn.GELU(),
                                   nn.Linear(d_model*ff_mult, d_model)),
                n1 = nn.LayerNorm(d_model), n2 = nn.LayerNorm(d_model), n3 = nn.LayerNorm(d_model),
            )))
        self.out_proj = nn.Linear(d_model, decoder_hidden)     # -> BioGPT hidden size

    def forward(self, img_feats, key_padding_mask=None):
        # img_feats: (B, S, img_dim). key_padding_mask: (B, S), where True marks padding.
        q  = self.queries.expand(img_feats.shape[0], -1, -1)   # (B, N, d_model)
        kv = self.img_proj(img_feats)                          # (B, S, d_model)
        for L in self.layers:
            h, _ = L["sa"](q, q, q)                            # queries coordinate among themselves
            q    = L["n1"](q + h)
            h, _ = L["ca"](q, kv, kv, key_padding_mask=key_padding_mask)  # queries read the image
            q    = L["n2"](q + h)
            q    = L["n3"](q + L["ff"](q))                     # feedforward
        return self.out_proj(q)                                # (B, N, decoder_hidden)

print("QFormer defined")

In [ ]:
# Bridge unit test: shape, finiteness, trainability and gradient flow on padded synthetic input.
import torch
torch.manual_seed(0)

# Two synthetic studies of unequal slice count, padded to the longer. Real studies vary in
# lung slice count, so the bridge must accept padding before batching is introduced.
lens = [154, 96]
S, B = max(lens), len(lens)
feats = torch.zeros(B, S, 512)
mask  = torch.ones(B, S, dtype=torch.bool)          # True = padding
for i, Ln in enumerate(lens):
    feats[i, :Ln] = torch.randn(Ln, 512)
    mask[i, :Ln]  = False                           # real slices = not padded

qf  = QFormer(img_dim=512, decoder_hidden=HID, n_queries=32)
out = qf(feats, key_padding_mask=mask)

n_train = sum(p.numel() for p in qf.parameters() if p.requires_grad)
print("input  :", tuple(feats.shape), "(B, slices, 512)")
print("output :", tuple(out.shape), f"(B, queries, {HID})  -> fits BioGPT as a soft prefix")
print("finite :", bool(torch.isfinite(out).all()))
print("trainable params:", f"{n_train/1e6:.1f}M")

out.sum().backward()                                # a loss must produce gradients
g = qf.queries.grad
print("gradients reach the learned queries:", bool(g is not None and torch.isfinite(g).all()))

# Optionally repeated on a cached sequence where load_feats remains defined.
try:
    real = load_feats("valid_1002_a_1").unsqueeze(0)          # (1, n_slices, 512)
    print("\nreal study check:", tuple(qf(real).shape), "from", tuple(real.shape))
except Exception:
    pass

In [ ]:
pip install sacremoses

In [ ]:
print(len(df))
print(len(ReportSet(df)))

In [ ]:
# Training loader: cached features paired with tokenised target reports.
!pip install sacremoses --quiet          # BioGPT's tokeniser needs this

import os, numpy as np, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

FEAT_DIR  = os.path.expanduser("~/feat_cache_2d")     # symlinked to scratch
TRAIN_CSV = "stage_a_train.csv"
DEV_CSV   = "stage_a_val.csv"
MAX_TOK   = 512                          # cap report length in tokens

tok = AutoTokenizer.from_pretrained("microsoft/BioGPT-Large")
PAD = tok.pad_token_id if tok.pad_token_id is not None else 0

def stem_of(v):   return v.replace(".nii.gz", "")
def feat_path(s): return os.path.join(FEAT_DIR, f"{s}.npy")

def cohort(csv_path):
    # A study is usable only once its features exist on disk. The difference between the cohort
    # manifest and the feature cache is the outstanding preprocessing.
    f = pd.read_csv(csv_path)
    f["stem"] = f["Volumename"].map(stem_of)
    have = f["stem"].map(lambda s: os.path.exists(feat_path(s)))
    print(f"{csv_path:20s} {have.sum():5d}/{len(f)} studies have cached features")
    return f[have].reset_index(drop=True)

train_frame = cohort(TRAIN_CSV)
dev_frame   = cohort(DEV_CSV)

# The code path is exercised on whatever is available, and the training cohort takes over as
# preprocessing completes, without modification to this cell.
WIRING_ONLY = len(train_frame) == 0
fit_frame   = train_frame if not WIRING_ONLY else dev_frame.iloc[:64].copy()
assert len(fit_frame) > 0, "no cached features at all, preprocessing has produced nothing"
if WIRING_ONLY:
    print("\nWIRING PROOF ONLY: no train features cached yet, using 64 held-out studies "
          "to exercise the code path. No weights are kept and no metric is reported "
          "from this run.")

class ReportSet(Dataset):
    def __init__(self, frame): self.frame = frame
    def __len__(self): return len(self.frame)
    def __getitem__(self, i):
        row   = self.frame.iloc[i]
        feats = torch.from_numpy(np.load(feat_path(row["stem"]))).float()   # (slices, 512)
        ids   = tok(row["report_text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        return feats, torch.tensor(ids, dtype=torch.long)

# Studies differ in both slice count and report length, so each is padded to the batch
# maximum and the two required masks are constructed.
def collate(batch):
    feats, ids = zip(*batch)
    S = max(f.shape[0] for f in feats); T = max(len(x) for x in ids); B = len(batch)
    feat_pad = torch.zeros(B, S, 512)
    key_pad  = torch.ones(B, S, dtype=torch.bool)       # True = padded slice, Q-Former ignores it
    id_pad   = torch.full((B, T), PAD, dtype=torch.long)
    attn     = torch.zeros(B, T, dtype=torch.long)      # 1 = real token
    for b, (f, x) in enumerate(zip(feats, ids)):
        feat_pad[b, :f.shape[0]] = f; key_pad[b, :f.shape[0]] = False
        id_pad[b, :len(x)] = x;       attn[b, :len(x)] = 1
    return feat_pad, key_pad, id_pad, attn

loader = DataLoader(ReportSet(fit_frame), batch_size=4, shuffle=True, collate_fn=collate)

# One batch is drawn and the shapes verified.
feat_pad, key_pad, id_pad, attn = next(iter(loader))
print("\nfeatures  :", tuple(feat_pad.shape), "(B, slices, 512)  feeds the Q-Former")
print("slice mask:", tuple(key_pad.shape),  "(B, slices)       True = padding")
print("report ids:", tuple(id_pad.shape),   "(B, tokens)       what BioGPT learns to produce")
print("token mask:", tuple(attn.shape),     "(B, tokens)       1 = real token")

if "QFormer" in globals() and "HID" in globals():
    prefix = QFormer(img_dim=512, decoder_hidden=HID, n_queries=32)(feat_pad, key_padding_mask=key_pad)
    print("Q-Former prefix:", tuple(prefix.shape), f"(B, 32, {HID})  the visual preamble for BioGPT")

## 9. Decoder adaptation

The third fix attaches LoRA adapters to the attention projections of BioGPT-Large while leaving all 1.5 billion base weights frozen. The baseline pipeline froze the decoder completely, which meant it had no mechanism for learning to use visual conditioning even when that conditioning was informative.

The cell that follows the adapters documents a correction. BioGPT-Large uses a scaled word embedding that already multiplies by the square root of the hidden size inside its own forward pass, so applying the scale a second time after retrieving embeddings drove the training loss to 62. The join now retrieves embeddings without rescaling. The overfitting check on a single fixed batch that follows is the cheapest available proof that the optimisation path is connected: a loss that falls towards zero on one repeated batch establishes that gradients reach the trainable parameters.

In [ ]:
# Attach LoRA adapters to BioGPT-Large. All base weights remain frozen.
!pip install peft --quiet

import torch
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

decoder = AutoModelForCausalLM.from_pretrained("microsoft/BioGPT-Large")

lora_cfg = LoraConfig(
    task_type      = TaskType.CAUSAL_LM,
    r              = 8,                      # rank: the proposal's r=8, drop to 4 on Viper if VRAM is tight
    lora_alpha     = 16,                     # scales the adapter update, standard choice of 2*r
    lora_dropout   = 0.05,
    target_modules = ["q_proj", "v_proj"],   # query and value projections, the LoRA paper's recommendation
)

decoder = get_peft_model(decoder, lora_cfg)
decoder.print_trainable_parameters()

In [ ]:
# Join the visual prefix to the report embeddings under a single loss.
import math, torch
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)                       # sanity check only. on Viper set "cuda"

torch.manual_seed(0)
qformer = QFormer(img_dim=512, decoder_hidden=HID, n_queries=32).to(device)
decoder.to(device).eval()

cfg = decoder.config
embed_scale = math.sqrt(cfg.hidden_size) if getattr(cfg, "scale_embedding", False) else 1.0
emb = decoder.get_input_embeddings()
# embed_scale is printed for the record only. It is not applied, since the embedding applies it.
print(f"scale_embedding={getattr(cfg,'scale_embedding',None)}  embed_scale={embed_scale:.2f} (already applied inside the embedding, not re-applied here)")

b_feat, b_key, b_ids, b_attn = next(iter(
    DataLoader(ReportSet(fit_frame), batch_size=2, shuffle=True, collate_fn=collate)))
CAP = 192
b_ids, b_attn = b_ids[:, :CAP], b_attn[:, :CAP]
b_feat, b_key = b_feat.to(device), b_key.to(device)
b_ids,  b_attn = b_ids.to(device), b_attn.to(device)
B = b_ids.shape[0]

prefix     = qformer(b_feat, key_padding_mask=b_key)      # (B, 32, HID) trainable
tok_embeds = emb(b_ids)                                   # (B, T, HID)  ALREADY scaled, no ×40
inputs_embeds = torch.cat([prefix, tok_embeds], dim=1)

prefix_attn = torch.ones(B, prefix.shape[1], dtype=b_attn.dtype, device=device)
attn_full   = torch.cat([prefix_attn, b_attn], dim=1)

prefix_lbl = torch.full((B, prefix.shape[1]), -100, dtype=torch.long, device=device)
tok_lbl    = b_ids.clone()
tok_lbl[b_attn == 0] = -100
labels     = torch.cat([prefix_lbl, tok_lbl], dim=1)

print(f"token-embed std (already scaled): {tok_embeds.std().item():.3f}")
print(f"Q-Former prefix std            : {prefix.std().item():.3f}")
print(f"chance loss ln(vocab)          : {math.log(cfg.vocab_size):.2f}  (vocab {cfg.vocab_size})")

out  = decoder(inputs_embeds=inputs_embeds, attention_mask=attn_full, labels=labels)
loss = out.loss
loss.backward()

q_grad    = qformer.queries.grad is not None and torch.isfinite(qformer.queries.grad).all()
lora_p    = next(p for n, p in decoder.named_parameters() if "lora" in n.lower() and p.requires_grad)
lora_grad = lora_p.grad is not None and torch.isfinite(lora_p.grad).all()

print(f"\nLOSS: {loss.item():.3f}")
print(f"gradients reach Q-Former queries: {bool(q_grad)}")
print(f"gradients reach LoRA adapters   : {bool(lora_grad)}")
print("verdict:", "OBJECTIVE SANE, ready to scale" if loss.item() < 20 else "STILL high, look at the std lines")

In [ ]:
# Overfit a single fixed batch to establish that gradients reach the trainable parameters.
import torch
from torch.utils.data import DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

torch.manual_seed(0)
qformer = QFormer(img_dim=512, decoder_hidden=HID, n_queries=32).to(device)
decoder.to(device).eval()            # dropout off so the overfit signal is clean
emb = decoder.get_input_embeddings()

# A single batch, held constant across every step.
b_feat, b_key, b_ids, b_attn = next(iter(
    DataLoader(ReportSet(fit_frame), batch_size=2, shuffle=True, collate_fn=collate)))
CAP = 128
b_ids, b_attn = b_ids[:, :CAP], b_attn[:, :CAP]
b_feat, b_key = b_feat.to(device), b_key.to(device)
b_ids,  b_attn = b_ids.to(device), b_attn.to(device)
B = b_ids.shape[0]

prefix_attn = torch.ones(B, 32, dtype=b_attn.dtype, device=device)
attn_full   = torch.cat([prefix_attn, b_attn], dim=1)
prefix_lbl  = torch.full((B, 32), -100, dtype=torch.long, device=device)
tok_lbl     = b_ids.clone(); tok_lbl[b_attn == 0] = -100
labels      = torch.cat([prefix_lbl, tok_lbl], dim=1)

# The optimiser covers only the trainable parameters: the bridge and the adapters.
params = [p for p in qformer.parameters() if p.requires_grad] + \
         [p for p in decoder.parameters() if p.requires_grad]
print(f"trainable tensors: {len(params)}  |  params: {sum(p.numel() for p in params)/1e6:.1f}M")
opt = torch.optim.AdamW(params, lr=5e-4)

for step in range(41):
    opt.zero_grad()
    prefix = qformer(b_feat, key_padding_mask=b_key)
    inputs_embeds = torch.cat([prefix, emb(b_ids)], dim=1)
    loss = decoder(inputs_embeds=inputs_embeds, attention_mask=attn_full, labels=labels).loss
    loss.backward()
    opt.step()
    if step % 5 == 0:
        print(f"step {step:2d}   loss {loss.item():.3f}")

print("\nverdict:", "LEARNS, optimisation path correct" if loss.item() < 1.0 else "FLAT, something is detached")

## 10. Caching and evaluation split

Volume caching is moved into a detached subprocess because the notebook kernel cannot background a shell job, and because a kernel restart would otherwise terminate a run measured in hours. The manifest makes the job resumable, so an interrupted run costs at most the volumes written since the last flush.

The evaluation split is locked before anything trains. The dataset ships train and validation partitions only, so validation serves as both development and test set, and two constraints make that defensible. The split is made by patient rather than by volume, since reconstructions and repeat scans of one patient share anatomy and would otherwise leak across the boundary. The test patients are then fixed with a recorded seed, so that the training pool can continue to grow as caching completes without any test study ever moving.

In [ ]:
# Launch volume caching as a detached subprocess.
script = r'''
import os, gc, shutil, tarfile, tempfile
import numpy as np, pandas as pd, torch
from huggingface_hub import hf_hub_download
from monai.transforms import (Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
                              Spacingd, ScaleIntensityRanged, MaskIntensityd, Lambdad,
                              Lambda, Resize)

SCRATCH_ROOT   = "/hpc/scratch/964931"
OUT_DIR        = os.path.join(SCRATCH_ROOT, "preproc_cache")
MANIFEST       = os.path.join(OUT_DIR, "manifest.csv")
TRAIN_CSV      = os.path.expanduser("~/stage_a_train.csv")
MAX_CACHE_GB   = 400          # scratch has ~1 TB; 2501 volumes land near 4 GB
SAFETY_FREE_GB = 50           # floor on scratch, not on home

def align_mask_to_image(data):
    img, msk = data["image"], data["mask"]
    if img.shape != msk.shape:
        msk = Resize(spatial_size=img.shape[1:], mode="nearest")(msk)
    msk.affine = img.affine.clone()
    data["mask"] = msk
    return data

preprocess = Compose([
    LoadImaged(keys=["image", "mask"]),
    EnsureChannelFirstd(keys=["image", "mask"]),
    Lambda(align_mask_to_image),
    Orientationd(keys=["image", "mask"], axcodes="RAS"),
    Spacingd(keys=["image", "mask"], pixdim=(1.5, 1.5, 2.0), mode=("bilinear", "nearest")),
    Lambdad(keys=["mask"], func=lambda m: (m > 0).float()),
    ScaleIntensityRanged(keys=["image"], a_min=-1000, a_max=400, b_min=0.0, b_max=1.0, clip=True),
    MaskIntensityd(keys=["image"], mask_key="mask"),
])

def ctrate_path(volname):
    split, patient, scan, _ = volname.replace(".nii.gz", "").split("_")
    base = f"{split}_{patient}"
    return f"dataset/{split}_fixed/{base}/{base}_{scan}/{volname}"

os.makedirs(OUT_DIR, exist_ok=True)
TMP  = tempfile.mkdtemp(prefix="radgenome_", dir=SCRATCH_ROOT)
want = set(pd.read_csv(TRAIN_CSV)["Volumename"])
rows = pd.read_csv(MANIFEST).to_dict("records") if os.path.exists(MANIFEST) else []
seen = {r["volumename"] for r in rows}

free_gb  = lambda: shutil.disk_usage(OUT_DIR).free / 1e9
cache_gb = lambda: sum(os.path.getsize(os.path.join(OUT_DIR, f))
                       for f in os.listdir(OUT_DIR)) / 1e9

print(f"targeting {len(want)} train volumes | {len(seen)} already logged", flush=True)
arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                      filename="dataset/train_region_mask.tar.gz")
print("mask archive ready:", arc, flush=True)

done = 0
with tarfile.open(arc, "r:gz") as tar:
    for m in tar:
        if not m.name.endswith("/lung.nii.gz"):
            continue
        vol  = m.name.split("/")[-2].replace("seg_", "") + ".nii.gz"
        stem = vol.replace(".nii.gz", "")
        if vol not in want or vol in seen:
            continue
        if cache_gb() >= MAX_CACHE_GB or free_gb() <= SAFETY_FREE_GB:
            print("disk limit hit, stopping clean.", flush=True); break

        msk_path = os.path.join(TMP, f"{stem}_lung.nii.gz")
        with open(msk_path, "wb") as g:
            g.write(tar.extractfile(m).read())
        vol_path = ""
        try:
            vol_path = hf_hub_download("ibrahimhamamci/CT-RATE", repo_type="dataset",
                                       filename=ctrate_path(vol), local_dir=TMP)
            out  = preprocess({"image": vol_path, "mask": msk_path})
            lung = int((out["mask"] > 0).sum())
            if lung < 1000:
                rows.append(dict(volumename=vol, stem=stem, split="train", status="fail",
                                 error="empty_mask", lung_voxels=lung, size_mb=0))
            else:
                op = os.path.join(OUT_DIR, f"{stem}.npz")
                np.savez_compressed(op,
                    image=out["image"][0].cpu().numpy().astype(np.float16),
                    affine=out["image"].affine.cpu().numpy().astype(np.float32))
                rows.append(dict(volumename=vol, stem=stem, split="train", status="ok",
                                 error="", lung_voxels=lung,
                                 size_mb=round(os.path.getsize(op)/1e6, 2)))
                done += 1
        except Exception as e:
            rows.append(dict(volumename=vol, stem=stem, split="train", status="fail",
                             error=str(e)[:200], lung_voxels=0, size_mb=0))
        finally:
            for f in (msk_path, vol_path):
                if f and os.path.exists(f):
                    os.remove(f)
            gc.collect()
        # flush every 25 so a walltime kill costs at most 25 volumes
        if done and done % 25 == 0:
            pd.DataFrame(rows).to_csv(MANIFEST, index=False)
            print(f"  {done} cached | {round(cache_gb(),1)} GB | free {round(free_gb(),1)} GB", flush=True)

pd.DataFrame(rows).to_csv(MANIFEST, index=False)
shutil.rmtree(TMP, ignore_errors=True)
print(f"FINISHED. {done} cached this run.", flush=True)
'''

with open(os.path.expanduser("~/cache_train.py"), "w") as f:
    f.write(script)

# The kernel does not support shell backgrounding, so the job is launched as a subprocess.
# start_new_session places it in its own process group, which prevents a kernel restart from
# terminating it.
import subprocess, sys, os

log  = open(os.path.expanduser("~/cache_train.log"), "a")
proc = subprocess.Popen([sys.executable, os.path.expanduser("~/cache_train.py")],
                        stdout=log, stderr=subprocess.STDOUT,
                        start_new_session=True)
print("caching launched, pid", proc.pid)
print("progress:  !tail -20 ~/cache_train.log")

# This survives a kernel restart but not the end of the session. For longer runs:
#   !sbatch --time=24:00:00 --mem=32G --cpus-per-task=4 --wrap="python $HOME/cache_train.py"

In [ ]:
!tail -20 ~/cache_train.log
!ps -p 4086

In [ ]:
# Lock the evaluation split by patient, with a recorded seed.
import os, numpy as np, pandas as pd

SCRATCH   = "/hpc/scratch/964931"
FEAT_DIR  = os.path.join(SCRATCH, "feat_cache_2d")
SPLIT_CSV = os.path.expanduser("~/splits_locked.csv")
SEED      = 42

def patient_of(stem):
    # valid_1002_a_1 maps to valid_1002, grouping reconstructions and repeat scans.
    return "_".join(stem.split("_")[:2])

val  = pd.read_csv(os.path.expanduser("~/stage_a_val.csv"))
trn  = pd.read_csv(os.path.expanduser("~/stage_a_train.csv"))
for f in (val, trn):
    f["stem"]    = f["Volumename"].str.replace(".nii.gz", "", regex=False)
    f["patient"] = f["stem"].map(patient_of)

if os.path.exists(SPLIT_CSV):
    splits = pd.read_csv(SPLIT_CSV)
    print("reusing locked split, numbers stay comparable across runs")
else:
    pats = np.array(sorted(val["patient"].unique()))
    rng  = np.random.default_rng(SEED); rng.shuffle(pats)
    n_test = int(0.40 * len(pats))          # ~40% of validation patients held out for test
    n_dev  = int(0.15 * len(pats))
    assign = ({p: "test" for p in pats[:n_test]} |
              {p: "dev"  for p in pats[n_test:n_test + n_dev]} |
              {p: "pool" for p in pats[n_test + n_dev:]})     # pool = extra training data
    val["fold"] = val["patient"].map(assign)
    trn["fold"] = "train"
    splits = pd.concat([val, trn], ignore_index=True)[["Volumename", "stem", "patient",
                                                       "report_text", "fold"]]
    splits.to_csv(SPLIT_CSV, index=False)
    print("split locked and written to", SPLIT_CSV)

# No patient may appear in two folds. This assertion is the leakage check.
overlap = splits.groupby("patient")["fold"].nunique()
assert (overlap == 1).all(), f"patient leakage in {(overlap > 1).sum()} patients"

# A study counts only once its features exist on disk.
splits["ready"] = splits["stem"].map(lambda s: os.path.exists(os.path.join(FEAT_DIR, f"{s}.npy")))
print("\ncached and usable per fold:")
print(splits.groupby("fold")["ready"].agg(["sum", "size"]))

# The training set is every available study that is not development or test, so studies join
# automatically as their features appear.
train_frame = splits[splits["ready"] & splits["fold"].isin(["train", "pool"])].reset_index(drop=True)
dev_frame   = splits[splits["ready"] & (splits["fold"] == "dev")].reset_index(drop=True)
test_frame  = splits[splits["ready"] & (splits["fold"] == "test")].reset_index(drop=True)
print(f"\ntrain {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}")
assert len(train_frame) > 0 and len(test_frame) > 0, "nothing cached, run cell A first"


## 11. Training and generation pipeline

The dataset, model and training loop are held constant across every arm reported in this notebook. Each encoder writes an array of shape (sequence, 512) into its own directory, so substituting one encoder for another substitutes a single path string and nothing else. That interface is what permits the later comparisons to be read as statements about encoders.

Generation is implemented directly rather than through the Hugging Face generation utility, because BioGPT does not accept precomputed input embeddings through that path on all library versions. The repetition penalty is not optional: report decoders trained on formulaic corpora collapse into repeated sentences without it.

In [ ]:
# Dataset and batching. Held constant across every arm.
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer

MAX_TOK = 384          # covers the 75th percentile report with headroom
tok = AutoTokenizer.from_pretrained("microsoft/BioGPT-Large")
PAD, EOS = tok.pad_token_id or 1, tok.eos_token_id or 2

class ReportSet(Dataset):
    def __init__(self, frame, feat_dir):
        self.frame, self.feat_dir = frame.reset_index(drop=True), feat_dir
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r = self.frame.iloc[i]
        f = torch.from_numpy(np.load(os.path.join(self.feat_dir, f"{r['stem']}.npy"))).float()
        ids = tok(r["report_text"], truncation=True, max_length=MAX_TOK - 1)["input_ids"] + [EOS]
        return f, torch.tensor(ids, dtype=torch.long), r["stem"]

def collate(batch):
    feats, ids, stems = zip(*batch)
    S, T, B = max(f.shape[0] for f in feats), max(len(x) for x in ids), len(batch)
    D = feats[0].shape[1]
    feat_pad = torch.zeros(B, S, D)
    key_pad  = torch.ones(B, S, dtype=torch.bool)      # True = padding the Q-Former ignores
    id_pad   = torch.full((B, T), PAD, dtype=torch.long)
    attn     = torch.zeros(B, T, dtype=torch.long)
    for b, (f, x) in enumerate(zip(feats, ids)):
        feat_pad[b, :f.shape[0]] = f; key_pad[b, :f.shape[0]] = False
        id_pad[b, :len(x)] = x;       attn[b, :len(x)] = 1
    return feat_pad, key_pad, id_pad, attn, list(stems)

def make_loader(frame, feat_dir, bs, shuffle):
    return DataLoader(ReportSet(frame, feat_dir), batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, num_workers=2, pin_memory=True)

_probe = next(iter(make_loader(train_frame, FEAT_DIR, 2, True)))
print("feats", tuple(_probe[0].shape), "| ids", tuple(_probe[2].shape))


In [ ]:
# Model definition: bridge and adapted decoder as one module.
import math, torch.nn as nn
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

class QFormer(nn.Module):
    """Learned queries cross-attend to the encoder sequence, then project to the
    decoder width. Replaces Sandeep's static soft prompt, which never looked at the
    image and so got ignored by the frozen decoder."""
    def __init__(self, img_dim, decoder_hidden, d_model=512, n_queries=32,
                 n_heads=8, n_layers=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.queries  = nn.Parameter(torch.randn(1, n_queries, d_model) * 0.02)
        self.img_proj = nn.Linear(img_dim, d_model)
        self.img_norm = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList(nn.ModuleDict(dict(
            sa=nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
            ca=nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
            ff=nn.Sequential(nn.Linear(d_model, d_model * ff_mult), nn.GELU(),
                             nn.Linear(d_model * ff_mult, d_model)),
            n1=nn.LayerNorm(d_model), n2=nn.LayerNorm(d_model), n3=nn.LayerNorm(d_model),
        )) for _ in range(n_layers))
        self.out_proj = nn.Linear(d_model, decoder_hidden)
        self.out_norm = nn.LayerNorm(decoder_hidden)

    def forward(self, img_feats, key_padding_mask=None):
        q  = self.queries.expand(img_feats.shape[0], -1, -1)
        kv = self.img_norm(self.img_proj(img_feats))
        for L in self.layers:
            h, _ = L["sa"](q, q, q);                          q = L["n1"](q + h)
            h, _ = L["ca"](q, kv, kv, key_padding_mask=key_padding_mask)
            q = L["n2"](q + h)
            q = L["n3"](q + L["ff"](q))
        return self.out_norm(self.out_proj(q))

class RegionReportModel(nn.Module):
    def __init__(self, img_dim=512, n_queries=32, lora_r=8, dtype=torch.bfloat16):
        super().__init__()
        self.decoder = AutoModelForCausalLM.from_pretrained(
            "microsoft/BioGPT-Large", torch_dtype=dtype)
        self.decoder.gradient_checkpointing_enable()          # trades compute for VRAM
        self.decoder.config.use_cache = False
        self.decoder = get_peft_model(self.decoder, LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=lora_r, lora_alpha=2 * lora_r,
            lora_dropout=0.05, target_modules=["q_proj", "v_proj"]))
        hid = self.decoder.config.hidden_size
        self.qformer = QFormer(img_dim=img_dim, decoder_hidden=hid)
        self.hid, self.dtype_ = hid, dtype

    def prefix(self, feats, key_pad):
        # The bridge runs in single precision for optimisation stability, then casts to the decoder dtype.
        return self.qformer(feats, key_padding_mask=key_pad).to(self.dtype_)

    def forward(self, feats, key_pad, ids, attn):
        p   = self.prefix(feats, key_pad)
        emb = self.decoder.get_input_embeddings()(ids)        # already scaled internally
        x   = torch.cat([p, emb], dim=1)
        a   = torch.cat([torch.ones(p.shape[:2], dtype=attn.dtype, device=attn.device), attn], 1)
        lbl = ids.clone(); lbl[attn == 0] = -100              # do not learn to predict padding
        lbl = torch.cat([torch.full(p.shape[:2], -100, dtype=torch.long, device=ids.device), lbl], 1)
        return self.decoder(inputs_embeds=x, attention_mask=a, labels=lbl).loss

    @torch.no_grad()
    def generate(self, feats, key_pad, max_new=320, rep_penalty=1.15):
        # Implemented directly rather than through the generation utility, because BioGPT does not
        # accept precomputed input embeddings through that path on all library versions. The
        # repetition penalty is required: report decoders collapse into repeated sentences without
        # it.
        self.eval()
        dev = feats.device
        cur = self.prefix(feats, key_pad)
        B   = cur.shape[0]
        attn = torch.ones(B, cur.shape[1], dtype=torch.long, device=dev)
        emb  = self.decoder.get_input_embeddings()
        past, out, alive = None, [], torch.ones(B, dtype=torch.bool, device=dev)
        for _ in range(max_new):
            o = self.decoder(inputs_embeds=cur, attention_mask=attn,
                             past_key_values=past, use_cache=True)
            past   = o.past_key_values
            logits = o.logits[:, -1, :].float()
            if out:
                prev = torch.stack(out, 1)
                s = torch.gather(logits, 1, prev)
                logits.scatter_(1, prev, torch.where(s < 0, s * rep_penalty, s / rep_penalty))
            nxt = logits.argmax(-1)
            nxt = torch.where(alive, nxt, torch.full_like(nxt, PAD))
            out.append(nxt)
            alive &= nxt != EOS
            if not alive.any():
                break
            cur  = emb(nxt.unsqueeze(1))
            attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=dev)], 1)
        seqs = torch.stack(out, 1)
        return [tok.decode([t for t in s.tolist() if t not in (PAD, EOS)],
                           skip_special_tokens=True) for s in seqs]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
model  = RegionReportModel(img_dim=_probe[0].shape[-1]).to(DEVICE)
n_tr   = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"device {DEVICE} | trainable {n_tr/1e6:.1f}M of {sum(p.numel() for p in model.parameters())/1e6:.0f}M")


In [ ]:
# Training loop. Effective batch 16 through gradient accumulation.
import time, json
from torch.optim.lr_scheduler import LambdaLR

RUN         = "stage_a_2d"
EPOCHS      = 6
BS, ACCUM   = 2, 8
LR          = 2e-4
CKPT        = os.path.join(SCRATCH, f"ckpt_{RUN}.pt")

train_loader = make_loader(train_frame, FEAT_DIR, BS, True)
dev_loader   = make_loader(dev_frame,   FEAT_DIR, BS, False)

params = [p for p in model.parameters() if p.requires_grad]
opt    = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
steps  = max(1, (len(train_loader) // ACCUM) * EPOCHS)
warm   = max(1, int(0.06 * steps))
sched  = LambdaLR(opt, lambda s: s / warm if s < warm else
                  0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))

def dev_loss():
    model.eval(); tot = n = 0
    with torch.no_grad():
        for f, k, i, a, _ in dev_loader:
            l = model(f.to(DEVICE), k.to(DEVICE), i.to(DEVICE), a.to(DEVICE))
            tot += l.item() * i.shape[0]; n += i.shape[0]
    return tot / max(n, 1)

hist, best = [], float("inf")
for ep in range(EPOCHS):
    model.train(); t0, run, seen = time.time(), 0.0, 0
    for step, (f, k, i, a, _) in enumerate(train_loader):
        loss = model(f.to(DEVICE), k.to(DEVICE), i.to(DEVICE), a.to(DEVICE)) / ACCUM
        loss.backward()
        run += loss.item() * ACCUM; seen += 1
        if (step + 1) % ACCUM == 0:
            torch.nn.utils.clip_grad_norm_(params, 1.0)
            opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        if seen % 100 == 0:
            print(f"  ep{ep} {seen}/{len(train_loader)} loss {run/seen:.3f}", flush=True)
    tr, dv = run / max(seen, 1), dev_loss()
    hist.append(dict(epoch=ep, train=tr, dev=dv))
    print(f"epoch {ep}  train {tr:.3f}  dev {dv:.3f}  ({time.time()-t0:.0f}s)")
    if dv < best:      # only the trainable parts need saving, the base model is frozen
        best = dv
        torch.save(dict(qformer=model.qformer.state_dict(),
                        lora={n: p for n, p in model.decoder.named_parameters() if "lora" in n},
                        epoch=ep, dev=dv), CKPT)
        print("  saved best ->", CKPT)

json.dump(hist, open(f"history_{RUN}.json", "w"), indent=2)

import matplotlib.pyplot as plt
plt.figure(figsize=(5, 3))
plt.plot([h["epoch"] for h in hist], [h["train"] for h in hist], marker="o", label="train")
plt.plot([h["epoch"] for h in hist], [h["dev"] for h in hist], marker="s", label="dev")
plt.xlabel("epoch"); plt.ylabel("cross-entropy"); plt.legend(); plt.tight_layout()
plt.savefig(f"loss_{RUN}.png", dpi=150); plt.show()


In [ ]:
!pip install --quiet nltk rouge-score bert-score sacrebleu

In [ ]:
print("bos", tok.bos_token_id, "| eos", tok.eos_token_id, "| pad", tok.pad_token_id)
_ids = tok("[RIGHT LUNG] It extends to the fissure.")["input_ids"]
print("first/last ids:", _ids[:5], _ids[-3:])
print("as tokens     :", tok.convert_ids_to_tokens(_ids[:5]))

f, k, i, a, st = next(iter(make_loader(test_frame, FEAT_DIR, 2, False)))
p = model.prefix(f.to(DEVICE), k.to(DEVICE))
o = model.decoder(inputs_embeds=p,
                  attention_mask=torch.ones(p.shape[:2], dtype=torch.long, device=DEVICE))
top = o.logits[:, -1, :].topk(5)
print("\ntop-5 first tokens:", tok.convert_ids_to_tokens(top.indices[0].tolist()))

In [ ]:
BOS = tok("test")["input_ids"][0]     # whatever the tokeniser actually prepends

In [ ]:
import types

BOS = tok("test")["input_ids"][0]        # the token the tokeniser actually prepends
print("BOS is", BOS, tok.convert_ids_to_tokens([BOS]))

@torch.no_grad()
def generate(self, feats, key_pad, max_new=320, rep_penalty=1.15, min_len=100):
    # The decoding length floor is raised from 12 to 100, matching the reference median. At 12
    # the decoder terminated after one section, predictions averaged 22 tokens against 118, and
    # a brevity penalty of 0.0126 suppressed every score. This is a decoding constraint on a
    # fixed model, not retraining.
    self.eval()
    dev, prefix = feats.device, self.prefix(feats, key_pad)
    B, emb = prefix.shape[0], self.decoder.get_input_embeddings()
    bos  = torch.full((B, 1), BOS, dtype=torch.long, device=dev)
    cur  = torch.cat([prefix, emb(bos)], dim=1)
    attn = torch.ones(B, cur.shape[1], dtype=torch.long, device=dev)
    past, out, alive = None, [], torch.ones(B, dtype=torch.bool, device=dev)
    for step in range(max_new):
        o = self.decoder(inputs_embeds=cur, attention_mask=attn,
                         past_key_values=past, use_cache=True)
        past   = o.past_key_values
        logits = o.logits[:, -1, :].float()
        logits[:, PAD] = -1e9
        if BOS != EOS:
            logits[:, BOS] = -1e9
        if step < min_len:
            logits[:, EOS] = -1e9
        if out:
            prev = torch.stack(out, 1)
            s = torch.gather(logits, 1, prev)
            logits.scatter_(1, prev, torch.where(s < 0, s * rep_penalty, s / rep_penalty))
        nxt = torch.where(alive, logits.argmax(-1),
                          torch.full((B,), PAD, device=dev, dtype=torch.long))
        out.append(nxt)
        alive &= nxt != EOS
        if not alive.any():
            break
        cur  = emb(nxt.unsqueeze(1))
        attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=dev)], 1)
    seqs = torch.stack(out, 1)
    return [tok.decode([t for t in s.tolist() if t not in (PAD, EOS, BOS)],
                       skip_special_tokens=True) for s in seqs]

model.generate = types.MethodType(generate, model)
print("patched: BOS fixed, min_len=100")

In [ ]:
!cp metrics_stage_a_2d.json metrics_stage_a_2d_minlen12.json
!cp preds_stage_a_2d.csv   preds_stage_a_2d_minlen12.csv
!ls -la *minlen12*

In [ ]:
# Generate on the held-out test set from the best development checkpoint.
import pandas as pd

ck = torch.load(CKPT, map_location=DEVICE)
model.qformer.load_state_dict(ck["qformer"])
missing = model.decoder.load_state_dict(ck["lora"], strict=False)
print(f"loaded epoch {ck['epoch']} (dev {ck['dev']:.3f})")

model.decoder.config.use_cache = True          # KV cache on for decoding
model.decoder.gradient_checkpointing_disable()

preds, refs, stems = [], [], []
test_loader = make_loader(test_frame, FEAT_DIR, 4, False)
for n, (f, k, i, a, st) in enumerate(test_loader):
    preds += model.generate(f.to(DEVICE), k.to(DEVICE))
    refs  += [tok.decode([t for t in row.tolist() if t != PAD], skip_special_tokens=True)
              for row in i]
    stems += st
    if n % 10 == 0:
        print(f"  {len(preds)}/{len(test_frame)} generated", flush=True)

out = pd.DataFrame(dict(stem=stems, reference=refs, prediction=preds))
out.to_csv(f"preds_{RUN}.csv", index=False)
print("\nEXAMPLE\nREF:", refs[0][:400], "\n\nGEN:", preds[0][:400])


In [ ]:
import sys
print("kernel python:", sys.executable)
!{sys.executable} -m pip install nltk rouge-score bert-score sacrebleu

In [ ]:
import nltk, rouge_score, bert_score
print("ok")

In [ ]:
!pip install --quiet nltk rouge-score bert-score sacrebleu

In [ ]:
# Metrics: the set committed to in the proposal, plus two structural measures.
import re, nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

df = pd.read_csv(f"preds_{RUN}.csv").fillna("")
P  = [str(x) for x in df["prediction"]]
R  = [str(x) for x in df["reference"]]

tokz = lambda s: re.findall(r"[a-z0-9]+", s.lower())
hyp, ref = [tokz(p) for p in P], [[tokz(r)] for r in R]
sm = SmoothingFunction().method1          # smoothing stated explicitly, BLEU-4 hits zeros otherwise

bleu = {f"BLEU-{n}": corpus_bleu(ref, hyp, weights=tuple([1/n]*n), smoothing_function=sm)
        for n in (1, 2, 3, 4)}

rs = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
sc = [rs.score(r, p) for r, p in zip(R, P)]
rouge = {f"ROUGE-{k[-1].upper()}": np.mean([s[k].fmeasure for s in sc])
         for k in ("rouge1", "rouge2", "rougeL")}

meteor = np.mean([meteor_score([tokz(r)], tokz(p)) for r, p in zip(R, P)])

from bert_score import score as bertscore
_, _, F1 = bertscore(P, R, lang="en", rescale_with_baseline=True, verbose=False)

# Structural fidelity. The task is structured reporting, so section identity and ordering
# constitute a claim that n-gram overlap cannot make on its own.
secs = lambda s: re.findall(r"\[([A-Z ]+)\]", s)
f1s, ords = [], []
for r, p in zip(R, P):
    a, b = secs(r), secs(p)
    inter = len(set(a) & set(b))
    prec  = inter / len(set(b)) if b else 0.0
    rec   = inter / len(set(a)) if a else 0.0
    f1s.append(0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec))
    keep = [x for x in b if x in set(a)]
    ords.append(float(keep == sorted(keep, key=lambda x: a.index(x))) if keep else 0.0)

results = {**bleu, **rouge, "METEOR": float(meteor), "BERTScore-F1": float(F1.mean()),
           "SectionF1": float(np.mean(f1s)), "SectionOrder": float(np.mean(ords)),
           "n_test": len(df)}
for k, v in results.items():
    print(f"{k:16s} {v:.4f}" if isinstance(v, float) else f"{k:16s} {v}")
json.dump(results, open(f"metrics_{RUN}.json", "w"), indent=2)


In [ ]:
import json, shutil, os, pandas as pd

# The run is frozen so that later cells cannot overwrite it.
for f in ["metrics_stage_a_2d.json", "preds_stage_a_2d.csv"]:
    shutil.copy(f, f.replace("stage_a_2d", "stage_a_2d_FINAL"))
shutil.copy(os.path.join(SCRATCH, "ckpt_stage_a_2d.pt"),
            os.path.join(SCRATCH, "ckpt_stage_a_2d_FINAL.pt"))
print("frozen:", [f for f in os.listdir(".") if "FINAL" in f])

# Progress of the training cache.
m = pd.read_csv(os.path.join(SCRATCH, "preproc_cache", "manifest.csv"))
print("\n", m.groupby(["split", "status"]).size(), sep="")

In [ ]:
m  = pd.read_csv(os.path.join(SCRATCH, "preproc_cache", "manifest.csv"))
tr = m[m["split"] == "train"]
print(tr["error"].value_counts().head(5))
print()
print(tr[["volumename", "error"]].head(2).to_string())

In [ ]:
import os, sys, subprocess, pandas as pd
from huggingface_hub import get_token

tokv = get_token()
print("token present:", bool(tokv))
assert tokv, "no token. run:  from huggingface_hub import login; login()  then rerun this cell"

# Failed training rows are cleared, otherwise the resume logic skips the entire cohort.
MANIFEST = os.path.join(SCRATCH, "preproc_cache", "manifest.csv")
m = pd.read_csv(MANIFEST)
keep = m[~((m["split"] == "train") & (m["status"] == "fail"))]
keep.to_csv(MANIFEST, index=False)
print(f"manifest: {len(m)} rows -> {len(keep)} (validation preserved)")

# Relaunched with the access token set explicitly in the child environment.
env = os.environ.copy()
env["HF_TOKEN"] = tokv
env["HUGGING_FACE_HUB_TOKEN"] = tokv

log  = open(os.path.expanduser("~/cache_train.log"), "a")
proc = subprocess.Popen([sys.executable, os.path.expanduser("~/cache_train.py")],
                        stdout=log, stderr=subprocess.STDOUT,
                        start_new_session=True, env=env)
print("relaunched, pid", proc.pid)

In [ ]:
import numpy as np, re, pandas as pd

df   = pd.read_csv(f"preds_{RUN}.csv").fillna("")
tokz = lambda s: re.findall(r"[a-z0-9]+", str(s).lower())
lp   = np.array([len(tokz(p)) for p in df["prediction"]])
lr   = np.array([len(tokz(r)) for r in df["reference"]])
ratio = lr.sum() / max(lp.sum(), 1)

print(f"pred length  mean {lp.mean():6.1f}  median {np.median(lp):5.0f}")
print(f"ref  length  mean {lr.mean():6.1f}  median {np.median(lr):5.0f}")
print(f"r/c ratio            {ratio:.2f}")
print(f"brevity penalty      {np.exp(1 - ratio):.4f}   <- multiplies every BLEU score")
print(f"empty predictions    {(lp == 0).sum()}")

secs = lambda s: re.findall(r"\[([A-Z ]+)\]", str(s))
print(f"sections per pred    {np.mean([len(secs(p)) for p in df['prediction']]):.2f}"
      f"  vs ref {np.mean([len(secs(r)) for r in df['reference']]):.2f}")

## 12. Volumetric arm

The volumetric arm reuses the split, bridge, adapters, schedule and test set of the two-dimensional arm, and changes only the feature directory it reads from.

One limitation constrains the interpretation of every result from this arm. The cached archive holds the lung-masked volume and its affine transform but not the 197 per-region segmentation masks, so regional features are pooled from the layer4 map into a fixed, anatomically ordered spatial grid rather than by segmentation label. The grid is axis aligned, so a query can specialise to a lobe position, but this is grid grounding rather than mask grounding, and the difference is material to any claim about region alignment.

In [ ]:
# Volumetric encoder, pooled into a fixed anatomically ordered grid.
import sys, torch, numpy as np
sys.path.append(os.path.expanduser("~/MedicalNet"))
from models import resnet

FEAT_DIR_3D = os.path.join(SCRATCH, "feat_cache_3d")
os.makedirs(FEAT_DIR_3D, exist_ok=True)
GRID = (4, 4, 3)                     # 48 tokens of 512 dims, comparable to the 2D slice count

m3d = resnet.resnet34(sample_input_D=64, sample_input_H=224, sample_input_W=224,
                      num_seg_classes=2, shortcut_type="A", no_cuda=(DEVICE != "cuda"))
sd  = torch.load(os.path.expanduser("~/MedicalNet/pretrain/resnet_34.pth"), map_location="cpu")
sd  = {k.replace("module.", ""): v for k, v in sd.get("state_dict", sd).items()}
rep = m3d.load_state_dict(sd, strict=False)
assert all("conv_seg" in k for k in rep.missing_keys), "pretrained backbone did not load"
m3d = m3d.eval().to(DEVICE)
for p in m3d.parameters():
    p.requires_grad = False

fmap = {}
m3d.layer4.register_forward_hook(lambda mod, i, o: fmap.__setitem__("x", o.detach()))

@torch.no_grad()
def encode_3d(npz_path):
    v = torch.from_numpy(np.load(npz_path)["image"]).float()
    if v.ndim == 3: v = v[None]
    v = v[None].to(DEVICE)                                        # (1,1,H,W,D)
    m3d(v)
    g = torch.nn.functional.adaptive_avg_pool3d(fmap["x"], GRID)  # (1,512,4,4,3)
    return g.flatten(2).transpose(1, 2)[0].cpu()                  # (48, 512) same interface as 2D

CACHE_DIR = os.path.join(SCRATCH, "preproc_cache")
todo = [s for s in pd.read_csv(SPLIT_CSV)["stem"]
        if os.path.exists(os.path.join(CACHE_DIR, f"{s}.npz"))
        and not os.path.exists(os.path.join(FEAT_DIR_3D, f"{s}.npy"))]
print(f"{len(todo)} volumes to encode in 3D")
for n, s in enumerate(todo):
    np.save(os.path.join(FEAT_DIR_3D, f"{s}.npy"),
            encode_3d(os.path.join(CACHE_DIR, f"{s}.npz")).numpy().astype(np.float16))
    if n % 100 == 0: print(f"  {n}/{len(todo)}", flush=True)
print("3D features done:", len(os.listdir(FEAT_DIR_3D)))


In [ ]:
# Volumetric arm: identical configuration, feature directory substituted.
FEAT_DIR = FEAT_DIR_3D
RUN      = "stage_b_3d"
CKPT     = os.path.join(SCRATCH, f"ckpt_{RUN}.pt")

splits["ready"] = splits["stem"].map(lambda s: os.path.exists(os.path.join(FEAT_DIR, f"{s}.npy")))
train_frame = splits[splits["ready"] & splits["fold"].isin(["train", "pool"])].reset_index(drop=True)
dev_frame   = splits[splits["ready"] & (splits["fold"] == "dev")].reset_index(drop=True)
test_frame  = splits[splits["ready"] & (splits["fold"] == "test")].reset_index(drop=True)
print(f"3D arm: train {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}")

model = RegionReportModel(img_dim=512).to(DEVICE)   # fresh weights, no carry-over from 2D

# The results table, once both arms have written metrics.
rows = []
for name, tag in [("Muralidharan 2025 (baseline)", None),
                  ("Ours, 2D ResNet-34 + Q-Former + LoRA", "stage_a_2d"),
                  ("Ours, Med3D + Q-Former + LoRA",        "stage_b_3d")]:
    if tag is None:
        rows.append(dict(System=name, **{"BLEU-4": 0.0108}))
    elif os.path.exists(f"metrics_{tag}.json"):
        rows.append(dict(System=name, **json.load(open(f"metrics_{tag}.json"))))
table = pd.DataFrame(rows).set_index("System")
table.to_csv("results_table.csv")
print(table.round(4).to_string())


#### Note on the table printed above: superseded, and not a report result

The table above belongs to the first pass of the volumetric arm, before the decoding length was made
common across arms in section 13.3. Two things in it appear nowhere in the report and should not be
read as results.

The row labelled `Muralidharan 2025 (baseline)` records a single BLEU-4 value from prior work on this
corpus. It was measured on a different split under tokenisation and smoothing that were not stated,
so it is not comparable with anything computed here. No baseline row of that kind appears in any table
in the report. Muralidharan (2025) is cited in the report as related work and as the source of the nine
lung-region anatomy paths and section names, and for nothing else.

The `Ours, 2D ResNet-34` row was produced at that arm's own tuned decoding length. Section 13.3 shows
that per-arm lengths change the BLEU ordering through the brevity penalty alone, so every number in the
report comes from the shared length of 120 tokens fixed in section 13.4.

In [ ]:
FEAT_2D = os.path.join(SCRATCH, "feat_cache_2d")
FEAT_3D = os.path.join(SCRATCH, "feat_cache_3d")

both = splits["stem"].map(lambda s: os.path.exists(os.path.join(FEAT_3D, f"{s}.npy"))
                                and os.path.exists(os.path.join(FEAT_2D, f"{s}.npy")))
splits["ready"] = both
train_frame = splits[splits["ready"] & splits["fold"].isin(["train", "pool"])].reset_index(drop=True)
dev_frame   = splits[splits["ready"] & (splits["fold"] == "dev")].reset_index(drop=True)
test_frame  = splits[splits["ready"] & (splits["fold"] == "test")].reset_index(drop=True)
print(f"matched cohort: train {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}")

## 13. Controlled encoder comparison

The remainder of the notebook is a single controlled study. Every arm shares one cohort, one patient-level split, one bridge configuration, one adapter configuration, one schedule and one test set. The only quantity that varies between arms is the frozen encoder, and later, the operator used to pool its feature map.

Three encoders are compared. ResNet-34 and VGG-16 are both two-dimensional ImageNet backbones ending at 512 channels with a 32-fold downsample, matched on everything except the presence of residual connections. Med3D is the volumetric encoder from section 12.

Two corrections to the volumetric path are recorded in the extraction cell and both changed the result materially. The volume was previously presented as (1, 1, H, W, D) while the pretrained kernels expect (N, C, D, H, W), so anatomy was being read on the wrong axis at the wrong scale. Intensities were previously supplied in the unit interval while MedicalNet was pretrained on z-scored volumes, which placed the input outside the distribution the pretrained weights were fitted to and removed most of the benefit of using a pretrained encoder at all.

In [ ]:
# Setup. Self-contained so that the section survives a kernel restart.
import os, re, gc, json, math, time, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

SCRATCH   = "/hpc/scratch/964931"
CACHE_DIR = os.path.join(SCRATCH, "preproc_cache")      # the preprocessed .npz volumes
SPLIT_CSV = os.path.expanduser("~/splits_locked.csv")   # patient level split, already locked
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"

# One feature directory per encoder. Every encoder writes the same (S, 512) interface, so
# substituting an encoder substitutes a single path string and nothing else.
FEAT = {
    "resnet34": os.path.join(SCRATCH, "feat_cache_2d"),
    "vgg16":    os.path.join(SCRATCH, "feat_cache_vgg16"),
    "med3d":    os.path.join(SCRATCH, "feat_cache_3d_fixed"),
}
for d in FEAT.values():
    os.makedirs(d, exist_ok=True)

from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("microsoft/BioGPT-Large")
PAD = tok.pad_token_id if tok.pad_token_id is not None else 1
EOS = tok.eos_token_id if tok.eos_token_id is not None else 2
BOS = tok("test")["input_ids"][0]      # whatever id the tokeniser actually prepends
MAX_TOK = 384                          # cap on the training target
MAX_NEW = 384                          # cap on generation, matched to the training cap

splits = pd.read_csv(SPLIT_CSV)
assert (splits.groupby("patient")["fold"].nunique() == 1).all(), "patient level leakage"
REPORT = dict(zip(splits["stem"], splits["report_text"]))   # stem -> full untruncated target

# QUICK runs one epoch on a reduced cohort to verify execution. No result from a QUICK run
# is reportable.
QUICK = False

n_npz = len([f for f in os.listdir(CACHE_DIR) if f.endswith(".npz")])
print("device:", DEVICE)
print("preprocessed volumes on disk:", n_npz)
for k, v in FEAT.items():
    print(f"  {k:9s} features cached: {len([f for f in os.listdir(v) if f.endswith('.npy')])}")
print("bos", BOS, "| eos", EOS, "| pad", PAD, "| QUICK", QUICK)

In [ ]:
# Feature extraction for the three encoders.
import torchvision

MEAN = torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1)
STD  = torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1)

# (D, H, W) reduced to 96 tokens over six axial bands. The lobes stack superior to inferior,
# so axial position carries region identity. This remains below the approximately 150 slice
# tokens produced by the two-dimensional arms.
GRID_3D = (6, 4, 4)

def _load_volume(npz_path):
    d = np.load(npz_path)
    v = torch.from_numpy(d["image"]).float()
    if v.ndim == 4:
        v = v[0]                                   # (H, W, D)
    return v

def _build_2d(name):
    if name == "resnet34":
        bb  = torchvision.models.resnet34(weights="IMAGENET1K_V1")
        net = nn.Sequential(*list(bb.children())[:-2])     # -> (B, 512, H/32, W/32)
    elif name == "vgg16":
        bb  = torchvision.models.vgg16(weights="IMAGENET1K_V1")
        net = bb.features                                  # -> (B, 512, H/32, W/32)
    else:
        raise ValueError(name)
    for p in net.parameters():
        p.requires_grad = False
    return net.eval().to(DEVICE)

@torch.no_grad()
def encode_2d(net, npz_path, slice_batch=16):
    img  = _load_volume(npz_path)
    lung = (img > 0).float()                       # the image was masked, so >0 is lung
    idx  = torch.where(lung.sum(dim=(0, 1)) > 0)[0]
    if len(idx) == 0:
        return None
    img, lung = img[..., idx], lung[..., idx]
    sl = img.permute(2, 0, 1).unsqueeze(1).repeat(1, 3, 1, 1)   # (d, 3, H, W)
    sl = (sl - MEAN) / STD
    lm = lung.permute(2, 0, 1).unsqueeze(1)                     # (d, 1, H, W)
    out = []
    for s in range(0, sl.shape[0], slice_batch):
        fb  = net(sl[s:s + slice_batch].to(DEVICE))
        mb  = F.interpolate(lm[s:s + slice_batch].to(DEVICE), size=fb.shape[-2:], mode="nearest")
        num = (fb * mb).sum(dim=(2, 3))
        den = mb.sum(dim=(2, 3)).clamp(min=1.0)
        out.append((num / den).float().cpu())
    return torch.cat(out, 0)                       # (lung slices, 512)

def _build_3d():
    import sys
    sys.path.append(os.path.expanduser("~/MedicalNet"))
    from models import resnet
    m  = resnet.resnet34(sample_input_D=64, sample_input_H=224, sample_input_W=224,
                         num_seg_classes=2, shortcut_type="A", no_cuda=(DEVICE != "cuda"))
    sd = torch.load(os.path.expanduser("~/MedicalNet/pretrain/resnet_34.pth"),
                    map_location="cpu", weights_only=False)
    sd = {k.replace("module.", ""): v for k, v in sd.get("state_dict", sd).items()}
    rep = m.load_state_dict(sd, strict=False)
    assert all("conv_seg" in k for k in rep.missing_keys), "pretrained 3D backbone did not load"
    for p in m.parameters():
        p.requires_grad = False
    return m.eval().to(DEVICE)

@torch.no_grad()
def encode_3d(net, hook, npz_path):
    v    = _load_volume(npz_path)                  # (H, W, D)
    lung = v > 0
    if int(lung.sum()) == 0:
        return None
    # Empty slices above and below the lung are dropped, as in the two-dimensional arms.
    # Otherwise the axial bands would allocate part of their budget to air and the two arms
    # would summarise different anatomical extents.
    idx  = torch.where(lung.sum(dim=(0, 1)) > 0)[0]
    v, lung = v[..., idx], lung[..., idx]
    mu = v[lung].mean()
    sd = v[lung].std().clamp(min=1e-6)
    v  = torch.where(lung, (v - mu) / sd, torch.zeros_like(v))   # correction 2
    v  = v.permute(2, 0, 1).contiguous()                         # correction 1 -> (D, H, W)
    net(v[None, None].to(DEVICE))
    g = F.adaptive_avg_pool3d(hook["x"].float(), GRID_3D)        # (1, 512, 6, 4, 4)
    return g.flatten(2).transpose(1, 2)[0].cpu()                 # (96, 512), same interface

def extract(name, limit=None):
    stems = sorted(s for s in splits["stem"]
                   if os.path.exists(os.path.join(CACHE_DIR, f"{s}.npz")))
    todo  = [s for s in stems if not os.path.exists(os.path.join(FEAT[name], f"{s}.npy"))]
    if limit:
        todo = todo[:limit]
    print(f"{name}: {len(stems)} volumes preprocessed, {len(todo)} still to encode", flush=True)
    if not todo:
        return
    if name == "med3d":
        net, hook = _build_3d(), {}
        net.layer4.register_forward_hook(lambda m, i, o: hook.__setitem__("x", o.detach()))
        fn = lambda p: encode_3d(net, hook, p)
    else:
        net = _build_2d(name)
        fn  = lambda p: encode_2d(net, p)
    t0 = time.time()
    for n, s in enumerate(todo):
        try:
            f = fn(os.path.join(CACHE_DIR, f"{s}.npz"))
        except RuntimeError as e:                  # almost always a VRAM spike on one big volume
            print("  failed:", s, str(e)[:120], flush=True)
            torch.cuda.empty_cache(); continue
        if f is None:
            print("  skipped, no lung voxels:", s, flush=True); continue
        np.save(os.path.join(FEAT[name], f"{s}.npy"), f.numpy().astype(np.float16))
        if n % 100 == 0:
            print(f"  {n}/{len(todo)}  ({time.time() - t0:.0f}s)", flush=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    print(f"{name} done: {len([f for f in os.listdir(FEAT[name]) if f.endswith('.npy')])} files\n",
          flush=True)

for arm in ["resnet34", "vgg16", "med3d"]:
    extract(arm)

In [ ]:
# Matched cohort: the intersection across all feature directories.
have   = {k: {f[:-4] for f in os.listdir(v) if f.endswith(".npy")} for k, v in FEAT.items()}
common = set.intersection(*have.values())
print("features per encoder:", {k: len(v) for k, v in have.items()})
print("matched intersection:", len(common))

splits["ready"] = splits["stem"].isin(common)
train_frame = splits[splits["ready"] & splits["fold"].isin(["train", "pool"])].reset_index(drop=True)
dev_frame   = splits[splits["ready"] & (splits["fold"] == "dev")].reset_index(drop=True)
test_frame  = splits[splits["ready"] & (splits["fold"] == "test")].reset_index(drop=True)

if QUICK:
    train_frame = train_frame.sample(min(60, len(train_frame)), random_state=SEED).reset_index(drop=True)
    dev_frame   = dev_frame.head(16).reset_index(drop=True)
    test_frame  = test_frame.head(24).reset_index(drop=True)

assert len(train_frame) and len(dev_frame) and len(test_frame), "a fold came out empty"
print(f"\ntrain {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}")
for f, nm in [(train_frame, "train"), (dev_frame, "dev"), (test_frame, "test")]:
    print(f"  {nm:5s} volumes {len(f):5d}  patients {f['patient'].nunique():5d}")

# No patient may appear in two folds, restated on the matched cohort.
_chk = pd.concat([train_frame.assign(f="tr"), dev_frame.assign(f="dv"), test_frame.assign(f="te")])
assert (_chk.groupby("patient")["f"].nunique() == 1).all(), "patient leakage in the matched cohort"

# Records the studies on which every reported number was computed.
_chk[["stem", "patient", "fold"]].to_csv("matched_cohort.csv", index=False)

# Proportion of the test reference truncated by the training token cap.
_len = np.array([len(tok(r)["input_ids"]) for r in test_frame["report_text"]])
print(f"\ntest reference length in tokens: median {np.median(_len):.0f}, "
      f"mean {_len.mean():.0f}, 95th {np.percentile(_len, 95):.0f}")
print(f"test references longer than the {MAX_TOK} token training cap: "
      f"{100 * (_len > MAX_TOK).mean():.1f}%")

In [ ]:
# Dataset and model. Identical for every arm.
from torch.utils.data import Dataset, DataLoader
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType

class ReportSet(Dataset):
    def __init__(self, frame, feat_dir):
        self.frame, self.feat_dir = frame.reset_index(drop=True), feat_dir
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, i):
        r   = self.frame.iloc[i]
        f   = torch.from_numpy(np.load(os.path.join(self.feat_dir, f"{r['stem']}.npy"))).float()
        ids = tok(r["report_text"], truncation=True, max_length=MAX_TOK - 1)["input_ids"] + [EOS]
        return f, torch.tensor(ids, dtype=torch.long), r["stem"]

def collate(batch):
    feats, ids, stems = zip(*batch)
    S, T, B = max(f.shape[0] for f in feats), max(len(x) for x in ids), len(batch)
    D = feats[0].shape[1]
    feat_pad = torch.zeros(B, S, D)
    key_pad  = torch.ones(B, S, dtype=torch.bool)     # True = padding the Q-Former ignores
    id_pad   = torch.full((B, T), PAD, dtype=torch.long)
    attn     = torch.zeros(B, T, dtype=torch.long)
    for b, (f, x) in enumerate(zip(feats, ids)):
        feat_pad[b, :f.shape[0]] = f; key_pad[b, :f.shape[0]] = False
        id_pad[b, :len(x)] = x;       attn[b, :len(x)] = 1
    return feat_pad, key_pad, id_pad, attn, list(stems)

def make_loader(frame, feat_dir, bs, shuffle):
    return DataLoader(ReportSet(frame, feat_dir), batch_size=bs, shuffle=shuffle,
                      collate_fn=collate, num_workers=2, pin_memory=True)

class QFormer(nn.Module):
    """Learned queries cross attend to the encoder sequence, then project to the decoder
    width. Replaces Sandeep's static soft prompt, which never looked at the image and so
    was ignored by the frozen decoder."""
    def __init__(self, img_dim, decoder_hidden, d_model=512, n_queries=32,
                 n_heads=8, n_layers=4, ff_mult=4, dropout=0.1):
        super().__init__()
        self.queries  = nn.Parameter(torch.randn(1, n_queries, d_model) * 0.02)
        self.img_proj = nn.Linear(img_dim, d_model)
        self.img_norm = nn.LayerNorm(d_model)
        self.layers = nn.ModuleList(nn.ModuleDict(dict(
            sa=nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
            ca=nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True),
            ff=nn.Sequential(nn.Linear(d_model, d_model * ff_mult), nn.GELU(),
                             nn.Linear(d_model * ff_mult, d_model)),
            n1=nn.LayerNorm(d_model), n2=nn.LayerNorm(d_model), n3=nn.LayerNorm(d_model),
        )) for _ in range(n_layers))
        self.out_proj = nn.Linear(d_model, decoder_hidden)
        self.out_norm = nn.LayerNorm(decoder_hidden)

    def forward(self, img_feats, key_padding_mask=None):
        q  = self.queries.expand(img_feats.shape[0], -1, -1)
        kv = self.img_norm(self.img_proj(img_feats))
        for L in self.layers:
            h, _ = L["sa"](q, q, q);  q = L["n1"](q + h)
            h, _ = L["ca"](q, kv, kv, key_padding_mask=key_padding_mask)
            q = L["n2"](q + h)
            q = L["n3"](q + L["ff"](q))
        return self.out_norm(self.out_proj(q))

class RegionReportModel(nn.Module):
    def __init__(self, img_dim=512, n_queries=32, lora_r=8, dtype=torch.bfloat16):
        super().__init__()
        self.decoder = AutoModelForCausalLM.from_pretrained(
            "microsoft/BioGPT-Large", torch_dtype=dtype)
        self.decoder.gradient_checkpointing_enable()      # trades compute for VRAM
        self.decoder.config.use_cache = False
        self.decoder = get_peft_model(self.decoder, LoraConfig(
            task_type=TaskType.CAUSAL_LM, r=lora_r, lora_alpha=2 * lora_r,
            lora_dropout=0.05, target_modules=["q_proj", "v_proj"]))
        hid = self.decoder.config.hidden_size
        self.qformer = QFormer(img_dim=img_dim, decoder_hidden=hid, n_queries=n_queries)
        self.hid, self.dtype_ = hid, dtype

    def prefix(self, feats, key_pad):
        # The bridge runs in single precision for optimisation stability, then casts to the decoder dtype.
        return self.qformer(feats, key_padding_mask=key_pad).to(self.dtype_)

    def forward(self, feats, key_pad, ids, attn):
        p   = self.prefix(feats, key_pad)
        emb = self.decoder.get_input_embeddings()(ids)    # BioGptScaledWordEmbedding already
        x   = torch.cat([p, emb], dim=1)                  # applies sqrt(hidden). do NOT rescale.
        a   = torch.cat([torch.ones(p.shape[:2], dtype=attn.dtype, device=attn.device), attn], 1)
        lbl = ids.clone(); lbl[attn == 0] = -100          # do not learn to predict padding
        lbl = torch.cat([torch.full(p.shape[:2], -100, dtype=torch.long, device=ids.device), lbl], 1)
        return self.decoder(inputs_embeds=x, attention_mask=a, labels=lbl).loss

    @torch.no_grad()
    def generate(self, feats, key_pad, max_new=MAX_NEW, rep_penalty=1.15, min_len=100):
        # Implemented directly rather than through the generation utility, because BioGPT does not
        # accept precomputed input embeddings through that path on all library versions. The length
        # floor is selected on the development set.
        self.eval()
        dev    = feats.device
        prefix = self.prefix(feats, key_pad)
        B, emb = prefix.shape[0], self.decoder.get_input_embeddings()
        bos    = torch.full((B, 1), BOS, dtype=torch.long, device=dev)
        cur    = torch.cat([prefix, emb(bos)], dim=1)
        attn   = torch.ones(B, cur.shape[1], dtype=torch.long, device=dev)
        past, out, alive = None, [], torch.ones(B, dtype=torch.bool, device=dev)
        for step in range(max_new):
            o = self.decoder(inputs_embeds=cur, attention_mask=attn,
                             past_key_values=past, use_cache=True)
            past   = o.past_key_values
            logits = o.logits[:, -1, :].float()
            logits[:, PAD] = -1e9
            if BOS != EOS:
                logits[:, BOS] = -1e9
            if step < min_len:
                logits[:, EOS] = -1e9
            if out:                                        # repetition penalty
                prev = torch.stack(out, 1)
                s = torch.gather(logits, 1, prev)
                logits.scatter_(1, prev, torch.where(s < 0, s * rep_penalty, s / rep_penalty))
            nxt = torch.where(alive, logits.argmax(-1),
                              torch.full((B,), PAD, device=dev, dtype=torch.long))
            out.append(nxt)
            alive &= nxt != EOS
            if not alive.any():
                break
            cur  = emb(nxt.unsqueeze(1))
            attn = torch.cat([attn, torch.ones(B, 1, dtype=torch.long, device=dev)], 1)
        seqs = torch.stack(out, 1)
        return [tok.decode([t for t in s.tolist() if t not in (PAD, EOS, BOS)],
                           skip_special_tokens=True) for s in seqs]

_probe = next(iter(make_loader(train_frame, FEAT["resnet34"], 2, False)))
print("probe feats", tuple(_probe[0].shape), "| ids", tuple(_probe[2].shape))
IMG_DIM = _probe[0].shape[-1]
assert IMG_DIM == 512, f"expected 512 dim features, got {IMG_DIM}"
print("model definition ready")

In [ ]:
# Training and evaluation harness. One function per stage, identical across arms.
!pip install --quiet nltk rouge-score bert-score sacremoses
import nltk
for pkg in ["punkt", "punkt_tab", "wordnet", "omw-1.4"]:
    try: nltk.download(pkg, quiet=True)
    except Exception: pass
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer
from bert_score import score as bertscore
from torch.optim.lr_scheduler import LambdaLR

EPOCHS      = 1 if QUICK else 6
BS, ACCUM   = 2, 8            # effective batch 16
LR          = 2e-4
GEN_BS      = 8               # generation only, no gradients, so it takes a bigger batch
MINLEN_GRID = [20, 100] if QUICK else [20, 70, 120]
DEV_SWEEP_N = 16 if QUICK else 64   # dev studies used for the min_len sweep. a subsample,
                                    # The sweep costs one full decode per grid point and the selection is stable
                                    # well before the full development set is used.

def train_arm(model, feat_dir, ckpt):
    train_loader = make_loader(train_frame, feat_dir, BS, True)
    dev_loader   = make_loader(dev_frame,   feat_dir, BS, False)
    params = [p for p in model.parameters() if p.requires_grad]
    opt    = torch.optim.AdamW(params, lr=LR, weight_decay=0.01)
    steps  = max(1, (len(train_loader) // ACCUM) * EPOCHS)
    warm   = max(1, int(0.06 * steps))
    sched  = LambdaLR(opt, lambda s: s / warm if s < warm else
                      0.5 * (1 + math.cos(math.pi * (s - warm) / max(1, steps - warm))))

    def dev_loss():
        model.eval(); tot = n = 0
        with torch.no_grad():
            for f, k, i, a, _ in dev_loader:
                l = model(f.to(DEVICE), k.to(DEVICE), i.to(DEVICE), a.to(DEVICE))
                tot += l.item() * i.shape[0]; n += i.shape[0]
        return tot / max(n, 1)

    hist, best = [], float("inf")
    for ep in range(EPOCHS):
        model.train(); t0, run, seen = time.time(), 0.0, 0
        opt.zero_grad(set_to_none=True)
        for step, (f, k, i, a, _) in enumerate(train_loader):
            loss = model(f.to(DEVICE), k.to(DEVICE), i.to(DEVICE), a.to(DEVICE)) / ACCUM
            loss.backward()
            run += loss.item() * ACCUM; seen += 1
            if (step + 1) % ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        tr, dv = run / max(seen, 1), dev_loss()
        hist.append(dict(epoch=ep, train=tr, dev=dv))
        print(f"  epoch {ep}  train {tr:.3f}  dev {dv:.3f}  ({time.time() - t0:.0f}s)", flush=True)
        if dv < best:
            best = dv
            torch.save(dict(qformer={n: p.detach().cpu().clone()
                                     for n, p in model.qformer.state_dict().items()},
                            lora={n: p.detach().cpu().clone()
                                  for n, p in model.decoder.named_parameters() if "lora" in n},
                            epoch=ep, dev=dv), ckpt)
            print("    saved best ->", ckpt, flush=True)
    return hist

def load_trainables(model, ckpt):
    """Loads the Q-Former and the LoRA adapters, and proves both actually landed.
    strict=False on its own hides a silent failure: if the peft key names ever shift you
    end up evaluating a base BioGPT with a trained bridge, and the output still reads as
    plausible English, so nothing visibly breaks. Three checks close that hole:
      1. no key in the checkpoint is unknown to the model,
      2. every checkpoint tensor is byte for byte present in the model after the load,
      3. the LoRA B matrices are non zero, which they only become through training
         (peft initialises B to zeros, so all zeros means an untrained adapter)."""
    ck = torch.load(ckpt, map_location="cpu", weights_only=False)
    model.qformer.load_state_dict(ck["qformer"])          # strict by default

    rep = model.decoder.load_state_dict({k: v.to(DEVICE) for k, v in ck["lora"].items()},
                                        strict=False)
    assert not rep.unexpected_keys, f"checkpoint holds keys the model lacks: {rep.unexpected_keys[:5]}"
    landed = [k for k in ck["lora"] if k not in set(rep.missing_keys)]
    assert len(landed) == len(ck["lora"]), \
        f"only {len(landed)} of {len(ck['lora'])} LoRA tensors landed"

    live = dict(model.decoder.named_parameters())
    for k, v in ck["lora"].items():
        assert torch.equal(live[k].detach().float().cpu(), v.float()), \
            f"{k} is in the model but does not match the checkpoint"

    nz = sum(1 for n, p in live.items()
             if "lora_B" in n and float(p.detach().float().abs().sum()) > 0)
    assert nz > 0, "every LoRA B matrix is still zero, the adapters were never trained"
    print(f"  loaded epoch {ck['epoch']} (dev {ck['dev']:.3f}); "
          f"{len(landed)} LoRA tensors verified, {nz} non zero B matrices")

@torch.no_grad()
def generate_on(model, feat_dir, frame, min_len, bs=GEN_BS, verbose=False):
    model.decoder.config.use_cache = True
    model.decoder.gradient_checkpointing_disable()
    preds, stems = [], []
    for n, (f, k, i, a, st) in enumerate(make_loader(frame, feat_dir, bs, False)):
        preds += model.generate(f.to(DEVICE), k.to(DEVICE), min_len=min_len)
        stems += st
        if verbose and n % 20 == 0:
            print(f"    {len(preds)}/{len(frame)}", flush=True)
    refs = [REPORT[s] for s in stems]          # the real report, not the truncated tokens
    return preds, refs, stems

TOKZ = lambda s: re.findall(r"[a-z0-9]+", str(s).lower())
SECS = lambda s: re.findall(r"\[([A-Z ]+)\]", str(s))
_SM  = SmoothingFunction().method1

def bleu4(preds, refs, smooth=True):
    hyp = [TOKZ(p) for p in preds]; ref = [[TOKZ(r)] for r in refs]
    return corpus_bleu(ref, hyp, weights=(0.25,) * 4,
                       smoothing_function=_SM if smooth else None)

def score_all(preds, refs):
    hyp = [TOKZ(p) for p in preds]; ref = [[TOKZ(r)] for r in refs]
    out = {f"BLEU-{n}": corpus_bleu(ref, hyp, weights=tuple([1 / n] * n),
                                    smoothing_function=_SM) for n in (1, 2, 3, 4)}
    out["BLEU-4 unsmoothed"] = corpus_bleu(ref, hyp, weights=(0.25,) * 4)

    rs = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
    sc = [rs.score(r, p) for r, p in zip(refs, preds)]
    for k, nm in [("rouge1", "ROUGE-1"), ("rouge2", "ROUGE-2"), ("rougeL", "ROUGE-L")]:
        out[nm] = float(np.mean([s[k].fmeasure for s in sc]))

    out["METEOR"] = float(np.mean([meteor_score([TOKZ(r)], TOKZ(p))
                                   for r, p in zip(refs, preds)]))
    _, _, F1 = bertscore(list(preds), list(refs), lang="en",
                         rescale_with_baseline=True, verbose=False)
    out["BERTScore-F1"] = float(F1.mean())

    # Structural fidelity. The task is structured reporting, so section identity and ordering
    # constitute a claim that n-gram overlap cannot make on its own.
    f1s, ords = [], []
    for r, p in zip(refs, preds):
        a, b  = SECS(r), SECS(p)
        inter = len(set(a) & set(b))
        prec  = inter / len(set(b)) if b else 0.0
        rec   = inter / len(set(a)) if a else 0.0
        f1s.append(0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec))
        keep = [x for x in b if x in set(a)]
        if len(set(keep)) >= 2:                 # order is undefined below two sections, so
            ords.append(float(keep == sorted(keep, key=lambda x: a.index(x))))
    out["SectionF1"]    = float(np.mean(f1s))
    out["SectionOrder"] = float(np.mean(ords)) if ords else float("nan")
    out["SectionOrder n"] = len(ords)

    lp = np.array([len(TOKZ(p)) for p in preds]); lr = np.array([len(TOKZ(r)) for r in refs])
    out["pred len"] = float(lp.mean()); out["ref len"] = float(lr.mean())
    out["n_test"]   = len(preds)
    return out

def run_arm(name):
    """Full pipeline for one encoder: train, load best, choose min_len on dev,
    generate on test once, score. Returns the metrics dict."""
    print(f"\n{'=' * 70}\nARM: {name}\n{'=' * 70}", flush=True)
    RUN, feat_dir = f"arm_{name}", FEAT[name]
    ckpt = os.path.join(SCRATCH, f"ckpt_{RUN}.pt")

    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)   # identical init every arm
    model = RegionReportModel(img_dim=IMG_DIM).to(DEVICE)
    print(f"  trainable {sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6:.1f}M")

    hist = train_arm(model, feat_dir, ckpt)
    json.dump(hist, open(f"history_{RUN}.json", "w"), indent=2)
    load_trainables(model, ckpt)

    # The length floor is selected on the development set. An earlier run selected it against
    # test BLEU, which is selection on the test partition. The test set is used once, below.
    sweep_frame = dev_frame.head(DEV_SWEEP_N)
    print(f"  min_len sweep on {len(sweep_frame)} dev studies:", flush=True)
    sweep = {}
    for ml in MINLEN_GRID:
        p, r, _ = generate_on(model, feat_dir, sweep_frame, ml)
        sweep[ml] = bleu4(p, r)
        print(f"    min_len {ml:4d}  dev BLEU-4 {sweep[ml]:.4f}", flush=True)
    best_ml = max(sweep, key=sweep.get)
    print(f"  chosen min_len = {best_ml} (selected on dev, applied to test once)", flush=True)

    preds, refs, stems = generate_on(model, feat_dir, test_frame, best_ml, verbose=True)
    pd.DataFrame(dict(stem=stems, reference=refs, prediction=preds)).to_csv(
        f"preds_{RUN}.csv", index=False)

    m = score_all(preds, refs)
    m["min_len"]   = best_ml
    m["dev_sweep"] = {str(k): round(v, 4) for k, v in sweep.items()}
    m["best_dev_loss"] = min(h["dev"] for h in hist)
    json.dump(m, open(f"metrics_{RUN}.json", "w"), indent=2)
    for k, v in m.items():
        if isinstance(v, float):
            print(f"  {k:20s} {v:.4f}")
    del model; gc.collect(); torch.cuda.empty_cache()
    return m

print(f"harness ready. EPOCHS={EPOCHS}, min_len grid={MINLEN_GRID}, QUICK={QUICK}")

In [ ]:
# Arm 1: ResNet-34.
m_resnet = run_arm("resnet34")

In [ ]:
# Arm 2: VGG-16. Matched to ResNet-34 except for residual connections.
m_vgg = run_arm("vgg16")

In [ ]:
# Arm 3: Med3D, with the axis order and intensity normalisation corrected.
m_med3d = run_arm("med3d")

In [ ]:
# Comparative encoder table. The prior baseline is reported separately, not as a table row,
# because it was measured on a different split under unstated tokenisation and smoothing.
import matplotlib.pyplot as plt

ARMS = [("ResNet-34 (2D, residual)",  "arm_resnet34"),
        ("VGG-16 (2D, no residual)",  "arm_vgg16"),
        ("Med3D ResNet-34 (3D)",      "arm_med3d")]

rows = []
for label, tag in ARMS:
    p = f"metrics_{tag}.json"
    if not os.path.exists(p):
        print(f"MISSING: {p}, that arm has not run"); continue
    d = json.load(open(p))
    rows.append(dict(Encoder=label, **{k: v for k, v in d.items() if isinstance(v, (int, float))}))

assert len(rows) == len(ARMS), "not every arm produced metrics, do not read the table yet"

COLS = ["BLEU-1", "BLEU-2", "BLEU-3", "BLEU-4", "BLEU-4 unsmoothed",
        "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR", "BERTScore-F1",
        "SectionF1", "SectionOrder", "min_len", "n_test"]
table = pd.DataFrame(rows).set_index("Encoder")[COLS]
table.to_csv("results_encoder_comparison.csv")

print("Table: encoder comparison, matched cohort, identical decoder and schedule")
print(table.round(4).to_string())
print(f"\nCohort: train {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}"
      f"  (patient level split, seed {SEED})")
print("\nContext, not a table row:")
print("  Muralidharan (2025), DenseNet-121 + soft prompt + frozen BioGPT-Large: BLEU-4 0.0108,")
print("  reported on a different split with unstated tokenisation and smoothing.")
print("  CT2Rep reports BLEU-4 in the 0.36 to 0.40 range on a related chest CT task.")

# A single figure, within the figure allowance.
fig, ax = plt.subplots(figsize=(7.2, 3.6))
mets  = ["BLEU-4", "ROUGE-L", "METEOR", "SectionF1"]
x     = np.arange(len(mets)); w = 0.26
for i, (label, tag) in enumerate(ARMS):
    ax.bar(x + (i - 1) * w, [table.loc[label, m] for m in mets], w, label=label)
ax.axhline(0.0108, ls="--", lw=1, c="grey")
ax.text(len(mets) - 0.5, 0.0108, " Muralidharan BLEU-4", va="bottom", ha="right", fontsize=8, c="grey")
ax.set_xticks(x); ax.set_xticklabels(mets); ax.set_ylabel("score")
ax.legend(fontsize=8, frameon=False)
for sp in ("top", "right"):
    ax.spines[sp].set_visible(False)
plt.tight_layout(); plt.savefig("fig_encoder_comparison.png", dpi=200); plt.show()
print("\nsaved results_encoder_comparison.csv and fig_encoder_comparison.png")

#### Note on the table printed above: superseded by the shared decoding length

This table was computed before the decoding length was made common. The `min_len` column shows 70 for
ResNet-34 and 120 for the other two arms, so those rows are not comparable with one another. Section
13.4 recomputes every arm at 120 tokens, and those are the values the report gives: BLEU-4 of 0.1267
for ResNet-34, 0.1348 for VGG-16 and 0.1058 for Med3D at seed 42.

Two further items here do not carry into the report. `SectionOrder` reaches 1.0000 for VGG-16, the arm
producing the fewest unique outputs, so it rewards template repetition rather than structural fidelity
and contradicts the diversity evidence in section 13.3. It appears in no table, figure or sentence of
the report. The printed line giving the prior BLEU-4 of 0.0108 is context only, for the reason stated
after the volumetric arm above, and the dashed line drawn at that value in the figure is not reproduced
in any report figure.

### 13.1 Qualitative inspection

Aggregate metrics do not establish what a model writes. The cells below rank test studies by sentence-level BLEU-4 for each arm and print the best, median and worst case, together with the sections most often omitted and most often invented.

In [ ]:
# Best, median and worst case per arm by sentence-level BLEU-4, with section-level errors.
from nltk.translate.bleu_score import sentence_bleu

for label, tag in ARMS:
    df = pd.read_csv(f"preds_{tag}.csv").fillna("")
    s  = [sentence_bleu([TOKZ(r)], TOKZ(p), weights=(0.25,) * 4, smoothing_function=_SM)
          for r, p in zip(df["reference"], df["prediction"])]
    df["sent_bleu4"] = s
    df = df.sort_values("sent_bleu4", ascending=False).reset_index(drop=True)
    df.to_csv(f"preds_{tag}_ranked.csv", index=False)

    print("=" * 78); print(label); print("=" * 78)
    for tagline, row in [("BEST", df.iloc[0]), ("MEDIAN", df.iloc[len(df) // 2]), ("WORST", df.iloc[-1])]:
        print(f"\n--- {tagline}  (sentence BLEU-4 {row['sent_bleu4']:.3f}, {row['stem']})")
        print("REF:", str(row["reference"])[:320].replace("\n", " | "))
        print("GEN:", str(row["prediction"])[:320].replace("\n", " | "))

    # Aggregated structural errors.
    miss, extra = [], []
    for r, p in zip(df["reference"], df["prediction"]):
        a, b = set(SECS(r)), set(SECS(p))
        miss  += list(a - b)
        extra += list(b - a)
    print("\nmost often MISSED sections :", pd.Series(miss).value_counts().head(5).to_dict())
    print("most often INVENTED sections:", pd.Series(extra).value_counts().head(5).to_dict())
    print(f"mean sections per prediction: {np.mean([len(SECS(p)) for p in df['prediction']]):.2f}"
          f"  vs reference {np.mean([len(SECS(r)) for r in df['reference']]):.2f}\n")

### 13.2 Control experiments

Overlap metrics cannot distinguish a model that reads an image from one that has learned what a chest CT report usually says, because on a corpus this formulaic the average report already overlaps substantially with any individual report. Two controls separate the two accounts.

Under the shuffled control each study receives another study's real features, deranged within fold and repaired wherever a pairing lands on the same patient. Only the correspondence between scan and report is destroyed; sequence lengths and feature statistics are unchanged. Under the zeros control each study receives an array of its own shape containing only zeros, so the bridge still emits a learned constant prefix and the decoder is measured on the language prior alone.

Interpretation bands are fixed before the results are seen: retention below 60 per cent of the real score indicates genuine visual conditioning, 60 to 85 per cent indicates partial conditioning, and above 85 per cent indicates that the score is substantially a language prior.

In [ ]:
# Shuffled and zeros controls.
CONTROL_ARMS = ["resnet34"]        # extended to the remaining encoders in a later section
CONTROL_SEED = 7

def build_control_dirs(base):
    src    = FEAT[base]
    sh_dir = src + "_shuffled"
    zr_dir = src + "_zeros"
    os.makedirs(sh_dir, exist_ok=True); os.makedirs(zr_dir, exist_ok=True)

    cohort = pd.concat([train_frame, dev_frame, test_frame])
    pat    = dict(zip(cohort["stem"], cohort["patient"]))
    rng    = np.random.default_rng(CONTROL_SEED)

    mapping = {}
    for fold, g in cohort.groupby("fold"):
        stems = list(g["stem"])
        order = [str(x) for x in rng.permutation(stems)]
        tgt   = {order[i]: order[(i + 1) % len(order)] for i in range(len(order))}
        # Pairings that land on the same patient are repaired. A study retaining its own features
        # is the same condition.
        for _ in range(200):
            bad = [s for s in tgt if pat[tgt[s]] == pat[s]]
            if not bad:
                break
            for s in bad:
                o = order[int(rng.integers(len(order)))]
                if o != s:
                    tgt[s], tgt[o] = tgt[o], tgt[s]
        assert not [s for s in tgt if pat[tgt[s]] == pat[s]], \
            f"same patient pairing survived in fold {fold}"
        mapping.update(tgt)

    n_link = n_zero = 0
    for s, t in mapping.items():
        dst = os.path.join(sh_dir, f"{s}.npy")
        if not os.path.exists(dst):
            os.symlink(os.path.join(src, f"{t}.npy"), dst); n_link += 1
        dz = os.path.join(zr_dir, f"{s}.npy")
        if not os.path.exists(dz):
            a = np.load(os.path.join(src, f"{s}.npy"))
            np.save(dz, np.zeros_like(a)); n_zero += 1

    FEAT[f"{base}_shuffled"] = sh_dir
    FEAT[f"{base}_zeros"]    = zr_dir
    print(f"{base}: {len(mapping)} studies deranged, {n_link} new links, {n_zero} new zero arrays")

    # Evidence that the derangement removed the original pairing.
    probe = list(mapping)[0]
    same  = sum(1 for s, t in mapping.items() if s == t)
    print(f"  example: {probe} now reads the features of {mapping[probe]}")
    print(f"  studies still holding their own features: {same} (must be 0)")
    assert same == 0

for base in CONTROL_ARMS:
    build_control_dirs(base)
    for cond in ["shuffled", "zeros"]:
        run_arm(f"{base}_{cond}")

# Control table.
print("\n" + "=" * 78)
print("CONTROL RESULTS")
print("=" * 78)
KEEP = ["BLEU-4", "ROUGE-L", "METEOR", "BERTScore-F1", "SectionF1", "pred len", "min_len"]
for base in CONTROL_ARMS:
    rows = {}
    for cond, tag in [("real",     f"arm_{base}"),
                      ("shuffled", f"arm_{base}_shuffled"),
                      ("zeros",    f"arm_{base}_zeros")]:
        p = f"metrics_{tag}.json"
        if os.path.exists(p):
            rows[cond] = {k: v for k, v in json.load(open(p)).items() if k in KEEP}
    ctrl = pd.DataFrame(rows).T[KEEP]
    ctrl.to_csv(f"results_control_{base}.csv")
    print(f"\n{base}")
    print(ctrl.round(4).to_string())

    real, shuf = ctrl.loc["real", "BLEU-4"], ctrl.loc["shuffled", "BLEU-4"]
    zero = ctrl.loc["zeros", "BLEU-4"] if "zeros" in ctrl.index else float("nan")
    keep_frac = shuf / real if real else float("nan")
    print(f"\n  shuffled retains {100 * keep_frac:.1f}% of real BLEU-4 "
          f"({shuf:.4f} of {real:.4f}); zero input scores {zero:.4f}")
    if keep_frac < 0.60:
        print("  READING: the image is doing real work. The drop under a broken image to "
              "report pairing is the evidence that the conditioning is genuine.")
    elif keep_frac < 0.85:
        print("  READING: partial conditioning. The image contributes, but a large share of "
              "the score is obtainable from the language prior alone. Report both numbers.")
    else:
        print("  READING: the image contributes little. The score is substantially a "
              "language prior over a formulaic corpus. This is the honest finding and it "
              "must lead the Discussion, not sit in the limitations.")

### 13.3 Length decomposition and output diversity

BLEU is the product of a brevity penalty and an n-gram precision. The arms select different decoding lengths on the development set, so they incur different brevity penalties, and any ordering between them must survive that factor being separated out.

Output diversity is measured against the reference corpus rather than against zero. These reports are formulaic by construction, so a high self-similarity among predictions carries information only in contrast with the same statistic computed on the references. The final subsection regenerates every arm at one shared decoding length derived from the pooled development sweeps, so that the encoder comparison is a comparison of encoders rather than of decoding lengths.

In [ ]:
# Length decomposition, output diversity, and regeneration at a shared decoding length.
from nltk.translate.bleu_score import sentence_bleu

RERUN_MATCHED_DECODING = True     # set to False to skip regeneration; the preceding subsections still run

ARM_KEYS = [(label, tag, tag.replace("arm_", "")) for label, tag in ARMS]

# Brevity penalty against n-gram precision.
def decompose(preds, refs):
    c  = sum(len(TOKZ(p)) for p in preds)
    r  = sum(len(TOKZ(x)) for x in refs)
    bp = 1.0 if c > r else math.exp(1 - r / max(c, 1))
    b4 = bleu4(preds, refs)
    return dict(**{"BLEU-4": b4, "brevity penalty": bp,
                   "precision only": b4 / bp if bp else float("nan"),
                   "pred tokens": c, "ref tokens": r, "ratio r/c": r / max(c, 1)})

rows = {}
for label, tag, key in ARM_KEYS:
    df = pd.read_csv(f"preds_{tag}.csv").fillna("")
    rows[label] = decompose(list(df["prediction"]), list(df["reference"]))
for extra in sorted(f for f in os.listdir(".")
                    if f.startswith("preds_arm_") and
                    ("_shuffled" in f or "_zeros" in f) and f.endswith(".csv")):
    df = pd.read_csv(extra).fillna("")
    rows["control: " + extra[len("preds_arm_"):-4]] = decompose(list(df["prediction"]),
                                                                list(df["reference"]))

dec = pd.DataFrame(rows).T
dec.to_csv("results_length_decomposition.csv")
print("L1. BLEU-4 decomposed. 'precision only' is BLEU with the length effect removed.")
print(dec.round(4).to_string())
print("\nIf the ranking on 'BLEU-4' differs from the ranking on 'precision only', then the")
print("headline ordering of the encoders is an artefact of report length, not of the encoder.")

# Output repetition, measured against the reference corpus.
def diversity(texts, n_pairs=400, seed=0):
    texts = [str(t) for t in texts]
    rng   = np.random.default_rng(seed)
    openings = [" ".join(TOKZ(t)[:12]) for t in texts]
    vc    = pd.Series(openings).value_counts()
    idx   = rng.integers(0, len(texts), size=(n_pairs, 2))
    pairs = [(int(a), int(b)) for a, b in idx if a != b]
    sb    = np.mean([sentence_bleu([TOKZ(texts[a])], TOKZ(texts[b]),
                                   weights=(0.25,) * 4, smoothing_function=_SM)
                     for a, b in pairs])
    return dict(**{"unique outputs": len(set(texts)) / len(texts),
                   "modal opening share": vc.iloc[0] / len(texts),
                   "self-BLEU-4": float(sb),
                   "mean sections": float(np.mean([len(SECS(t)) for t in texts]))}), vc.index[0]

div_rows, modal = {}, {}
_ref = list(pd.read_csv(f"preds_{ARM_KEYS[0][1]}.csv").fillna("")["reference"])
div_rows["REFERENCE corpus"], modal["REFERENCE corpus"] = diversity(_ref)
for label, tag, key in ARM_KEYS:
    df = pd.read_csv(f"preds_{tag}.csv").fillna("")
    div_rows[label], modal[label] = diversity(list(df["prediction"]))

div = pd.DataFrame(div_rows).T
div.to_csv("results_diversity.csv")
print("\n\nL2. Output diversity. The reference row is the floor: these reports really are")
print("formulaic, so only the gap between a model row and the reference row is evidence.")
print(div.round(4).to_string())
print("\nmost common opening, first twelve tokens:")
for k, v in modal.items():
    print(f"  {k:28s} {v[:66]}")
print("\nA 'unique outputs' figure well below the reference row, or a 'self-BLEU-4' well")
print("above it, means the model is producing near identical reports for different scans.")

# Every arm regenerated at one shared decoding length.
if RERUN_MATCHED_DECODING:
    sweeps = {key: json.load(open(f"metrics_{tag}.json"))["dev_sweep"]
              for _, tag, key in ARM_KEYS}
    grid   = sorted({int(k) for s in sweeps.values() for k in s})
    pooled = {g: float(np.mean([sweeps[k][str(g)] for k in sweeps])) for g in grid}
    COMMON_MIN_LEN = max(pooled, key=pooled.get)
    print(f"\n\nL3. Pooled dev BLEU-4 by min_len: "
          f"{ {k: round(v, 4) for k, v in pooled.items()} }")
    print(f"shared decoding length = {COMMON_MIN_LEN}, the value maximising mean dev BLEU-4")
    print("across all arms. Chosen on dev, applied to test once, identical for every arm.\n")

    def regen_at(key, min_len):
        ckpt = os.path.join(SCRATCH, f"ckpt_arm_{key}.pt")
        torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
        model = RegionReportModel(img_dim=IMG_DIM).to(DEVICE)
        load_trainables(model, ckpt)
        p, r, s = generate_on(model, FEAT[key], test_frame, min_len)
        del model; gc.collect(); torch.cuda.empty_cache()
        return p, r, s

    matched = {}
    for label, tag, key in ARM_KEYS:
        print(f"  regenerating {label} at min_len {COMMON_MIN_LEN}", flush=True)
        p, r, s = regen_at(key, COMMON_MIN_LEN)
        pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(
            f"preds_{tag}_ml{COMMON_MIN_LEN}.csv", index=False)
        m = score_all(p, r); m["min_len"] = COMMON_MIN_LEN
        json.dump(m, open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json", "w"), indent=2)
        matched[label] = m

    COLS2 = ["BLEU-1", "BLEU-4", "ROUGE-1", "ROUGE-L", "METEOR",
             "BERTScore-F1", "SectionF1", "SectionOrder", "pred len", "n_test"]
    mt = pd.DataFrame(matched).T[COLS2]
    mt.to_csv("results_matched_decoding.csv")
    print("\nTable: encoder comparison at a single shared decoding length")
    print(mt.round(4).to_string())
    print("\nThis is the table to put in the dissertation. Every arm now differs only in")
    print("the frozen encoder. Report the per-arm-tuned table alongside it if you like, but")
    print("this is the one that supports a claim about encoders.")
else:
    print("\n\nL3 skipped. RERUN_MATCHED_DECODING is False, so the encoder comparison is")
    print("still confounded by decoding length. Do not claim one encoder beats another.")

### 13.4 Controls at a shared decoding length

The controls in section 13.2 were decoded at lengths selected independently per arm, which is the confound isolated in section 13.3. The comparison is repeated here at the shared length of 120 tokens, using the existing checkpoints and generation only.

In [ ]:
# Controls regenerated at the shared decoding length. Generation only, no retraining.
COMMON_MIN_LEN = 120          # from L3: pooled dev BLEU-4 {20: 0.0098, 70: 0.1008, 120: 0.1095}

# The control feature directories are re-registered in case of a kernel restart. The
# directories themselves persist on scratch.
for base in ["resnet34"]:
    FEAT[f"{base}_shuffled"] = FEAT[base] + "_shuffled"
    FEAT[f"{base}_zeros"]    = FEAT[base] + "_zeros"
for k in ["resnet34_shuffled", "resnet34_zeros"]:
    assert os.path.isdir(FEAT[k]), f"{FEAT[k]} missing, rerun build_control_dirs in cell K"

def regen_at(key, min_len):
    """Load a trained arm and decode the test set at a chosen length. No training."""
    ckpt = os.path.join(SCRATCH, f"ckpt_arm_{key}.pt")
    torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model = RegionReportModel(img_dim=IMG_DIM).to(DEVICE)
    load_trainables(model, ckpt)
    p, r, s = generate_on(model, FEAT[key], test_frame, min_len)
    del model; gc.collect(); torch.cuda.empty_cache()
    return p, r, s

for key in ["resnet34_shuffled", "resnet34_zeros"]:
    tag = f"arm_{key}"
    out = f"preds_{tag}_ml{COMMON_MIN_LEN}.csv"
    if os.path.exists(out):
        print(f"  {key} already regenerated, skipping")
        continue
    print(f"  regenerating {key} at min_len {COMMON_MIN_LEN}", flush=True)
    p, r, s = regen_at(key, COMMON_MIN_LEN)
    pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(out, index=False)
    m = score_all(p, r); m["min_len"] = COMMON_MIN_LEN
    json.dump(m, open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json", "w"), indent=2)

# Corrected control table.
KEEP = ["BLEU-4", "ROUGE-L", "METEOR", "BERTScore-F1", "SectionF1", "pred len"]
rows = {}
for cond, tag in [("real",     "arm_resnet34"),
                  ("shuffled", "arm_resnet34_shuffled"),
                  ("zeros",    "arm_resnet34_zeros")]:
    m = json.load(open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json"))
    rows[cond] = {k: v for k, v in m.items() if k in KEEP}

ctrl = pd.DataFrame(rows).T[KEEP]
ctrl.to_csv("results_control_matched_decoding.csv")
print("\n" + "=" * 78)
print(f"CONTROL RESULTS AT A SHARED DECODING LENGTH (min_len {COMMON_MIN_LEN})")
print("=" * 78)
print(ctrl.round(4).to_string())

real, shuf, zero = (float(ctrl.loc[c, "BLEU-4"]) for c in ["real", "shuffled", "zeros"])
print(f"\n  real {real:.4f} | shuffled {shuf:.4f} ({100 * shuf / real:.1f}% retained) "
      f"| zeros {zero:.4f}")
print(f"  margin of a real scan over a blank input: {real - zero:+.4f} BLEU-4")
print("\n  Read this against cell N before writing a word of it up. A margin smaller")
print("  than the bootstrap interval is not a margin, it is noise, and saying so is")
print("  the finding.")

### 13.5 Uncertainty over the test set

Every comparison in this study separates two BLEU-4 values in the second or third decimal place on a single test set. The bootstrap resamples the 409 test studies with replacement and recomputes corpus BLEU-4. Comparisons are paired, in that both arms are scored on the identical resample, so variation shared by the two arms cancels and what remains is attributable to the arm.

This quantifies sampling variation over patients. It does not quantify variation over training runs, which is addressed in section 13.7.

In [ ]:
# Paired bootstrap over test studies.
N_BOOT = 500

def load_preds(tag, ml=COMMON_MIN_LEN):
    df = pd.read_csv(f"preds_{tag}_ml{ml}.csv").fillna("")
    return list(df["prediction"]), list(df["reference"])

BOOT_ARMS = {
    "ResNet-34 (2D)":     "arm_resnet34",
    "VGG-16 (2D)":        "arm_vgg16",
    "Med3D (3D)":         "arm_med3d",
    "control: shuffled":  "arm_resnet34_shuffled",
    "control: zeros":     "arm_resnet34_zeros",
}
BOOT_ARMS = {k: v for k, v in BOOT_ARMS.items()
             if os.path.exists(f"preds_{v}_ml{COMMON_MIN_LEN}.csv")}

series = {k: load_preds(v) for k, v in BOOT_ARMS.items()}
n_test = len(next(iter(series.values()))[0])
rng    = np.random.default_rng(0)
idx    = rng.integers(0, n_test, size=(N_BOOT, n_test))   # shared across arms = paired

print(f"bootstrapping {len(series)} arms, {N_BOOT} resamples of {n_test} studies", flush=True)
boot = {}
for name, (P, R) in series.items():
    boot[name] = np.array([bleu4([P[i] for i in row], [R[i] for i in row]) for row in idx])
    print(f"  {name:22s} done", flush=True)

ci = pd.DataFrame({
    name: {"BLEU-4": bleu4(P, R),
           "CI low":  float(np.percentile(boot[name], 2.5)),
           "CI high": float(np.percentile(boot[name], 97.5))}
    for name, (P, R) in series.items()}).T
ci["+/- half width"] = (ci["CI high"] - ci["CI low"]) / 2
ci.to_csv("results_bootstrap_ci.csv")
print("\nBLEU-4 with a 95% bootstrap interval over test studies")
print(ci.round(4).to_string())

# The paired comparisons on which the reported claims depend.
PAIRS = [("VGG-16 (2D)",    "ResNet-34 (2D)"),    # is the headline encoder ranking real
         ("ResNet-34 (2D)", "Med3D (3D)"),        # did fix 1 lose, and by a real margin
         ("ResNet-34 (2D)", "control: zeros"),    # does a real scan beat a blank array
         ("ResNet-34 (2D)", "control: shuffled")] # does a matched scan beat a mismatched one

print("\nPaired differences. 'p(A>B)' is the share of resamples where A wins.")
print("An interval that straddles zero means the difference is not established.\n")
out = []
for a, b in PAIRS:
    if a not in boot or b not in boot:
        continue
    d  = boot[a] - boot[b]
    lo, hi = np.percentile(d, [2.5, 97.5])
    sig = "established" if (lo > 0 or hi < 0) else "NOT established"
    out.append(dict(comparison=f"{a} - {b}", delta=float(d.mean()),
                    lo=float(lo), hi=float(hi), p_a_gt_b=float((d > 0).mean()),
                    verdict=sig))
    print(f"  {a:22s} - {b:22s}  {d.mean():+.4f}  "
          f"[{lo:+.4f}, {hi:+.4f}]  p(A>B) {(d > 0).mean():.2f}   {sig}")
pd.DataFrame(out).to_csv("results_paired_bootstrap.csv", index=False)

print("\nWrite the results chapter from THIS table. Any sentence of the form 'X")
print("outperforms Y' needs a row here marked established. Everything else is")
print("reported as a difference that the evidence does not separate from zero.")

### 13.6 Blank input controls for every encoder

The conclusion drawn from the controls is a claim about the pipeline rather than about one encoder, so the zeros control is extended to the remaining two arms. The volumetric row carries the most weight, since a volumetric encoder whose blank-input twin matches its real-input score has contributed nothing measurable.

In [ ]:
# Zeros control on the remaining two encoders.
def build_zeros_dir(base):
    src = FEAT[base]
    dst = src + "_zeros"
    os.makedirs(dst, exist_ok=True)
    cohort = pd.concat([train_frame, dev_frame, test_frame])
    n = 0
    for s in cohort["stem"]:
        d = os.path.join(dst, f"{s}.npy")
        if not os.path.exists(d):
            np.save(d, np.zeros_like(np.load(os.path.join(src, f"{s}.npy"))))
            n += 1
    FEAT[f"{base}_zeros"] = dst
    print(f"{base}: {n} new zero arrays in {dst}")

for base in ["vgg16", "med3d"]:
    build_zeros_dir(base)
    tag = f"arm_{base}_zeros"
    if os.path.exists(f"metrics_{tag}.json"):
        print(f"  {tag} already trained, skipping")
    else:
        run_arm(f"{base}_zeros")               # trains, sweeps min_len on dev, scores
    # Regenerated at the shared decoding length for comparability.
    if not os.path.exists(f"preds_{tag}_ml{COMMON_MIN_LEN}.csv"):
        p, r, s = regen_at(f"{base}_zeros", COMMON_MIN_LEN)
        pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(
            f"preds_{tag}_ml{COMMON_MIN_LEN}.csv", index=False)
        m = score_all(p, r); m["min_len"] = COMMON_MIN_LEN
        json.dump(m, open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json", "w"), indent=2)

print("\n" + "=" * 78)
print("REAL INPUT VERSUS BLANK INPUT, ALL THREE ENCODERS, SHARED DECODING LENGTH")
print("=" * 78)
rows = {}
for label, base in [("ResNet-34 (2D)", "resnet34"),
                    ("VGG-16 (2D)",    "vgg16"),
                    ("Med3D (3D)",     "med3d")]:
    r_ = json.load(open(f"metrics_arm_{base}_ml{COMMON_MIN_LEN}.json"))
    z_ = json.load(open(f"metrics_arm_{base}_zeros_ml{COMMON_MIN_LEN}.json"))
    rows[label] = {"real BLEU-4": r_["BLEU-4"], "zeros BLEU-4": z_["BLEU-4"],
                   "margin": r_["BLEU-4"] - z_["BLEU-4"],
                   "real SectionF1": r_["SectionF1"], "zeros SectionF1": z_["SectionF1"]}
abl = pd.DataFrame(rows).T
abl.to_csv("results_real_vs_blank.csv")
print(abl.round(4).to_string())
print("\nThis is the single most important table in the dissertation. It is the one")
print("that tells an examiner you checked whether your own system works, rather than")
print("reporting a number and hoping.")

### 13.7 Seed variance

All results to this point derive from a single initialisation. With 789 training studies and 20.4 million trainable parameters, run-to-run variation is not negligible, and an ordering between encoders that does not survive a change of seed is not an ordering between encoders.

Two additional seeds are trained per arm. Decoding length is held at the shared value rather than re-selected per seed, since re-selecting would reintroduce the length confound the shared value was chosen to remove.

In [ ]:
# Seed variance. Two additional seeds per arm at the shared decoding length.
EXTRA_SEEDS = [7, 2024]
SEED_ARMS   = ["resnet34", "vgg16", "med3d"]

def run_seed(name, seed, min_len=COMMON_MIN_LEN):
    RUN  = f"seed_{name}_s{seed}"
    ckpt = os.path.join(SCRATCH, f"ckpt_{RUN}.pt")
    if os.path.exists(f"metrics_{RUN}.json"):
        print(f"  {RUN} already done, skipping")
        return json.load(open(f"metrics_{RUN}.json"))
    print(f"\n--- {name}, seed {seed} ---", flush=True)
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    model = RegionReportModel(img_dim=IMG_DIM).to(DEVICE)
    hist  = train_arm(model, FEAT[name], ckpt)
    load_trainables(model, ckpt)
    p, r, s = generate_on(model, FEAT[name], test_frame, min_len)
    pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(f"preds_{RUN}.csv", index=False)
    m = score_all(p, r)
    m.update(min_len=min_len, seed=seed, arm=name,
             best_dev_loss=min(h["dev"] for h in hist))
    json.dump(m, open(f"metrics_{RUN}.json", "w"), indent=2)
    del model; gc.collect(); torch.cuda.empty_cache()
    print(f"  seed {seed} BLEU-4 {m['BLEU-4']:.4f}", flush=True)
    return m

for name in SEED_ARMS:
    for sd in EXTRA_SEEDS:
        run_seed(name, sd)

# Mean and spread across seeds.
METRICS = ["BLEU-4", "ROUGE-L", "METEOR", "BERTScore-F1", "SectionF1"]
rows = []
for name in SEED_ARMS:
    v42 = json.load(open(f"metrics_arm_{name}_ml{COMMON_MIN_LEN}.json"))
    v42["seed"] = SEED
    vals = [v42]
    for sd in EXTRA_SEEDS:
        f = f"metrics_seed_{name}_s{sd}.json"
        if os.path.exists(f):
            vals.append(json.load(open(f)))
    for v in vals:
        rows.append(dict(arm=name, seed=v.get("seed"), **{k: v[k] for k in METRICS}))

seeds_df = pd.DataFrame(rows)
seeds_df.to_csv("results_seeds_raw.csv", index=False)
agg = seeds_df.groupby("arm")[METRICS].agg(["mean", "std", "count"])
agg.to_csv("results_seeds_summary.csv")
print("\n" + "=" * 78)
print("SEED VARIANCE, SHARED DECODING LENGTH")
print("=" * 78)
print(seeds_df.round(4).to_string(index=False))
print("\nmean and standard deviation across seeds")
print(agg.round(4).to_string())

print("\nreport encoders as mean plus or minus sd:")
for name in SEED_ARMS:
    g = seeds_df[seeds_df["arm"] == name]["BLEU-4"]
    print(f"  {name:10s} {g.mean():.4f} +/- {g.std():.4f}  (n={len(g)})")
print("\nIf the intervals of two encoders overlap, the results chapter reports them")
print("as indistinguishable. That sentence is worth more marks than a ranking you")
print("cannot defend when the external asks how many times you ran it.")

### 13.8 Claims audit

Sections 13.5 and 13.7 measure different quantities and can disagree. The bootstrap resamples patients from one trained model; the seed replication retrains on the same patients. Where training variation exceeds sampling variation, a comparison that is significant under the bootstrap alone may still be an artefact of initialisation.

A claim is therefore admitted only when the paired bootstrap interval excludes zero and the sign of the difference is identical in every seed. The audit below applies that rule and records, for each comparison, whether it can be asserted.

In [ ]:
# Claims audit: bootstrap interval and seed sign agreement combined.
seeds_df = pd.read_csv("results_seeds_raw.csv")
piv = seeds_df.pivot(index="seed", columns="arm", values="BLEU-4")
print("BLEU-4 by seed and arm")
print(piv.round(4).to_string())

print("\nnoise budget, BLEU-4")
try:
    ci = pd.read_csv("results_bootstrap_ci.csv", index_col=0)
    half = {"resnet34": "ResNet-34 (2D)", "vgg16": "VGG-16 (2D)", "med3d": "Med3D (3D)"}
    for arm, label in half.items():
        sd = seeds_df[seeds_df["arm"] == arm]["BLEU-4"].std()
        hw = float(ci.loc[label, "+/- half width"]) if label in ci.index else float("nan")
        print(f"  {arm:10s} seed sd {sd:.4f} | test bootstrap half width {hw:.4f} "
              f"| seed noise is {sd / hw:.1f}x the test noise")
except FileNotFoundError:
    print("  results_bootstrap_ci.csv not found, rerun cell N")

# Paired differences by seed.
CLAIMS = [("vgg16", "resnet34"), ("resnet34", "med3d"), ("vgg16", "med3d")]
try:
    pb = pd.read_csv("results_paired_bootstrap.csv")
    LAB = {"resnet34": "ResNet-34 (2D)", "vgg16": "VGG-16 (2D)", "med3d": "Med3D (3D)"}
    boot_lookup = {}
    for _, r in pb.iterrows():
        boot_lookup[r["comparison"]] = r
except FileNotFoundError:
    boot_lookup = {}

print("\nPer seed paired differences. A claim needs the same sign in every seed.\n")
rows = []
for a, b in CLAIMS:
    d = piv[a] - piv[b]
    key = f"{LAB[a]} - {LAB[b]}"
    rev = f"{LAB[b]} - {LAB[a]}"
    if key in boot_lookup:
        bstat, bverdict = boot_lookup[key]["delta"], boot_lookup[key]["verdict"]
    elif rev in boot_lookup:
        bstat, bverdict = -boot_lookup[rev]["delta"], boot_lookup[rev]["verdict"]
    else:
        bstat, bverdict = float("nan"), "not run"
    consistent = bool((d > 0).all() or (d < 0).all())
    survives   = consistent and bverdict == "established"
    rows.append(dict(claim=f"{a} vs {b}",
                     **{f"seed {s}": round(float(d[s]), 4) for s in piv.index},
                     mean=round(float(d.mean()), 4), sd=round(float(d.std()), 4),
                     bootstrap=bverdict, same_sign_every_seed=consistent,
                     VERDICT="claimable" if survives else "NOT claimable"))
claims = pd.DataFrame(rows)
claims.to_csv("results_claims_audit.csv", index=False)
print(claims.to_string(index=False))

print("\n" + "=" * 78)
print("WHAT THE DISSERTATION IS ALLOWED TO SAY")
print("=" * 78)
for _, r in claims.iterrows():
    print(f"  {r['claim']:22s} {r['VERDICT']}")
print("""
Sentences that survive:
  the two dimensional encoders outperform the three dimensional Med3D encoder,
  consistently across seeds and outside the test set interval.
Sentences that do not survive and must be deleted:
  any ranking between ResNet-34 and VGG-16. Report them as indistinguishable and
  say why: run to run variation exceeds the difference between them.
""")

### 13.9 Clinical content

Overlap metrics reward reproduction of the reporting style of the source institution, which is precisely what a language prior supplies. They cannot establish whether a generated report names the finding present in the scan.

Each report is therefore reduced to a set of finding categories and a set of sides, and predictions are scored against references on those sets. This is a simplified rule-based approximation of the clinical efficacy metrics used in the chest radiograph literature, where a trained labeller performs the extraction. No trained labeller exists for chest CT within the scope of this project, so the extractor is lexical and its limitations are stated rather than hidden.

Negation is handled at clause level. Radiology reports assert absence as frequently as presence, and a matcher that ignores negation records a finding wherever absence is documented. Clause level rather than a preceding token window is used because this corpus places the negation cue after the finding as often as before it.

In [ ]:
# Clinical content: finding categories and laterality, with clause-level negation handling.
FINDINGS = {
    "nodule":            r"nodul",
    "ground glass":      r"ground[- ]?glass",
    "consolidation":     r"consolidat",
    "atelectasis":       r"atelectas",
    "emphysema":         r"emphysem",
    "bronchiectasis":    r"bronchiectas",
    "fibrosis":          r"fibrot|fibros",
    "pleural effusion":  r"effusion",
    "calcification":     r"calcifi",
    "septal thickening": r"septal (line|thicken)|interlobular",
    "tree in bud":       r"tree in bud|budding tree|centriacinar",
    "mass":              r"\bmass\b|lesion",
    "pneumothorax":      r"pneumothorax",
    "sequelae":          r"sequel|\bscar",
}
SIDES = {"left": r"\bleft\b", "right": r"\bright\b", "bilateral": r"both lungs|bilateral"}

# Cues marking a clause as an assertion of absence. Normal and unremarkable are included
# because this corpus uses them to close a region containing no finding, so a finding named
# in such a clause is being excluded rather than reported.
NEG_CUES = re.compile(
    r"\b(no|not|non|without|absent|negative for|free of|unremarkable|normal)\b"
    r"|\bnot (observed|detected|seen|present|identified|noted)\b"
)
CLAUSE = re.compile(r"[.;|]")

def positive_clauses(text):
    """Clauses that assert presence. Section headings are stripped first so that a
    heading never carries a finding into the wrong clause."""
    t = re.sub(r"\[[A-Z ]+\]", " ", str(text).lower())
    return [c for c in CLAUSE.split(t) if c.strip() and not NEG_CUES.search(c)]

def label_set(text, patterns):
    found = set()
    for c in positive_clauses(text):
        for k, p in patterns.items():
            if re.search(p, c):
                found.add(k)
    return found

def prf(gold, pred):
    if not gold and not pred:
        return 1.0, 1.0, 1.0
    inter = len(gold & pred)
    p = inter / len(pred) if pred else 0.0
    r = inter / len(gold) if gold else 0.0
    f = 0.0 if p + r == 0 else 2 * p * r / (p + r)
    return p, r, f

def content_scores(preds, refs, name):
    fp = [prf(label_set(r, FINDINGS), label_set(p, FINDINGS)) for p, r in zip(preds, refs)]
    sp = [prf(label_set(r, SIDES),    label_set(p, SIDES))    for p, r in zip(preds, refs)]
    # A side asserted by the prediction and absent from the reference. Clinically this is the
    # most consequential failure, since it places a finding in the wrong lung.
    wrong_side = np.mean([len(label_set(p, SIDES) - label_set(r, SIDES)) > 0
                          for p, r in zip(preds, refs)])
    return dict(arm=name,
                finding_P=np.mean([x[0] for x in fp]),
                finding_R=np.mean([x[1] for x in fp]),
                finding_F1=np.mean([x[2] for x in fp]),
                side_F1=np.mean([x[2] for x in sp]),
                side_hallucination_rate=float(wrong_side))

# Diagnostics on the negation filter.
ref_df   = pd.read_csv(f"preds_arm_resnet34_ml{COMMON_MIN_LEN}.csv").fillna("")
refs_all = list(ref_df["reference"])
_all = _kept = 0
for t in refs_all:
    cl = [c for c in CLAUSE.split(re.sub(r"\[[A-Z ]+\]", " ", str(t).lower())) if c.strip()]
    _all += len(cl); _kept += len(positive_clauses(t))
print(f"negation filter: {_all - _kept} of {_all} reference clauses dropped as negated "
      f"({100 * (1 - _kept / max(_all, 1)):.1f}%)")
print("\nsample of dropped clauses, check these really are assertions of absence:")
_drop = [c for t in refs_all[:200]
         for c in CLAUSE.split(re.sub(r"\[[A-Z ]+\]", " ", str(t).lower()))
         if c.strip() and NEG_CUES.search(c)]
for c in _drop[:5]:
    print("   -", c.strip()[:100])
print("\nsample of kept clauses, check these really are assertions of presence:")
for c in [c for t in refs_all[:50] for c in positive_clauses(t)][:5]:
    print("   +", c.strip()[:100])

# Majority baseline, fitted on the training partition only.
train_texts = [REPORT[s] for s in train_frame["stem"]]
freq = pd.Series([f for t in train_texts for f in label_set(t, FINDINGS)]).value_counts()
MAJORITY = set(freq[freq > 0.5 * len(train_texts)].index)
print(f"\nmajority finding set fitted on {len(train_texts)} training reports: "
      f"{sorted(MAJORITY) if MAJORITY else 'empty'}")
print("finding prevalence in train:")
print((freq / len(train_texts)).round(3).to_string())

maj_text = ". ".join(MAJORITY) if MAJORITY else ""
rows = [content_scores([maj_text] * len(refs_all), refs_all, "MAJORITY baseline")]

for base, label in [("resnet34", "ResNet-34 real"), ("vgg16", "VGG-16 real"),
                    ("med3d", "Med3D real"),
                    ("resnet34_zeros", "ResNet-34 zeros"), ("vgg16_zeros", "VGG-16 zeros"),
                    ("med3d_zeros", "Med3D zeros"),
                    ("resnet34_topk", "ResNet-34 top-k real"),
                    ("resnet34_meanmax", "ResNet-34 mean+max real")]:
    f = f"preds_arm_{base}_ml{COMMON_MIN_LEN}.csv"
    if not os.path.exists(f):
        continue
    df = pd.read_csv(f).fillna("")
    rows.append(content_scores(list(df["prediction"]), list(df["reference"]), label))

content = pd.DataFrame(rows).set_index("arm")
content.to_csv("results_clinical_content.csv")
print("\n" + "=" * 78)
print("CLINICAL CONTENT, NOT WORDING")
print("=" * 78)
print(content.round(4).to_string())

# Per finding recall.
print("\nper finding recall, real arms only. a category the model never produces is")
print("a category it cannot report, no matter what the BLEU score says.\n")
rec = {}
for base, label in [("resnet34", "ResNet-34"), ("vgg16", "VGG-16"), ("med3d", "Med3D")]:
    f = f"preds_arm_{base}_ml{COMMON_MIN_LEN}.csv"
    if not os.path.exists(f):
        continue
    df = pd.read_csv(f).fillna("")
    g = [label_set(r, FINDINGS) for r in df["reference"]]
    p = [label_set(x, FINDINGS) for x in df["prediction"]]
    rec[label] = {k: (sum(1 for gi, pi in zip(g, p) if k in gi and k in pi) /
                      max(sum(1 for gi in g if k in gi), 1)) for k in FINDINGS}
recall = pd.DataFrame(rec)
gold_all = [label_set(r, FINDINGS) for r in refs_all]
recall["prevalence in test"] = [sum(1 for g in gold_all if k in g) / len(gold_all)
                                for k in FINDINGS]
recall.to_csv("results_finding_recall.csv")
print(recall.round(3).to_string())

print("""
How to read this. If 'real' and 'zeros' land on the same finding F1, the visual
pathway contributes nothing a clinician would notice, and the honest conclusion
is that the pipeline reproduces the reporting style of the corpus rather than the
contents of the scan. If both sit near the MAJORITY baseline, the same holds with
no model at all as the comparator. Say which of these the numbers show, in the
first paragraph of the Discussion, before any BLEU number is quoted.

Limitation to state in the writeup: the extractor is lexical and the negation
filter discards a whole clause rather than resolving scope, so a clause that both
asserts one finding and rules out another is dropped entirely. The printed
diagnostic above gives the drop rate, and a hand check of a sample is the only
validation available without a trained CT labeller.
""")

#### Note on the cohort frame in the line printed above

The prevalence line above reads `fitted on 1341 training reports`. That count is the whole pinned
cohort of 1,341 studies rather than the 789 studies in the training fold, because this cell drew
prevalence from the full frame before the fold restriction was applied. The consequence is confined to
the majority baseline, which is degenerate in any case: no finding category reaches 50 per cent
prevalence, so the majority set is empty and its score of 0.1002 is identical to predicting nothing.
Section 13.14 explains why that baseline establishes nothing and replaces it.

Every baseline quoted in the report is refitted in section 13.14 on the 789 training studies alone and
applied once to the 409 test studies: 0.3640 for the best fixed finding set, 0.2807 for always
predicting the single most common finding, and 0.1002 for predicting nothing.

### 13.10 Cohort integrity

The background caching job continued to write after the reported results were produced, enlarging the preprocessed archive from 1341 to 1893 studies. Any cell that recomputes the cohort as an intersection over feature directories therefore widens it silently, which would make subsequent arms incomparable with those already reported.

The cohort is pinned rather than recomputed. The zeros feature directory was written once, from the original cohort, and has not been modified since, so its file list defines the study set for the remainder of the notebook. Derived directories are filled to that set, and the pin is asserted against the recorded test set size.

In [ ]:
# Cohort repair. Pins the study set and fills derived directories.
FROZEN_N = 1341        # cohort size behind every result already in this notebook

base = ["resnet34", "vgg16", "med3d"]
have = {k: {f[:-4] for f in os.listdir(FEAT[k]) if f.endswith(".npy")} for k in FEAT}
print("features per directory")
for k, v in sorted(have.items()):
    print(f"  {k:22s} {len(v):5d}")

common_now = set.intersection(*(have[k] for k in base))
print(f"\nintersection over the three encoders now: {len(common_now)}")

anchor = have.get("resnet34_zeros", set())
if len(anchor) == FROZEN_N:
    COHORT = anchor & common_now
    print(f"pinning to the ResNet zeros directory: {len(COHORT)} studies")
else:
    COHORT = common_now
    print(f"WARNING: resnet34_zeros holds {len(anchor)}, not {FROZEN_N}. "
          f"falling back to the current intersection, {len(COHORT)} studies.")

if len(COHORT) != FROZEN_N:
    print(f"\nWARNING: cohort is {len(COHORT)}, expected {FROZEN_N}. Do not mix these")
    print("runs with the frozen tables until you have worked out why.")
else:
    print("cohort matches the frozen results exactly, safe to continue")

drift = common_now - COHORT
if drift:
    print(f"\n{len(drift)} studies were encoded after the frozen runs and are being "
          f"excluded to keep the comparison honest:")
    for s in sorted(drift)[:10]:
        print("   ", s)

# The frames are rebuilt against the pinned cohort.
splits["ready"] = splits["stem"].isin(COHORT)
train_frame = splits[splits["ready"] & splits["fold"].isin(["train", "pool"])].reset_index(drop=True)
dev_frame   = splits[splits["ready"] & (splits["fold"] == "dev")].reset_index(drop=True)
test_frame  = splits[splits["ready"] & (splits["fold"] == "test")].reset_index(drop=True)
print(f"\npinned cohort: train {len(train_frame)} | dev {len(dev_frame)} | test {len(test_frame)}")
assert len(test_frame) == 409, f"test set is {len(test_frame)}, the frozen results used 409"

# Derived directories are filled.
# Zeros directories are deterministic functions of their source, so a missing entry is
# regenerated rather than treated as grounds to reduce the cohort.
def repair_zeros(key):
    src_key = key[:-len("_zeros")]
    if src_key not in FEAT:
        print(f"  {key}: source {src_key} not registered, skipping"); return
    missing = COHORT - have.get(key, set())
    if not missing:
        print(f"  {key}: complete"); return
    n = 0
    for s in missing:
        sp = os.path.join(FEAT[src_key], f"{s}.npy")
        if not os.path.exists(sp):
            continue
        np.save(os.path.join(FEAT[key], f"{s}.npy"),
                np.zeros_like(np.load(sp)))
        n += 1
    print(f"  {key}: wrote {n} missing arrays of {len(missing)}")

print("\nrepairing derived directories")
for key in [k for k in list(FEAT) if k.endswith("_zeros")]:
    repair_zeros(key)

# The shuffled directory cannot be regenerated without repeating the derangement, which
# would alter which study reads which features, so an incomplete shuffled directory is
# reported rather than patched.
for key in [k for k in FEAT if k.endswith("_shuffled")]:
    miss = COHORT - have.get(key, set())
    if miss:
        print(f"  {key}: SHORT by {len(miss)}. rerun build_control_dirs in cell K "
              f"if you need this arm again. it is not needed for S, T or U.")

# Final check on every directory the remaining sections read.
need = ["resnet34", "vgg16", "med3d", "resnet34_zeros", "vgg16_zeros", "med3d_zeros"]
print("\nreadiness for S")
ok = True
for k in need:
    if k not in FEAT:
        print(f"  {k:22s} NOT REGISTERED, rerun cells M and O"); ok = False; continue
    got = {f[:-4] for f in os.listdir(FEAT[k]) if f.endswith(".npy")}
    miss = COHORT - got
    print(f"  {k:22s} {'complete' if not miss else f'MISSING {len(miss)}'}")
    ok &= not miss
print("\nready" if ok else "\nNOT ready, fix the directories above before rerunning S")

### 13.11 Seed variance on the control arms

The margins between real and blank input in section 13.6 each derive from a single run on both sides, and section 13.7 establishes that a single run on this setup moves by approximately 0.02 BLEU-4. Two additional seeds are trained for each blank-input arm so that the margin can be reported with a spread on both sides.

In [ ]:
# Seed variance on the blank-input arms.
CONTROL_SEED_ARMS = ["resnet34_zeros", "vgg16_zeros"]     # add "med3d_zeros" if time allows

for name in CONTROL_SEED_ARMS:
    assert name in FEAT, f"{name} not registered in FEAT, rerun cells M and O"
    for sd in EXTRA_SEEDS:
        run_seed(name, sd)

# Real against blank input, with a spread on both sides.
def collect(name):
    vals = []
    base_ml = f"metrics_arm_{name}_ml{COMMON_MIN_LEN}.json"
    if os.path.exists(base_ml):
        v = json.load(open(base_ml)); v["seed"] = SEED; vals.append(v)
    for sd in EXTRA_SEEDS:
        f = f"metrics_seed_{name}_s{sd}.json"
        if os.path.exists(f):
            vals.append(json.load(open(f)))
    return [v["BLEU-4"] for v in vals]

print("\n" + "=" * 78)
print("REAL VERSUS BLANK INPUT, WITH SEED SPREAD ON BOTH ARMS")
print("=" * 78)
rows = []
for base in ["resnet34", "vgg16", "med3d"]:
    r_, z_ = collect(base), collect(f"{base}_zeros")
    if not z_:
        continue
    rows.append(dict(encoder=base,
                     real_mean=np.mean(r_), real_sd=np.std(r_, ddof=1) if len(r_) > 1 else np.nan,
                     real_n=len(r_),
                     zeros_mean=np.mean(z_), zeros_sd=np.std(z_, ddof=1) if len(z_) > 1 else np.nan,
                     zeros_n=len(z_),
                     margin=np.mean(r_) - np.mean(z_)))
rv = pd.DataFrame(rows).set_index("encoder")
rv["margin in sd units"] = rv["margin"] / rv[["real_sd", "zeros_sd"]].max(axis=1)
rv.to_csv("results_real_vs_blank_seeded.csv")
print(rv.round(4).to_string())
print("""
A margin under one standard deviation is not evidence that the image is being
read. Report it as such. A margin over two is evidence, and then the claim is
that the encoder contributes measurably but that the language prior still
supplies the majority of the score. Both readings are defensible findings. What
is not defensible is quoting the margin without the spread beside it.
""")

### 13.12 Pooling operator

The controls establish that a real scan barely separates from a blank array. One explanation is that the encoder carries no usable signal. A second is testable and is examined here.

Slice features are pooled by taking the masked mean of the layer4 map. At a 32-fold downsample and 1.5 mm in-plane resolution, a single feature position covers approximately 48 mm of anatomy and a lung slice contributes on the order of one hundred valid positions, so a 4 mm nodule occupies one position in a hundred and is divided by that hundred before the bridge receives it. Diffuse findings survive an average; focal findings, which constitute most of what these reports describe, do not.

Two replacement operators are computed from the same forward pass over the same volumes with the same frozen encoder. The first is the mean of the K highest activations per channel within the lung mask, which remains 512-dimensional and is therefore matched to the existing arm on everything except the operator. The second concatenates masked mean with masked maximum, which carries strictly more information but doubles the input width and is consequently the weaker experiment. This treatment of per-slice pooling as an instance selection problem follows the multiple instance learning formulation used for focal lesions in large fields.

In [ ]:
# Pooling operators: top-k mean and mean concatenated with maximum, from one forward pass.
TOPK = 16          # of the ~100 to 250 lung positions per slice at feature resolution

FEAT["resnet34_topk"]    = os.path.join(SCRATCH, "feat_cache_2d_topk")
FEAT["resnet34_meanmax"] = os.path.join(SCRATCH, "feat_cache_2d_meanmax")
for k in ["resnet34_topk", "resnet34_meanmax"]:
    os.makedirs(FEAT[k], exist_ok=True)

@torch.no_grad()
def encode_2d_pooled(net, npz_path, slice_batch=16):
    """Identical to encode_2d up to the pooling step, then returns both variants.
    Keeping one loading and one forward pass guarantees the two operators see byte
    identical feature maps, so any difference downstream is the operator."""
    img  = _load_volume(npz_path)
    lung = (img > 0).float()
    idx  = torch.where(lung.sum(dim=(0, 1)) > 0)[0]
    if len(idx) == 0:
        return None, None
    img, lung = img[..., idx], lung[..., idx]
    sl = img.permute(2, 0, 1).unsqueeze(1).repeat(1, 3, 1, 1)
    sl = (sl - MEAN) / STD
    lm = lung.permute(2, 0, 1).unsqueeze(1)

    tk_out, mm_out = [], []
    for s in range(0, sl.shape[0], slice_batch):
        fb = net(sl[s:s + slice_batch].to(DEVICE))                       # (b, 512, h, w)
        mb = F.interpolate(lm[s:s + slice_batch].to(DEVICE),
                           size=fb.shape[-2:], mode="nearest")           # (b, 1, h, w)

        xf = fb.flatten(2)                                               # (b, 512, P)
        mf = mb.flatten(2)                                               # (b, 1, P)

        # Masked mean, identical to the operator used in the extraction above.
        den  = mf.sum(dim=2).clamp(min=1.0)                              # (b, 1)
        mean = (xf * mf).sum(dim=2) / den                                # (b, 512)

        # Positions outside the lung are removed rather than set to zero. layer4 is post-activation,
        # so a zero fill would compete with genuine low activations under a maximum and prevail
        # wherever the true minimum is zero.
        xm = xf.masked_fill(mf == 0, float("-inf"))

        mx = xm.amax(dim=2)                                              # (b, 512)
        mx = torch.where(torch.isfinite(mx), mx, torch.zeros_like(mx))

        k = min(TOPK, xf.shape[2])
        vals, _ = xm.topk(k=k, dim=2)                                    # (b, 512, k)
        fin  = torch.isfinite(vals)
        topk = vals.masked_fill(~fin, 0).sum(2) / fin.sum(2).clamp(min=1)

        tk_out.append(topk.float().cpu())
        mm_out.append(torch.cat([mean, mx], dim=1).float().cpu())        # (b, 1024)
    return torch.cat(tk_out, 0), torch.cat(mm_out, 0)

def extract_pooled(limit=None):
    # COHORT is the pinned study set on which every reported result was computed. The archive
    # is now larger, because the background caching job continued to write, so encoding every
    # volume on disk would widen the cohort. Extraction is restricted to the pin.
    assert "COHORT" in globals(), "run cell S0 first, it defines the pinned cohort"
    stems = sorted(s for s in COHORT
                   if os.path.exists(os.path.join(CACHE_DIR, f"{s}.npz")))
    print(f"  npz files on disk: "
          f"{len([f for f in os.listdir(CACHE_DIR) if f.endswith('.npz')])}, "
          f"pinned cohort: {len(COHORT)}, encoding: {len(stems)}")
    todo  = [s for s in stems
             if not (os.path.exists(os.path.join(FEAT["resnet34_topk"], f"{s}.npy")) and
                     os.path.exists(os.path.join(FEAT["resnet34_meanmax"], f"{s}.npy")))]
    if limit:
        todo = todo[:limit]
    print(f"pooling variants: {len(stems)} volumes preprocessed, {len(todo)} to encode",
          flush=True)
    if not todo:
        print("  nothing to do")
        return
    net = _build_2d("resnet34")
    t0 = time.time()
    for n, s in enumerate(todo):
        try:
            tk, mm = encode_2d_pooled(net, os.path.join(CACHE_DIR, f"{s}.npz"))
        except RuntimeError as e:
            print("  failed:", s, str(e)[:120], flush=True)
            torch.cuda.empty_cache(); continue
        if tk is None:
            print("  skipped, no lung voxels:", s, flush=True); continue
        np.save(os.path.join(FEAT["resnet34_topk"],    f"{s}.npy"), tk.numpy().astype(np.float16))
        np.save(os.path.join(FEAT["resnet34_meanmax"], f"{s}.npy"), mm.numpy().astype(np.float16))
        if n % 100 == 0:
            print(f"  {n}/{len(todo)}  ({time.time() - t0:.0f}s)", flush=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    for k in ["resnet34_topk", "resnet34_meanmax"]:
        print(f"  {k}: {len([f for f in os.listdir(FEAT[k]) if f.endswith('.npy')])} files")

extract_pooled()

# The cohort must not move. If the new directories hold a different study set from the
# mean-pooled arm, the pooling operator is no longer the only quantity that varies.
for k in ["resnet34_topk", "resnet34_meanmax"]:
    got  = {f[:-4] for f in os.listdir(FEAT[k]) if f.endswith(".npy")}
    miss = COHORT - got
    print(f"{k}: {len(got)} files on disk, {len(miss)} of the pinned cohort missing")
    assert not miss, f"{k} is missing {len(miss)} cohort studies, rerun extract_pooled"

_first  = sorted(COHORT)[0]
_probe  = np.load(os.path.join(FEAT["resnet34_topk"],    f"{_first}.npy"))
_probe2 = np.load(os.path.join(FEAT["resnet34_meanmax"], f"{_first}.npy"))
print(f"\ntopk feature shape {_probe.shape} | meanmax feature shape {_probe2.shape}")
assert _probe.shape[1] == 512 and _probe2.shape[1] == 1024
print("cohort unchanged, widths as expected, ready to train")

### 13.13 Pooling ablation

The success criterion is fixed before the arms are trained, so that the reading of the outcome cannot be selected to suit the number obtained.

The primary outcome is the margin of real input over blank input, not the absolute score. An operator that raises every arm including its blank-input twin has improved the fluency of the decoder and has established nothing about whether the scan is read. The threshold is the seed standard deviation measured in section 13.7, which is 0.0210 BLEU-4 for the ResNet family. The mean-pooled arm returns a margin of 0.0051, approximately one quarter of that value.

Both outcomes are reportable. A margin above threshold locates the bottleneck in the pooling operator. A margin below threshold establishes that the language prior account survives a targeted and mechanistically motivated attempt to overturn it, which is a stronger position than the controls alone support.

In [ ]:
# Pooling arms, evaluated against a criterion fixed before training.
SD_SEED   = 0.0210        # from results_seeds_raw.csv, resnet34 across seeds 42, 7, 2024
BASE_MARGIN = 0.0051      # mean pooled ResNet-34, real minus zeros, cell O
POOL_ARMS = [("resnet34_topk", 512), ("resnet34_meanmax", 1024)]

def run_arm_width(name, width):
    """run_arm reads IMG_DIM from the global scope, so the width is swapped around
    the call and restored afterwards. The meanmax arm is 1024 wide and every other
    arm in the notebook is 512."""
    global IMG_DIM
    keep = IMG_DIM
    try:
        IMG_DIM = width
        return run_arm(name)
    finally:
        IMG_DIM = keep

for name, width in POOL_ARMS:
    if os.path.exists(f"metrics_arm_{name}.json"):
        print(f"  arm_{name} already trained, skipping")
    else:
        run_arm_width(name, width)
    # Regenerated at the shared decoding length for comparability.
    tag = f"arm_{name}"
    if not os.path.exists(f"preds_{tag}_ml{COMMON_MIN_LEN}.csv"):
        keep, IMG_DIM = IMG_DIM, width
        p, r, s = regen_at(name, COMMON_MIN_LEN)
        IMG_DIM = keep
        pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(
            f"preds_{tag}_ml{COMMON_MIN_LEN}.csv", index=False)
        m = score_all(p, r); m["min_len"] = COMMON_MIN_LEN
        json.dump(m, open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json", "w"), indent=2)

# Blank input control for each pooling operator. The primary outcome is a margin, so every
# pooling arm requires its own blank-input counterpart.
for name, width in POOL_ARMS:
    build_zeros_dir(name)                      # defined in cell O
    zname, ztag = f"{name}_zeros", f"arm_{name}_zeros"
    if os.path.exists(f"metrics_{ztag}.json"):
        print(f"  {ztag} already trained, skipping")
    else:
        run_arm_width(zname, width)
    if not os.path.exists(f"preds_{ztag}_ml{COMMON_MIN_LEN}.csv"):
        keep, IMG_DIM = IMG_DIM, width
        p, r, s = regen_at(zname, COMMON_MIN_LEN)
        IMG_DIM = keep
        pd.DataFrame(dict(stem=s, reference=r, prediction=p)).to_csv(
            f"preds_{ztag}_ml{COMMON_MIN_LEN}.csv", index=False)
        m = score_all(p, r); m["min_len"] = COMMON_MIN_LEN
        json.dump(m, open(f"metrics_{ztag}_ml{COMMON_MIN_LEN}.json", "w"), indent=2)

# Pooling table.
def ml(tag):
    return json.load(open(f"metrics_{tag}_ml{COMMON_MIN_LEN}.json"))

rows = []
for label, name in [("mean (as built)", "resnet34"),
                    ("top-k mean",      "resnet34_topk"),
                    ("mean + max",      "resnet34_meanmax")]:
    try:
        r_, z_ = ml(f"arm_{name}"), ml(f"arm_{name}_zeros")
    except FileNotFoundError:
        print(f"  {name} incomplete, skipping"); continue
    rows.append(dict(pooling=label, width=512 if name != "resnet34_meanmax" else 1024,
                     real_BLEU4=r_["BLEU-4"], zeros_BLEU4=z_["BLEU-4"],
                     margin=r_["BLEU-4"] - z_["BLEU-4"],
                     margin_in_sd=(r_["BLEU-4"] - z_["BLEU-4"]) / SD_SEED,
                     real_SectionF1=r_["SectionF1"], real_METEOR=r_["METEOR"]))
pool = pd.DataFrame(rows).set_index("pooling")
pool.to_csv("results_pooling_ablation.csv")

print("\n" + "=" * 78)
print("POOLING ABLATION. SAME ENCODER, SAME COHORT, SAME BRIDGE, SAME DECODER.")
print("The primary outcome is the margin column, not the real BLEU-4 column.")
print("=" * 78)
print(pool.round(4).to_string())

best = pool["margin"].idxmax()
bm   = float(pool.loc[best, "margin"])
print(f"\nbest operator by margin: {best} at {bm:+.4f} BLEU-4, "
      f"{bm / SD_SEED:.2f} seed standard deviations")
print(f"mean pooling, for reference: {BASE_MARGIN:+.4f}, {BASE_MARGIN / SD_SEED:.2f} sd")

if bm > SD_SEED:
    print("""
VERDICT: the pooling operator was the bottleneck.
Rerun the clinical content scoring in section 13.9 with the winning arm added,
because a visual pathway that works on BLEU still has to be shown to name the
right findings. Then rerun the seed replication in section 13.11 with this arm,
since a single run margin is exactly the mistake that replication catches. Only after both does the Discussion lead with it.""")
else:
    print("""
VERDICT: the margin is still inside run to run noise.
The language prior conclusion holds, and it now holds against a targeted attempt
to overturn it. This is the stronger version of the finding, and this cell is the
evidence for it. Write it as: the failure is not an artefact of one pooling
choice, because replacing masked mean pooling with an operator designed to
preserve focal activations did not widen the gap between a real scan and a blank
array. Say what that implies, which is that the bottleneck sits upstream, in a
frozen ImageNet encoder that was never trained to represent lung pathology, or in
789 training studies, and not in the summarisation step.""")

### 13.14 Final corrections

Three corrections close the experimental work.

The comparison list used for the paired bootstrap omitted one pair, which caused the audit in section 13.8 to record that comparison as untested and to withhold a claim the seed evidence supports. All pairs are now enumerated rather than listed by hand.

The majority baseline in section 13.9 returned an empty set, because no finding category reaches 50 per cent prevalence in the training reports. Its score therefore measures a model that predicts nothing, and exceeding it establishes nothing. Two baselines replace it: the single most prevalent finding, and the best fixed finding set obtained by a prevalence threshold sweep fitted on the training partition and applied once to the test partition.

The volumetric blank-input arm is extended to three seeds, since it carries the largest single result in the study.

In [ ]:
# Final corrections: complete pairwise bootstrap, non-degenerate baselines, and volumetric control seeds.
from itertools import combinations

N_BOOT = 500
ML     = COMMON_MIN_LEN
ARMS_V = {"ResNet-34 (2D)": "arm_resnet34", "VGG-16 (2D)": "arm_vgg16",
          "Med3D (3D)": "arm_med3d",
          "control: shuffled": "arm_resnet34_shuffled",
          "control: zeros": "arm_resnet34_zeros",
          "top-k mean": "arm_resnet34_topk", "mean + max": "arm_resnet34_meanmax"}
ARMS_V = {k: v for k, v in ARMS_V.items() if os.path.exists(f"preds_{v}_ml{ML}.csv")}

series = {}
for k, v in ARMS_V.items():
    df = pd.read_csv(f"preds_{v}_ml{ML}.csv").fillna("")
    series[k] = (list(df["prediction"]), list(df["reference"]))

n_test = len(next(iter(series.values()))[0])
idx = np.random.default_rng(0).integers(0, n_test, size=(N_BOOT, n_test))
print(f"bootstrapping {len(series)} arms over {n_test} studies", flush=True)
boot = {k: np.array([bleu4([P[i] for i in row], [R[i] for i in row]) for row in idx])
        for k, (P, R) in series.items()}

rows = []
for a, b in combinations(series, 2):
    d = boot[a] - boot[b]
    lo, hi = np.percentile(d, [2.5, 97.5])
    rows.append(dict(A=a, B=b, delta=float(d.mean()), lo=float(lo), hi=float(hi),
                     p_a_gt_b=float((d > 0).mean()),
                     verdict="established" if (lo > 0 or hi < 0) else "NOT established"))
pb = pd.DataFrame(rows)
pb.to_csv("results_paired_bootstrap_all.csv", index=False)
print("\nAll pairwise bootstrap comparisons at the shared decoding length")
print(pb.round(4).to_string(index=False))

# Claims audit, rebuilt over the complete comparison set.
seeds_df = pd.read_csv("results_seeds_raw.csv")
piv  = seeds_df.pivot(index="seed", columns="arm", values="BLEU-4")
LAB  = {"resnet34": "ResNet-34 (2D)", "vgg16": "VGG-16 (2D)", "med3d": "Med3D (3D)"}
look = {(r["A"], r["B"]): r for _, r in pb.iterrows()}

out = []
for a, b in combinations([c for c in piv.columns if c in LAB], 2):
    d = piv[a] - piv[b]
    # Explicit None checks. The lookup holds pandas rows, and a disjunction over a Series
    # raises rather than falling through, since a Series has no defined truth value.
    r = look.get((LAB[a], LAB[b]))
    if r is None:
        r = look.get((LAB[b], LAB[a]))
    bv = r["verdict"] if r is not None else "not run"
    same = bool((d > 0).all() or (d < 0).all())
    out.append(dict(claim=f"{a} vs {b}",
                    **{f"seed {s}": round(float(d[s]), 4) for s in piv.index},
                    mean=round(float(d.mean()), 4), sd=round(float(d.std()), 4),
                    bootstrap=bv, same_sign_every_seed=same,
                    VERDICT="claimable" if (same and bv == "established") else "NOT claimable"))
claims = pd.DataFrame(out)
claims.to_csv("results_claims_audit.csv", index=False)
print("\nClaims audit, rebuilt")
print(claims.to_string(index=False))

# Non-degenerate baselines.
train_sets = [label_set(REPORT[s], FINDINGS) for s in train_frame["stem"]]
prev = pd.Series([f for st in train_sets for f in st]).value_counts() / len(train_sets)

def fixed_set_f1(pred_set, gold_sets):
    return float(np.mean([prf(g, pred_set)[2] for g in gold_sets]))

# The threshold is swept on the training partition and the selected set applied once to test.
grid = sorted(set(prev.round(3)))
best_t = max(grid, key=lambda t: fixed_set_f1(set(prev[prev >= t].index), train_sets))
BEST_SET = set(prev[prev >= best_t].index)
TOP1     = {prev.index[0]}
print(f"\nprevalence threshold chosen on train: {best_t:.3f} -> {sorted(BEST_SET)}")
print(f"single most prevalent finding: {sorted(TOP1)} at {prev.iloc[0]:.3f}")

ref_df   = pd.read_csv(f"preds_arm_resnet34_ml{ML}.csv").fillna("")
gold_test = [label_set(r, FINDINGS) for r in ref_df["reference"]]

base_rows = {
    "baseline: predict nothing":     fixed_set_f1(set(), gold_test),
    "baseline: always most common":  fixed_set_f1(TOP1, gold_test),
    "baseline: best fixed set":      fixed_set_f1(BEST_SET, gold_test),
}
for label, base in [("ResNet-34 real", "resnet34"), ("VGG-16 real", "vgg16"),
                    ("Med3D real", "med3d"), ("ResNet-34 zeros", "resnet34_zeros"),
                    ("VGG-16 zeros", "vgg16_zeros"), ("Med3D zeros", "med3d_zeros"),
                    ("top-k real", "resnet34_topk"), ("mean+max real", "resnet34_meanmax")]:
    f = f"preds_arm_{base}_ml{ML}.csv"
    if not os.path.exists(f):
        continue
    df = pd.read_csv(f).fillna("")
    base_rows[label] = float(np.mean([prf(label_set(r, FINDINGS), label_set(p, FINDINGS))[2]
                                      for p, r in zip(df["prediction"], df["reference"])]))
bl = pd.Series(base_rows).sort_values(ascending=False).to_frame("finding F1")
bl.to_csv("results_clinical_baselines.csv")
print("\n" + "=" * 78)
print("FINDING F1 AGAINST BASELINES THAT ARE NOT DEGENERATE")
print("=" * 78)
print(bl.round(4).to_string())
print("\nThe empty set baseline is the one cell R accidentally used. Quote the best")
print("fixed set row instead, because that is the hardest thing a model with no")
print("image has to beat, and it is fitted on train and applied to test once.")

# Additional seeds on the volumetric blank-input arm.
for sd in EXTRA_SEEDS:
    run_seed("med3d_zeros", sd)

def collect(name):
    vals = []
    f0 = f"metrics_arm_{name}_ml{ML}.json"
    if os.path.exists(f0):
        vals.append(json.load(open(f0))["BLEU-4"])
    for sd in EXTRA_SEEDS:
        f = f"metrics_seed_{name}_s{sd}.json"
        if os.path.exists(f):
            vals.append(json.load(open(f))["BLEU-4"])
    return vals

rows = []
for base in ["resnet34", "vgg16", "med3d"]:
    r_, z_ = collect(base), collect(f"{base}_zeros")
    if len(z_) < 2:
        continue
    sd_pool = max(np.std(r_, ddof=1), np.std(z_, ddof=1))
    rows.append(dict(encoder=base, real_mean=np.mean(r_), real_sd=np.std(r_, ddof=1),
                     zeros_mean=np.mean(z_), zeros_sd=np.std(z_, ddof=1),
                     margin=np.mean(r_) - np.mean(z_),
                     margin_in_sd=(np.mean(r_) - np.mean(z_)) / sd_pool,
                     n=f"{len(r_)} vs {len(z_)}"))
rv = pd.DataFrame(rows).set_index("encoder")
rv.to_csv("results_real_vs_blank_seeded.csv")
print("\n" + "=" * 78)
print("REAL VERSUS BLANK, THREE SEEDS EACH SIDE")
print("=" * 78)
print(rv.round(4).to_string())
print("""
Experimental work ends here. Everything from this point is writing.
""")

## 14. Report figures

Every figure the report embeds is regenerated here from the artifacts that earlier sections wrote to disk, so each figure is traceable to a named results file rather than to a transcription. Nothing in this section trains, generates or scores. The section is self contained: it runs after a kernel restart without executing anything above it, and it skips cleanly with a printed message when an artifact is absent. One cell requires the network and is marked as such; every other cell runs offline.

In [ ]:
# Figure setup. Paths, a shared style, and the shared decoding length recovered from the
# files on disk rather than hard coded, so that the section stays correct if the sweep in
# 13.3 is ever rerun with a different outcome.
import os, re, json, glob, math, shutil, tarfile, textwrap
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

SCRATCH   = "/hpc/scratch/964931"
CACHE_DIR = os.path.join(SCRATCH, "preproc_cache")       # preprocessed .npz volumes
FIG_DIR   = os.path.expanduser("~/figures_report")
os.makedirs(FIG_DIR, exist_ok=True)

_ml = sorted({int(m.group(1)) for f in os.listdir(".")
              for m in [re.match(r"preds_arm_resnet34_ml(\d+)\.csv$", f)] if m})
ML = _ml[-1] if _ml else 120
print("shared decoding length recovered from disk:", ML)

plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300,
    "font.size": 9, "axes.titlesize": 10, "axes.labelsize": 9,
    "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False,
})
COL = {"resnet34": "#1f77b4", "vgg16": "#ff7f0e", "med3d": "#2ca02c",
       "zeros": "#7f7f7f", "shuffled": "#9467bd"}

def save(fig, name):
    p = os.path.join(FIG_DIR, name)
    fig.savefig(p, bbox_inches="tight")
    print("saved", p)

def have(*files):
    missing = [f for f in files if not os.path.exists(f)]
    for f in missing:
        print(f"MISSING: {f}, the figure that needs it is skipped")
    return not missing

print("figures will be written to:", FIG_DIR)

### 14.1 Figures from result files

In [ ]:
# Figure: loss per epoch for the three encoder arms, with the blank input arm overlaid.
# Expected: all four development curves settle in a narrow band. That proximity is the
# visual form of the central finding, that the language prior supplies most of the
# achievable loss reduction and the image contributes at the margin.
HIST = [("ResNet-34",              "history_arm_resnet34.json",       COL["resnet34"], "-"),
        ("VGG-16",                 "history_arm_vgg16.json",          COL["vgg16"],    "-"),
        ("Med3D",                  "history_arm_med3d.json",          COL["med3d"],    "-"),
        ("ResNet-34, blank input", "history_arm_resnet34_zeros.json", COL["zeros"],    "--")]

fig, axes = plt.subplots(1, 2, figsize=(7.6, 3.0), sharey=True)
n_drawn = 0
for label, path, c, ls in HIST:
    if not os.path.exists(path):
        print(f"MISSING: {path}, curve skipped"); continue
    h  = json.load(open(path))
    ep = [d["epoch"] + 1 for d in h]
    axes[0].plot(ep, [d["train"] for d in h], ls, color=c, label=label)
    axes[1].plot(ep, [d["dev"]   for d in h], ls, color=c, label=label)
    n_drawn += 1
axes[0].set_title("training loss")
axes[1].set_title("development loss")
for ax in axes:
    ax.set_xlabel("epoch")
axes[0].set_ylabel("token level cross entropy")
axes[1].legend(fontsize=8)
fig.tight_layout()
if n_drawn:
    save(fig, "fig_loss_curves.png")
plt.show()

In [ ]:
# Figure: paired bootstrap differences in BLEU-4, drawn as a forest plot. Each interval is
# the central 95% of the paired difference over 500 shared resamples of the test studies,
# from 13.14 (complete pairwise set) or 13.5 as the fallback. An interval clear of zero is
# the criterion the text uses for the word "outperforms"; an interval straddling zero is
# reported as a difference the evidence does not separate from zero.
src = "results_paired_bootstrap_all.csv"
if not os.path.exists(src):
    src = "results_paired_bootstrap.csv"
if have(src):
    pb = pd.read_csv(src)
    if {"A", "B"}.issubset(pb.columns):
        pb["comparison"] = pb["A"] + "  vs  " + pb["B"]
    pb = pb.sort_values("delta").reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(7.2, 0.42 * len(pb) + 1.2))
    for i, r in pb.iterrows():
        c = "#1f77b4" if r["verdict"] == "established" else "#aaaaaa"
        ax.plot([r["lo"], r["hi"]], [i, i], color=c, lw=2)
        ax.plot(r["delta"], i, "o", color=c, ms=5)
    ax.axvline(0, color="#444444", lw=1)
    ax.set_yticks(range(len(pb)))
    ax.set_yticklabels(pb["comparison"], fontsize=8)
    ax.set_xlabel("paired BLEU-4 difference, A minus B, with a 95% bootstrap interval")
    ax.set_title("blue: established, interval clear of zero. grey: not established", fontsize=9)
    fig.tight_layout()
    save(fig, "fig_bootstrap_forest.png")
    plt.show()

In [ ]:
# Figure: real input against blank input per encoder. Bars are the mean test BLEU-4 across
# seeds, the error bar is one standard deviation, black points are the individual real seed
# runs. Expected: pairs of similar height with a small annotated margin. The reading rules
# sit in 13.6; this figure only puts the margin and the spread in the same frame, which is
# the condition under which the margin may be quoted at all.
if have("results_real_vs_blank_seeded.csv"):
    rv = pd.read_csv("results_real_vs_blank_seeded.csv").set_index("encoder")
    x  = np.arange(len(rv)); w = 0.36
    fig, ax = plt.subplots(figsize=(6.4, 3.2))
    ax.bar(x - w / 2, rv["real_mean"],  w, yerr=rv["real_sd"].fillna(0),  capsize=3,
           label="real scan", color="#1f77b4")
    ax.bar(x + w / 2, rv["zeros_mean"], w, yerr=rv["zeros_sd"].fillna(0), capsize=3,
           label="blank input", color="#9a9a9a")
    if os.path.exists("results_seeds_raw.csv"):
        sr = pd.read_csv("results_seeds_raw.csv")
        for i, enc in enumerate(rv.index):
            pts = sr[sr["arm"] == enc]["BLEU-4"]
            ax.plot(np.full(len(pts), i - w / 2), pts, "k.", ms=4, alpha=0.7)
    for i, enc in enumerate(rv.index):
        top = max(rv.loc[enc, "real_mean"], rv.loc[enc, "zeros_mean"])
        ax.annotate(f'{rv.loc[enc, "margin"]:+.4f}', (i, top), xytext=(0, 14),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(rv.index)
    ax.set_ylabel("test BLEU-4")
    ax.legend(fontsize=8)
    ax.set_title("margin annotated per pair; error bar is one sd across seeds", fontsize=9)
    fig.tight_layout()
    save(fig, "fig_real_vs_blank.png")
    plt.show()

In [ ]:
# Figure: output diversity against the reference corpus. The dashed line is the reference
# row from 13.3 and it is the floor, because the corpus itself is formulaic. Only the gap
# between a model bar and that line is evidence of collapse beyond what the domain imposes.
if have("results_diversity.csv"):
    dv  = pd.read_csv("results_diversity.csv", index_col=0)
    ref = dv.loc["REFERENCE corpus"]
    mod = dv.drop(index="REFERENCE corpus")
    fig, axes = plt.subplots(1, 2, figsize=(7.4, 3.1))
    for ax, colname, better in [(axes[0], "unique outputs", "higher is more diverse"),
                                (axes[1], "self-BLEU-4",    "lower is more diverse")]:
        ax.bar(range(len(mod)), mod[colname], color="#1f77b4", width=0.55)
        ax.axhline(ref[colname], ls="--", c="#444444", lw=1)
        ax.text(len(mod) - 0.45, ref[colname], " reference corpus", fontsize=7.5,
                va="bottom", ha="right", color="#444444")
        ax.set_xticks(range(len(mod)))
        ax.set_xticklabels(mod.index, rotation=20, ha="right", fontsize=8)
        ax.set_title(f"{colname}  ({better})", fontsize=9)
    fig.tight_layout()
    save(fig, "fig_output_diversity.png")
    plt.show()

In [ ]:
# Figure: pooling ablation margins against the criterion fixed before training. The
# criterion, one seed standard deviation of ResNet-34 BLEU-4, is recomputed from
# results_seeds_raw.csv rather than hard coded, so the figure and the 13.13 table cannot
# drift apart.
if have("results_pooling_ablation.csv"):
    pool = pd.read_csv("results_pooling_ablation.csv", index_col=0)
    if os.path.exists("results_seeds_raw.csv"):
        sr   = pd.read_csv("results_seeds_raw.csv")
        crit = float(sr[sr["arm"] == "resnet34"]["BLEU-4"].std(ddof=1))
    else:
        crit = float((pool["margin"] / pool["margin_in_sd"]).iloc[0])
    fig, ax = plt.subplots(figsize=(5.8, 3.2))
    ax.bar(range(len(pool)), pool["margin"], width=0.5, color="#1f77b4")
    ax.axhline(crit, ls="--", c="#b22222", lw=1)
    ax.text(len(pool) - 0.45, crit, " criterion: one seed sd", fontsize=7.5,
            va="bottom", ha="right", color="#b22222")
    for i, (name, r) in enumerate(pool.iterrows()):
        ax.annotate(f'{r["margin"]:+.4f}', (i, r["margin"]), xytext=(0, 4),
                    textcoords="offset points", ha="center", fontsize=8)
    ax.set_xticks(range(len(pool)))
    ax.set_xticklabels(pool.index)
    ax.set_ylabel("real minus blank, test BLEU-4")
    ax.set_title("primary outcome is the margin, not the raw score", fontsize=9)
    fig.tight_layout()
    save(fig, "fig_pooling_margins.png")
    plt.show()

In [ ]:
# Figure: per finding recall for the real arms, ordered by test prevalence, with prevalence
# printed in the right margin. A category a model never produces has zero recall whatever
# its BLEU score, which is why this figure stands next to the n gram metrics.
if have("results_finding_recall.csv"):
    fr = pd.read_csv("results_finding_recall.csv", index_col=0)
    fr = fr.sort_values("prevalence in test")
    y  = np.arange(len(fr))
    fig, ax = plt.subplots(figsize=(6.6, 0.32 * len(fr) + 1.1))
    markers = {"ResNet-34": ("o", COL["resnet34"]),
               "VGG-16":    ("s", COL["vgg16"]),
               "Med3D":     ("^", COL["med3d"])}
    for colname in fr.columns:
        if colname == "prevalence in test":
            continue
        mk, c = markers.get(colname, ("o", "#333333"))
        ax.plot(fr[colname], y, mk, color=c, ms=5, alpha=0.85, label=colname)
    for i, (name, r) in enumerate(fr.iterrows()):
        ax.text(1.03, i, f'{r["prevalence in test"]:.2f}', fontsize=7, va="center")
    ax.set_yticks(y); ax.set_yticklabels(fr.index, fontsize=8)
    ax.set_xlim(-0.02, 1.14)
    ax.set_xlabel("recall on test  (right margin: prevalence in test)")
    ax.legend(fontsize=8, loc="lower right")
    fig.tight_layout()
    save(fig, "fig_finding_recall.png")
    plt.show()

### 14.2 Qualitative example

In [ ]:
# One reference beside the generation for the same study, from the ResNet-34 arm at the
# shared decoding length. The display case is the study nearest the arm's median 4 gram
# overlap, so the panel shows typical behaviour rather than a flattering outlier. The
# overlap score below is used only to choose a display example, never as a reported metric.
def overlap4(p, r):
    tp, tr = str(p).lower().split(), str(r).lower().split()
    gp = {" ".join(tp[i:i + 4]) for i in range(max(0, len(tp) - 3))}
    gr = {" ".join(tr[i:i + 4]) for i in range(max(0, len(tr) - 3))}
    if not gp or not gr:
        return 0.0
    inter = len(gp & gr)
    prec, rec = inter / len(gp), inter / len(gr)
    return 0.0 if prec + rec == 0 else 2 * prec * rec / (prec + rec)

src = f"preds_arm_resnet34_ml{ML}.csv"
if have(src):
    df = pd.read_csv(src).fillna("")
    df["ov"] = [overlap4(p, r) for p, r in zip(df["prediction"], df["reference"])]
    row = df.sort_values("ov").iloc[len(df) // 2]
    print(f"study {row['stem']}, chosen as the median case (overlap {row['ov']:.3f})\n")

    wrap = lambda t: textwrap.fill(str(t), width=58)
    left, right = wrap(row["reference"]), wrap(row["prediction"])
    with open(os.path.join(FIG_DIR, "example_report_pair.txt"), "w") as f:
        f.write(f"study: {row['stem']}\n\nREFERENCE\n{left}\n\nGENERATED (ResNet-34, min_len {ML})\n{right}\n")

    fig, ax = plt.subplots(figsize=(8.2, max(3.2, 0.14 * max(left.count("\n"), right.count("\n")) + 1.6)))
    ax.axis("off")
    ax.text(0.00, 1.0, "REFERENCE",  fontsize=9, fontweight="bold", va="top", family="monospace")
    ax.text(0.53, 1.0, "GENERATED",  fontsize=9, fontweight="bold", va="top", family="monospace")
    ax.text(0.00, 0.93, left,  fontsize=7.5, va="top", family="monospace")
    ax.text(0.53, 0.93, right, fontsize=7.5, va="top", family="monospace")
    fig.tight_layout()
    save(fig, "fig_report_example.png")
    plt.show()
    print("\nplain text copy saved beside it as example_report_pair.txt")

### 14.3 Cached volume montage

In [ ]:
# Figure: what the model sees. Mid slices in three planes from cached npz volumes, after
# the lung window of -1000 to 400 HU scaled to [0, 1], RAS orientation, 1.5 x 1.5 x 2.0 mm
# spacing, and lung masking. Expected: lung parenchyma on a black field, since everything
# outside the lung mask is zeroed before the encoder ever sees the volume.
if not os.path.isdir(CACHE_DIR):
    print(f"SKIPPED: {CACHE_DIR} is not present on this machine")
else:
    stems = sorted(f[:-4] for f in os.listdir(CACHE_DIR) if f.endswith(".npz"))
    pick  = ([s for s in stems if s.startswith("valid")] or stems)[:3]
    if not pick:
        print("SKIPPED: no cached npz volumes found")
    else:
        fig, axes = plt.subplots(len(pick), 3, figsize=(7.2, 2.4 * len(pick)))
        axes = np.atleast_2d(axes)
        for r, s in enumerate(pick):
            img = np.load(os.path.join(CACHE_DIR, f"{s}.npz"))["image"].astype(np.float32)
            H, W, D = img.shape
            planes = [(np.rot90(img[:, :, D // 2]), "axial"),
                      (np.rot90(img[:, W // 2, :]), "coronal"),
                      (np.rot90(img[H // 2, :, :]), "sagittal")]
            for c, (sl, name) in enumerate(planes):
                ax = axes[r, c]
                ax.imshow(sl, cmap="gray", vmin=0, vmax=1)
                ax.set_xticks([]); ax.set_yticks([])
                if r == 0:
                    ax.set_title(name)
                if c == 0:
                    ax.set_ylabel(s, fontsize=7)
        fig.tight_layout()
        save(fig, "fig_cached_volume_montage.png")
        plt.show()

### 14.4 Preprocessing and affine evidence, network required

In [ ]:
# NETWORK REQUIRED. One volume and its lung mask are fetched, which needs the gated
# Hugging Face access that section 1 set up. Every other cell in section 14 runs offline;
# if this cell cannot reach the hub it prints why and the rest of the section is unaffected.
#
# Two figures come out.
#   1. Raw HU slice, windowed slice, lung masked slice: the preprocessing panel.
#   2. The mask affine defect and its repair, with in box lung voxel counts in the titles.
#      Resampling the mask through its shipped world affine sends it outside the image box,
#      while a grid resize followed by copying the image affine keeps the voxel level
#      alignment the dataset actually has. This pairing is the degree of difficulty
#      evidence for the fix adopted in section 4.
try:
    import sys, subprocess, importlib, tempfile
    import tarfile as _tar
    import nibabel as nib
    from huggingface_hub import hf_hub_download
    from monai.transforms import Resize

    try:
        import scipy  # nibabel.processing imports scipy lazily and fails late without it
    except ImportError:
        pin = "scipy==1.11.4" if np.__version__.startswith("1.") else "scipy"
        subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                        "--only-binary=:all:", pin], check=True)
        import nibabel.processing
    importlib.reload(nibabel.processing)   # clears the lazy import tripwire after a fresh install

    # One cached ok study, preferring the validation split by stem prefix, because the
    # manifest on a given machine may cover only one split.
    mani_path = next(p for p in [os.path.join(CACHE_DIR, "manifest.csv"),
                                 os.path.expanduser("~/preproc_cache/manifest.csv")]
                     if os.path.exists(p))
    mani = pd.read_csv(mani_path)
    ok   = mani[mani["status"] == "ok"]
    vv   = ok[ok["stem"].str.startswith("valid")]
    pick = (vv if len(vv) else ok).iloc[0]
    vol, stem = pick["volumename"], pick["stem"]
    arc_split = "valid" if stem.startswith("valid") else "train"
    print(f"using {vol}  (mask archive: {arc_split})")

    split, patient, scan, _ = vol.replace(".nii.gz", "").split("_")
    rel = f"dataset/{split}_fixed/{split}_{patient}/{split}_{patient}_{scan}/{vol}"
    vol_path = hf_hub_download("ibrahimhamamci/CT-RATE", repo_type="dataset", filename=rel)
    arc = hf_hub_download("RadGenome/RadGenome-ChestCT", repo_type="dataset",
                          filename=f"dataset/{arc_split}_region_mask.tar.gz")

    tmp, msk_path = tempfile.mkdtemp(), None
    with _tar.open(arc, "r:gz") as t:
        for mem in t:
            if mem.name.endswith("/lung.nii.gz") and mem.name.split("/")[-2] == f"seg_{stem}":
                msk_path = os.path.join(tmp, "lung.nii.gz")
                with open(msk_path, "wb") as g:
                    g.write(t.extractfile(mem).read())
                break
    assert msk_path, f"lung mask for seg_{stem} not found in the {arc_split} archive"

    img_n, msk_n = nib.load(vol_path), nib.load(msk_path)
    img, msk_raw = img_n.get_fdata(), msk_n.get_fdata()
    z = img.shape[2] // 2
    print(f"image {img.shape}, mask {msk_raw.shape}")

    # The branch the pipeline takes: grid resize when the shapes differ, then the image
    # affine replaces the shipped mask affine.
    if msk_raw.shape != img.shape:
        msk_fix = np.asarray(Resize(spatial_size=img.shape, mode="nearest")(msk_raw[None])[0])
        print(f"alignment branch: grid resize {msk_raw.shape} -> {img.shape}, copy image affine")
    else:
        msk_fix = msk_raw
        print("alignment branch: shapes already match, copy image affine only")

    # The defective alternative, produced only for the figure: a world coordinate resample
    # that trusts the shipped mask affine.
    naive = nibabel.processing.resample_from_to(msk_n, img_n, order=0).get_fdata()
    inbox_naive, inbox_fix = int((naive > 0).sum()), int((msk_fix > 0).sum())

    win = np.clip((img + 1000.0) / 1400.0, 0, 1)

    fig, axes = plt.subplots(1, 3, figsize=(8.4, 3.0))
    panels = [(img[:, :, z], "raw, HU"),
              (win[:, :, z], "windowed to [0, 1], -1000 to 400 HU"),
              (win[:, :, z] * (msk_fix[:, :, z] > 0), "lung masked")]
    for ax, (sl, ttl) in zip(axes, panels):
        ax.imshow(np.rot90(sl), cmap="gray")
        ax.set_title(ttl, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    save(fig, "fig_preprocessing_panel.png")
    plt.show()

    fig, axes = plt.subplots(1, 2, figsize=(7.0, 3.5))
    pairs = [(naive,   f"shipped affine, world resample\nlung voxels in box: {inbox_naive:,}"),
             (msk_fix, f"grid resize, copied affine\nlung voxels in box: {inbox_fix:,}")]
    for ax, (mk, ttl) in zip(axes, pairs):
        ax.imshow(np.rot90(win[:, :, z]), cmap="gray")
        ov = np.rot90((np.asarray(mk)[:, :, z] > 0).astype(float))
        ax.imshow(np.ma.masked_where(ov == 0, ov), cmap="autumn", alpha=0.45)
        ax.set_title(ttl, fontsize=9); ax.set_xticks([]); ax.set_yticks([])
    fig.tight_layout()
    save(fig, "fig_affine_fix.png")
    plt.show()

except Exception as e:
    print("SKIPPED, this cell needs the network and the gated datasets:", repr(e))

### 14.5 Manifest and archive

In [ ]:
# Manifest and archive. Figures produced by earlier sections are gathered in as well, so
# that ~/figures_report.tar.gz is the single object to pull down when assembling the report.
for extra in ["fig_encoder_comparison.png"] + sorted(glob.glob("loss_*.png")):
    if os.path.exists(extra):
        shutil.copy(extra, FIG_DIR)

files = sorted(os.listdir(FIG_DIR))
print(f"{len(files)} files in {FIG_DIR}")
for f in files:
    kb = os.path.getsize(os.path.join(FIG_DIR, f)) / 1024
    print(f"  {f:38s} {kb:8.1f} kB")

out = os.path.expanduser("~/figures_report.tar.gz")
with tarfile.open(out, "w:gz") as t:
    t.add(FIG_DIR, arcname="figures_report")
print(f"\narchive: {out}  ({os.path.getsize(out) / 1e6:.1f} MB)")